In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

---
# MAS Enhancement Plan — Databricks GenAI Engineer Certification Objectives

> **Goal:** Map every Section 2 (Data Preparation) and Section 3 (Application Development) certification objective to our current MAS implementation, identify gaps, and propose concrete enhancements.

---

## Current MAS Architecture Summary

| Layer | Component | Implementation |
| --- | --- | --- |
| **Data Ingestion** | S3 → Auto Loader → `raw_news` (48.5K articles) | Bronze streaming table |
| **Gatekeeper** | spaCy EntityRuler NER + SQL RLIKE | Filters \~95% non-crypto |
| **Chunking/Prep** | `body[:500]` truncation in `gold.py` firehose | Fixed 500-char window |
| **Retrieval** | VS `causal_chains_idx` (few-shot) + `news_firehose_idx` (RAG) | Delta Sync embedding via `databricks-gte-large-en` |
| **Agent Graph** | 9-node LangGraph StateGraph (5-node causality sub-pipeline) | LangChain/LangGraph |
| **LLM** | Foundation Model API (`gemini-3-1-pro` via `mas-news-causal` endpoint) | FMAPI pay-per-token |
| **Evaluation** | 5 deterministic + 3 LLM judge + 3 reference scorers | MLflow GenAI scorers |
| **Serving** | MLflow PyFunc → Model Serving with inference tables | CPU Small, scale-to-zero |
| **Pipeline** | SDP `pipeline.py` (3 materialized views) | Serverless batch |
| **Flywheel** | VS bootstrap: cold-start → 50 accepted → live VS queries | Self-improving retrieval |

---

## Section 2: Data Preparation — Gap Analysis & Enhancement Plan

| # | Objective | Current Status | Gap | Enhancement |
| --- | --- | --- | --- | --- |
| 2.1 | **Apply chunking strategy for document structure & model constraints** | Fixed 500-char `text_window` in gatekeeper, 500-char `body_truncated` in firehose | No semantic chunking; single fixed-size window ignores document structure (headline vs body vs quotes) | **Add hierarchical chunking**: (1) headline chunk (always full), (2) body lead-paragraph chunk (first 300 chars), (3) sliding-window overlap chunks for long bodies (512 tokens, 64-token overlap). Use `langchain.text_splitter.RecursiveCharacterTextSplitter` with token-aware splitting matched to `databricks-gte-large-en` context (512 tokens). Store chunks in `news_chunks` Delta table. |
| 2.2 | **Filter extraneous content that degrades RAG quality** | Gatekeeper filters non-crypto; no body-level noise filtering | Raw bodies contain: ads, disclaimers, "Read more" links, boilerplate, social media embeds, author bios | **Add content sanitizer node** (`content_filter.py`): regex strip ads/disclaimers, detect/remove boilerplate via TF-IDF similarity to template corpus, sentence-level relevance scoring (keep sentences with NER entities). Add as pre-processing step before chunking. |
| 2.3 | **Choose appropriate Python package for document extraction** | Implicit — data arrives as NDJSON from scrapers | No explicit document parsing layer for diverse formats | **Document this design choice explicitly**: our pipeline receives pre-extracted text (NDJSON). For expansion to PDF/HTML sources, implement `unstructured` (for PDFs, images) or `beautifulsoup4` + `readability-lxml` (for HTML). Add a `document_parser.py` module with format dispatch. |
| 2.4 | **Define operations to write chunked text into Delta Lake** | `news_firehose` MV writes 500-char truncated text to Delta | No separate chunks table; no chunk-level metadata | **Create `03_gold.news_chunks` streaming table**: schema = `(chunk_id, story_id, chunk_index, chunk_text, chunk_type, token_count, published_at)`. Write via SDP `@dp.materialized_view`. VS Delta Sync index on `chunk_text` column. |
| 2.5 | **Identify needed source documents for RAG quality** | `raw_news` (48.5K articles) is the sole source | Missing: historical price context, on-chain data, regulatory filings, analyst reports | **Add source diversity**: (1) OHLC gold table already exists (TRD-5 dollar bars); (2) Create `regulatory_filings` bronze table from SEC EDGAR RSS; (3) Use `news_firehose_idx` precedent retrieval to surface historically similar events. Document source-to-quality mapping in taxonomy. |
| 2.6 | **Use tools and metrics to evaluate retrieval performance** | Section 3/14 in this notebook test VS cold-start/bootstrap; `few_shot_retriever` measures `min_vs_results` | No formal retrieval metrics (precision@k, recall@k, MRR, NDCG) | **Add retrieval evaluation module** (`eval/retrieval_scorers.py`): (1) Manually label 50 query-document pairs; (2) Compute precision@5, recall@5, MRR, NDCG@5; (3) Compare VS results vs BM25 baseline; (4) Log to MLflow. Add as Section 3.5 in this notebook. |
| 2.7 | **Design retrieval with advanced chunking strategies** | Single-level headline embedding for VS | No parent-document retrieval, no multi-vector, no hypothetical document embeddings | **Implement multi-level retrieval**: (1) **Small-to-big**: embed chunks → retrieve chunk → return parent document; (2) **HyDE** (Hypothetical Document Embeddings): generate synthetic causal chain from headline, embed synthetic doc, retrieve similar real chains; (3) **Multi-vector**: embed headline + body-lead separately, fuse scores. Requires the `news_chunks` table from 2.4. |
| 2.8 | **Explain role of re-ranking in information retrieval** | No re-ranking; raw VS cosine similarity scores used directly | Top-k results from VS may include false positives (semantically similar but causally irrelevant) | **Add cross-encoder re-ranker**: After VS top-10 retrieval, apply `databricks-gte-large-en` or a fine-tuned cross-encoder to re-score (query, document) pairs. Keep top-5 after re-ranking. Implement in `rag_retriever.py` as a post-retrieval step. Log `retrieval.rerank_precision_lift` metric. |

---

## Section 3: Application Development — Gap Analysis & Enhancement Plan

| # | Objective | Current Status | Gap | Enhancement |
| --- | --- | --- | --- | --- |
| 3.1 | **Select LangChain/similar tools for GenAI application** | LangGraph `StateGraph` for agent orchestration; MLflow for logging/serving | Well covered. Could demonstrate tool selection rationale more explicitly. | **Document tool selection matrix** in README: LangGraph (stateful multi-step), MLflow (lifecycle), Pydantic (schema validation), spaCy (NER). Add comparison cell showing why LangGraph > LangChain sequential chains for this use case (conditional routing, bounded loops). |
| 3.2 | **Qualitatively assess responses for quality & safety issues** | 3 LLM judges (direction, mechanism, grounding); `mas_component_evaluation` Sections 8-9 | No explicit safety guardrail assessment; no toxicity/hallucination detection | **Add quality assessment rubric**: (1) Hallucination detection — verify `evidence_spans` are verbatim from body (already in `evidence_spans_verbatim` scorer); (2) Add `safety_scorer` checking for market manipulation language, unsubstantiated claims; (3) Create "failure mode gallery" visualization showing worst-case outputs. |
| 3.3 | **Select chunking strategy based on model & retrieval evaluation** | Fixed 500-char (implicitly chosen for speed) | No A/B evaluation of chunk sizes; no retrieval-quality-driven selection | **Run chunking experiment**: Test 256/512/1024 token chunks with 0/64/128 overlap. Measure retrieval precision@5 + downstream `validator_accepted` rate for each config. Select optimal via MLflow experiment comparison. |
| 3.4 | **Augment prompt with context from user input (key fields, terms, intents)** | `few_shot_retriever` adds similar examples; `evidence_extractor` extracts key spans; `rag_retriever` adds precedent news | Good coverage. Could make intent extraction more explicit. | **Add explicit intent classification**: Before hypothesis generation, classify article intent (breaking news / analysis / opinion / regulatory filing) and inject intent-specific prompt augmentation. Use `uncertainty_router` output to modulate prompt depth. |
| 3.5 | **Create prompt that adjusts LLM response from baseline to desired output** | YAML prompts in `prompts/` dir (evidence_extractor, hypothesis_generator, 3 judges) | Prompt engineering is done but not A/B tested; no baseline→optimized comparison | **Add prompt versioning experiment**: (1) Log baseline prompt in MLflow; (2) Create optimized prompt with chain-of-thought + few-shot examples; (3) Compare via `mlflow.evaluate()` on same test set; (4) Use MLflow Prompt Registry with "staging" and "production" aliases. |
| 3.6 | **Implement LLM guardrails to prevent negative outcomes** | `mechanism_validator` rejects invalid mechanisms; bounded reflection (max 1 retry); `CausalChain` Pydantic validation | No input guardrails (prompt injection); no output toxicity filter; no PII detection | **Add guardrail layers**: (1) **Input**: Lakeguard-style prompt injection detection before LLM calls; (2) **Output**: Regex + LLM judge for market manipulation language; (3) **Schema**: Already have Pydantic — add `confidence` floor (reject < 0.3); (4) **Rate limiting**: Track LLM call count per article, hard-cap at 15 calls. |
| 3.7 | **Select best LLM based on application attributes** | `gemini-3-1-pro` via FMAPI (chosen for structured output quality + cost) | No documented model selection process; no comparison data | **Run model comparison**: Test `gemini-3-1-pro` vs `meta-llama-3.3-70b-instruct` vs `claude-sonnet-4` on 50-article subset. Compare: schema_valid rate, validator_accepted rate, latency, cost. Log to MLflow. Select based on quality×cost Pareto front. |
| 3.8 | **Select embedding model context length based on docs & queries** | `databricks-gte-large-en` (512 token context) for VS Delta Sync | No explicit context length analysis; headline embeddings well within 512 but body chunks might exceed | **Validate embedding fit**: Compute token distribution of headlines (should be < 128) and body chunks (must be < 512). If body chunks exceed, switch to `databricks-bge-large-en` (8192 context) or truncate with warning. Document choice in config. |
| 3.9 | **Select model from hub/marketplace based on metadata/model cards** | Implicit selection of `gemini-3-1-pro` | No model card comparison documented | **Add model selection notebook**: Query Databricks Marketplace models, compare model cards for structured extraction tasks. Document: parameter count, context window, structured output support, pricing tier. |
| 3.10 | **Select best model based on experiment metrics** | MLflow experiment `/Shared/mas-component-eval` logs all metrics | Metrics exist but no formal model comparison workflow | **Implement MLflow model comparison**: Use `mlflow.evaluate()` with our scorer catalog across 3 candidate LLMs. Generate comparison dashboard. Use `mlflow.search_runs()` to find Pareto-optimal model. |
| 3.11 | **Utilize MLflow and Agent Framework for agentic systems** | MLflow PyFunc wrapping LangGraph; `mlflow.trace()` on predict; MLflow model registry in UC | Strong coverage. Could add MLflow Tracing to individual nodes. | **Add per-node MLflow tracing**: Wrap each of 9 graph nodes with `@mlflow.trace()` decorator. Enables latency breakdown per node in MLflow UI. Add `mlflow.log_input()` for the test dataset. |
| 3.12 | **Compare evaluation and monitoring phases of GenAI lifecycle** | Offline eval (this notebook + `e2e_mas_evaluation`); inference tables for monitoring | No online monitoring dashboard; no drift detection; no feedback loop | **Add monitoring layer**: (1) Create AI Gateway inference table dashboard showing: request volume, latency p50/p95, error rate, validation_status distribution over time; (2) Add `confidence` drift detection (alert if mean confidence drops below 0.6); (3) Implement human feedback loop via labeling UI → `eval/labeling.py` → VS flywheel. |
| 3.13 | **Enable multi-agent systems to leverage Genie Spaces or conversational API** | Not implemented — MAS is batch-only via Model Serving | No conversational interface; no Genie Space integration | **Add Genie Space integration**: (1) Create a Genie Space backed by `news_causal_features` gold table — enables natural language queries over causal analysis results; (2) Add a conversational wrapper endpoint that accepts free-form questions, routes to appropriate MAS components, and returns natural language summaries; (3) Implement tool-calling agent that can invoke the MAS endpoint as a tool within a Genie conversation. |

---

## Priority Roadmap (Proposed)

| Phase | Enhancements | Effort | Impact on Cert Objectives |
| --- | --- | --- | --- |
| **P0 (Quick wins)** | 2.6 retrieval metrics, 3.11 MLflow tracing, 3.5 prompt versioning | 2-3 days | 3 objectives fully covered |
| **P1 (Chunking)** | 2.1 hierarchical chunking, 2.4 chunks table, 2.7 multi-level retrieval, 3.3 chunk evaluation | 1 week | 4 objectives, major RAG improvement |
| **P2 (Guardrails)** | 2.2 content filtering, 3.6 input/output guardrails, 3.2 quality rubric | 3-4 days | 3 objectives, safety coverage |
| **P3 (Model selection)** | 3.7 LLM comparison, 3.8 embedding validation, 3.9 marketplace selection, 3.10 experiment comparison | 3-4 days | 4 objectives |
| **P4 (Monitoring)** | 3.12 monitoring dashboard + drift detection, 2.8 re-ranking | 3-4 days | 2 objectives |
| **P5 (Genie)** | 3.13 Genie Space + conversational API | 2-3 days | 1 objective (key differentiator) |

---

## What's Already Strong (No Enhancement Needed)

* **LangGraph multi-agent orchestration** — conditional routing, bounded loops, state management (covers 3.1)
* **MLflow Agent Framework** — PyFunc wrapping, UC model registry, inference tables (covers 3.11 partially)
* **Pydantic schema validation** — `CausalChain` model enforces output structure (covers 3.6 partially)
* **SDP pipeline architecture** — `pipeline.py` → materialized views with expectations (covers 2.4 partially)
* **VS bootstrap flywheel** — self-improving retrieval quality (covers 2.7 partially)
* **Deterministic + LLM judge evaluation** — comprehensive scorer catalog (covers 3.2 partially)

---
# P1 Chunking — Implementation Plan

> **Certification objectives:** 2.1 (chunking strategy), 2.4 (write chunks to Delta), 2.7 (advanced retrieval with chunking), 3.3 (evaluate chunk config)
> **Estimated effort:** 1 week | **Files touched:** 6 new + 3 modified

---

## Motivation (Data-Backed)

| Metric | Value |
| --- | --- |
| Total articles | 48,550 |
| Avg headline length | 73 chars (well within 512-token embedding limit) |
| Avg body length | 1,996 chars |
| Median body length | 897 chars |
| P95 body length | 7,188 chars |
| Max body length | 143,131 chars |
| **Articles truncated by 500-char limit** | **62.5%** |
| **Avg content lost (for truncated articles)** | **40.9%** |

The current `body[:500]` in `gold.py` (`gold_news_firehose`) and `text_window=500` in `GatekeeperConfig` discard **>40% of body content** for the majority of articles. This means:
- VS embeddings miss key causal signals buried deeper in articles
- Evidence extractor LLM sees incomplete context
- Gatekeeper NER misses entities outside the 500-char window

---

## Architecture Changes

```
Current Flow:
  raw_news → silver.news_cleaned → gold.news_firehose (body[:500]) → VS Delta Sync (headline-embedded)

New Flow:
  raw_news → silver.news_cleaned → silver.news_chunks (hierarchical chunks) → gold.news_firehose_v2 (chunk-level) → VS Delta Sync (chunk-embedded)
                                                                           ↘ gold.news_causal_features (use parent doc)
```

---

## Implementation Steps

### Step 1: `ChunkingConfig` frozen dataclass (`config.py` modification)

```python
@dataclass(frozen=True)
class ChunkingConfig:
    """Hierarchical chunking parameters for news articles."""
    # Token-aware splitting matched to databricks-gte-large-en (512 token context)
    max_chunk_tokens: int = 384        # leave 128 token headroom for model
    overlap_tokens: int = 64           # sliding window overlap
    headline_always_full: bool = True  # headline is always its own chunk
    lead_paragraph_chars: int = 300    # first 300 chars as dedicated "lead" chunk
    min_chunk_chars: int = 50          # discard fragments below this
    tokenizer_model: str = "cl100k_base"  # tiktoken model for token counting
```

Add `chunking: ChunkingConfig = field(default_factory=ChunkingConfig)` to `MASConfig`.

---

### Step 2: `chunking.py` module (NEW — `src/trading_databricks/agents/chunking.py`)

**\~150 lines.** Hierarchical chunking logic:

```
def chunk_article(headline: str, body: str, config: ChunkingConfig) -> list[dict]:
    """
    Returns list of chunk dicts:
      {
        chunk_index: int,
        chunk_text: str,
        chunk_type: "headline" | "lead" | "body_window",
        token_count: int,
      }
    """
```

**Strategy:**
1. **Chunk 0 — Headline** (always full, avg 73 chars ≈ 18 tokens)
2. **Chunk 1 — Lead paragraph** (first 300 chars of body — captures the journalistic "inverted pyramid" lede)
3. **Chunks 2..N — Sliding body windows** (384 tokens, 64 overlap) via `RecursiveCharacterTextSplitter` with tiktoken token counting

**Separator hierarchy** (respects document structure):
```python
separators = ["\n\n", "\n", ". ", ", ", " "]
```

**Dependency:** `langchain-text-splitters` (already in env via `langgraph` dep) + `tiktoken`.

---

### Step 3: `silver.news_chunks` streaming table (`pipeline.py` modification)

**Schema:**
```sql
CREATE TABLE silver.news_chunks (
  chunk_id      STRING NOT NULL,   -- "{story_id}_{chunk_index}"
  story_id      STRING NOT NULL,   -- FK to news_cleaned
  chunk_index   INT NOT NULL,      -- 0-based position within article
  chunk_text    STRING NOT NULL,   -- the actual chunk content
  chunk_type    STRING NOT NULL,   -- "headline" | "lead" | "body_window"
  token_count   INT NOT NULL,      -- tiktoken token count
  published_at  TIMESTAMP,         -- denormalized for VS filtering
  source        STRING             -- denormalized for VS filtering
)
CLUSTER BY (published_at)
```

**Pipeline addition** in `pipeline.py`:
```python
@dp.materialized_view(
    name=f"{silver_schema}.news_chunks",
    comment="Hierarchical chunks from news articles for VS embedding.",
    cluster_by=["published_at"],
)
@dp.expect_or_drop("valid_chunk_text", "chunk_text IS NOT NULL AND LENGTH(TRIM(chunk_text)) > 0")
def silver_news_chunks():
    # Uses pandas UDF to apply chunk_article() across all cleaned articles
    ...
```

**Expected row count:** 48,550 articles × avg \~4 chunks/article ≈ **\~194K chunks**

---

### Step 4: VS Delta Sync index on `news_chunks` (NEW index)

**Index name:** `smart-trading-dev.03_gold.news_chunks_idx`
**Embedding source column:** `chunk_text`
**Embedding model:** `databricks-gte-large-en` (512 token context — matches `max_chunk_tokens=384`)
**Primary key:** `chunk_id`
**Sync mode:** TRIGGERED (batch pipeline)

```python
# Via Databricks SDK or Unity Catalog DDL:
CREATE VECTOR SEARCH INDEX news_chunks_idx
ON silver.news_chunks (chunk_text)
USING MODEL `databricks-gte-large-en`
OPTIONS (primary_key = 'chunk_id')
```

---

### Step 5: Multi-level retrieval (`rag_retriever.py` enhancement)

**Objective 2.7 — Small-to-big pattern:**
1. Embed query (headline + top evidence span)
2. Retrieve top-10 **chunks** from `news_chunks_idx`
3. Group chunks by `story_id` → deduplicate to top-5 **parent articles**
4. Return full parent article context (headline + full body from `news_cleaned`)

**Why small-to-big wins here:**
- Chunk-level embedding captures specific causal signals ("SEC approves...")
- Parent-level return gives LLM full context for mechanism extraction
- Avoids the "needle in haystack" problem of embedding full 7K-char articles

**HyDE extension** (Hypothetical Document Embeddings):
```python
def generate_hyde_query(headline: str, evidence: list[dict]) -> str:
    """Generate a synthetic causal chain from headline to use as retrieval query.
    Embeds the hypothetical doc, retrieves real docs similar to it."""
    # Quick LLM call to generate 1-sentence hypothetical causal chain
    # Then embed THAT instead of raw headline → better semantic match
```

---

### Step 6: Chunking evaluation experiment (Objective 3.3)

**Experiment design** (add as Section 3.5 in this notebook):

| Config ID | max_tokens | overlap | Strategy |
| --- | --- | --- | --- |
| A (baseline) | 500 chars | 0 | Current `body[:500]` |
| B | 256 tokens | 0 | Small chunks, no overlap |
| C | 384 tokens | 64 | Medium + overlap (proposed default) |
| D | 512 tokens | 128 | Max context, large overlap |

**Metrics per config:**
1. **Retrieval precision@5** — what % of top-5 VS hits are causally relevant?
2. **Retrieval recall** — does the correct parent article appear in top-10?
3. **Downstream `validator_accepted` rate** — does better retrieval → better causal chains?
4. **Token efficiency** — chunks/article ratio, total index size

**Evaluation flow:**
```python
for config in [A, B, C, D]:
    chunks = chunk_all_articles(config)
    index = build_temp_vs_index(chunks)
    retrieval_scores = evaluate_retrieval(index, labeled_pairs)
    e2e_scores = run_mas_pipeline(config)  # uses new chunks
    mlflow.log_metrics({f"{config.name}.precision_at_5": ..., ...})
```

Compare via `mlflow.search_runs()` → select Pareto-optimal config.

---

## File Manifest

| File | Action | Lines | Objective |
| --- | --- | --- | --- |
| `src/trading_databricks/agents/chunking.py` | **NEW** | \~150 | 2.1 |
| `src/trading_databricks/agents/config.py` | MODIFY (add `ChunkingConfig`) | +15 | 2.1 |
| `src/trading_databricks/agents/pipeline.py` | MODIFY (add `silver_news_chunks` MV) | +40 | 2.4 |
| `src/trading_databricks/agents/nodes/rag_retriever.py` | MODIFY (small-to-big + HyDE) | +60 | 2.7 |
| `notebooks/qa/mas/chunking_experiment.py` | **NEW** notebook | \~200 | 3.3 |
| `tests/agents/test_chunking.py` | **NEW** | \~80 | — |
| `resources/agents_pipeline.pipeline.yml` | MODIFY (if separate pipeline needed) | +5 | 2.4 |

---

## Dependency Chain

```
Step 1 (ChunkingConfig)  ← prerequisite for all
     ↓
Step 2 (chunking.py)     ← pure function, testable immediately
     ↓
Step 3 (pipeline MV)     ← writes chunks to Delta
     ↓
Step 4 (VS index)        ← Delta Sync auto-embeds chunks
     ↓
Step 5 (rag_retriever)   ← small-to-big retrieval
     ↓
Step 6 (experiment)      ← A/B evaluation across configs
```

---

## Risks & Mitigations

| Risk | Mitigation |
| --- | --- |
| tiktoken not available in SDP serverless | Use `pandas_udf` with broadcast — tiktoken is pure Python, installs via `%pip` in pipeline config |
| VS index build time for 194K chunks | Use TRIGGERED sync mode (batch), not CONTINUOUS |
| Token budget: 384 tokens × 194K chunks = \~74M tokens to embed | `databricks-gte-large-en` Delta Sync handles this automatically — no manual embedding needed |
| HyDE adds LLM latency to retrieval path | Make HyDE optional via config flag; fall back to direct headline query |
| Overlap chunks create redundancy in VS | Deduplicate by `story_id` in retrieval step (already planned in small-to-big) |
| Breaking change to `news_firehose` consumers | Keep `news_firehose` MV unchanged; add `news_chunks` as parallel table. Migrate consumers incrementally. |

---

## Success Criteria

1. `news_chunks` table materializes with \~194K rows, all `token_count ≤ 384`
2. VS index returns relevant chunks with cosine > 0.7 for test headlines
3. Retrieval precision@5 improves from baseline (measure in experiment)
4. `validator_accepted` rate ≥ current level (no regression)
5. All 4 cert objectives (2.1, 2.4, 2.7, 3.3) demonstrably covered with code + evaluation artifacts

In [0]:
# ── P1 Validation: Test hierarchical chunking on real articles ─────────────────
import importlib.util
import sys
import types
from dataclasses import dataclass

import polars as pl
from pyspark.sql import functions as F

# ── Load chunking module from workspace (bypasses stale package cache) ──────
@dataclass(frozen=True)
class ChunkingConfig:
    """Mirrors trading_databricks.agents.config.ChunkingConfig."""
    max_chunk_tokens: int = 384
    overlap_tokens: int = 64
    headline_always_full: bool = True
    lead_paragraph_chars: int = 300
    min_chunk_chars: int = 50
    chars_per_token: float = 3.5
    separators: tuple = ("\n\n", "\n", ". ", ", ", " ")

_mock_cfg = types.ModuleType("trading_databricks.agents.config")
_mock_cfg.ChunkingConfig = ChunkingConfig
sys.modules["trading_databricks.agents.config"] = _mock_cfg

if "trading_databricks.agents.chunking" in sys.modules:
    del sys.modules["trading_databricks.agents.chunking"]
importlib.invalidate_caches()

_path = "/Workspace/Users/aaronng.dev1@gmail.com/trading-databricks/src/trading_databricks/agents/chunking.py"
_spec = importlib.util.spec_from_file_location("trading_databricks.agents.chunking", _path)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
chunk_article = _mod.chunk_article

config = ChunkingConfig()
print(f"ChunkingConfig: max_chunk_tokens={config.max_chunk_tokens}, "
      f"overlap={config.overlap_tokens}, lead_chars={config.lead_paragraph_chars}, "
      f"chars_per_token={config.chars_per_token}")

# ── Chunk 500 real articles (truncation-affected subset) ────────────────────
samples = (
    spark.table("`smart-trading-dev`.`01_bronze`.raw_news")
    .select("story_id", "headline", "body")
    .filter(F.length("body") > 500)
    .orderBy(F.rand(seed=42))
    .limit(500)
    .collect()
)

all_chunks = []
for row in samples:
    chunks = chunk_article(row["headline"] or "", row["body"] or "", config=config)
    for c in chunks:
        c["story_id"] = row["story_id"]
    all_chunks.extend(chunks)

chunks_df = pl.DataFrame(all_chunks)

# ── Results ────────────────────────────────────────────────────────────────
print(f"\nArticles: {len(samples)} | Chunks: {len(all_chunks)} | Avg: {len(all_chunks)/len(samples):.2f}/article")
print("\nToken distribution by chunk_type:")
print("─" * 65)
for ct in ["headline", "lead", "body_window"]:
    subset = chunks_df.filter(pl.col("chunk_type") == ct)
    if len(subset) == 0:
        continue
    tokens = subset["token_count"]
    print(f"  {ct:12s}  n={len(subset):5d}  "
          f"mean={tokens.mean():.0f}  p50={tokens.median():.0f}  "
          f"p95={tokens.quantile(0.95):.0f}  max={tokens.max()}")

# Token budget assertion (allow +1 rounding tolerance)
max_tok = chunks_df["token_count"].max()
assert max_tok <= config.max_chunk_tokens + 1, f"FAIL: max={max_tok} > {config.max_chunk_tokens}+1"
print(f"\n✓ All chunks within token budget (max={max_tok} ≤ {config.max_chunk_tokens}+1 rounding)")

# Extrapolation
total_articles = 48_550
est_total = int(len(all_chunks) / len(samples) * total_articles * 0.625 + total_articles * 0.375 * 2)
print(f"\n── Full corpus estimate: ~{est_total:,} chunks (~{est_total:,} VS embeddings)")

### P1 Implementation Status

| Step | Description | Status | File |
| --- | --- | --- | --- |
| 1 | `ChunkingConfig` frozen dataclass | ✓ Done | `config.py` |
| 2 | `chunking.py` module (hierarchical splitting) | ✓ Done | `agents/chunking.py` (NEW) |
| 3 | `silver.news_chunks` materialized view | ✓ Done | `agents/pipeline.py` |
| 4 | VS Delta Sync index on chunks | Pending deploy | DDL below |
| 5 | Small-to-big retrieval + HyDE | ✓ Done | `nodes/rag_retriever.py` |
| 6 | Chunking evaluation experiment | Next step | Separate notebook |

---

#### Step 4: VS Index Creation (run after pipeline deploys `silver.news_chunks`)

```sql
-- Create Vector Search endpoint (if not exists)
CREATE VECTOR SEARCH ENDPOINT IF NOT EXISTS `mas-vs-endpoint`;

-- Create Delta Sync index on chunk_text column
CREATE VECTOR SEARCH INDEX `smart-trading-dev`.`02_silver`.`news_chunks_idx`
ON TABLE `smart-trading-dev`.`02_silver`.`news_chunks`
(
  chunk_text EMBEDDING
    databricks_model_endpoint_name = 'databricks-gte-large-en'
)
WITH (
  primary_key = 'chunk_id',
  pipeline_type = 'TRIGGERED',
  embedding_dimension = 1024
);
```

#### Architecture Summary (after P1)

```
raw_news (48.5K)
  │
  ▼
silver.news_cleaned (dedup + normalize)
  │
  ├──▶ silver.news_chunks (~178K chunks)  ───▶ VS news_chunks_idx (chunk-embedded)
  │       │
  │       └── rag_retriever (small-to-big: chunk hits → parent articles)
  │
  ├──▶ gold.news_firehose (headline-embedded)  ───▶ VS news_firehose_idx
  │
  └──▶ gold.news_causal_features (MAS agent output)
```

# MAS Component Evaluation Notebook

> **Purpose:** Evaluate every single component of the Multi-Agent System in isolation, then as an integrated graph, then across the full production stack (SDP pipeline, Model Serving, Vector Search). Each section runs one component with controlled inputs, measures quality metrics, and renders diagnostic visualizations.

> **Created:** 2026-04-20 | **Graph version:** 9-node (5-node causality sub-pipeline) | **Model:** `smart-trading-dev.03_gold.mas_news_causal` v6

---

## Architecture Under Test

```
Outer Pipeline                          Causality Sub-Pipeline
─────────────                          ──────────────────────
context_retriever (V1 stub)            evidence_extractor (LLM)
       │                                      │
few_shot_retriever (VS + cold-start)   rag_retriever (firehose VS stub)
       │                                      │
uncertainty_router (threshold logic)   hypothesis_generator (LLM + taxonomy)
       │                                      │
       └──── route_article ──────────▶ judge_panel (3 LLM judges)
                                              │
                                       reflector (bounded loop)
                                              │
                                       mechanism_validator (5-step rules)
```

**Infrastructure under test:** SDP pipeline (`pipeline.py`), Gold table (`news_causal_features`), Serving endpoint (`mas-news-causal`), VS flywheel (`causal_chains_idx`)

---

## Gap Analysis vs Existing QA

| Existing Asset | What It Tests | Gap |
| --- | --- | --- |
| `01_gatekeeper.py` – `06_mechanism_validator.py` | Per-node CLI scripts | Not interactive, can't run on serverless, no visualizations |
| `e2e_mas_evaluation` notebook | Black-box via `run_evaluation()` | No component isolation, no sub-pipeline diagnostics |
| `e2e_mas_deep_dive_research` notebook | Full graph with MLflow logging | No evidence_extractor / rag_retriever / hypothesis_generator / judge_panel / reflector isolation |
| `eval/scorers.py` | 5 deterministic + 3 judge + 3 reference scorers | Missing: gatekeeper parity, VS quality, judge calibration, reflection improvement |

**This notebook fills every gap above.**

---

## Notebook Structure (15 Sections, \~70 Cells)

| # | Section | Component | LLM Calls? | Cells |
| --- | --- | --- | --- | --- |
| 0 | Setup & Environment | — | No | 5 |
| 1 | Gatekeeper | `gatekeeper.py` + `gold.py` RLIKE | No | 6 |
| 2 | Context Retriever | `context_retriever.py` | No | 2 |
| 3 | Few-Shot Retriever | `few_shot_retriever.py` + VS | No | 7 |
| 4 | Uncertainty Router | `uncertainty_router.py` | No | 4 |
| 5 | Evidence Extractor | `evidence_extractor.py` | Yes (\~20) | 5 |
| 6 | RAG Retriever | `rag_retriever.py` | No | 3 |
| 7 | Hypothesis Generator | `hypothesis_generator.py` | Yes (\~20-40) | 7 |
| 8 | Judge Panel | `judge_panel.py` | Yes (\~60) | 6 |
| 9 | Reflector | `reflector.py` | No | 4 |
| 10 | Mechanism Validator | `mechanism_validator.py` | No (embeddings only) | 6 |
| 11 | Full Graph E2E | `graph.py` | Yes (\~120*3) | 6 |
| 12 | SDP Pipeline Gold | `pipeline.py` + `gold.py` | No | 5 |
| 13 | Serving Endpoint | `serve.py` | Yes (endpoint) | 4 |
| 14 | VS Bootstrap Flywheel | VS index + staging table | No | 4 |
| 15 | Summary Dashboard | All metrics | No | 2 |

**Estimated LLM calls:** \~400-500 (\~$0.50-1.00 on FMAPI)

---

## Conventions

* **Visualizations:** Plotly dark theme (`template='plotly_dark'`, bg `#0D1117`/`#161B22`)
* **Metrics:** All logged to MLflow experiment `/Shared/mas-component-eval`
* **Assertions:** Every section ends with hard `assert` gates
* **Data:** 200-row test set (120 crypto + 80 non-crypto) from `raw_news`
* **Config:** `MASConfig()` default frozen dataclasses throughout

---
## Section 0: Setup & Environment

5 cells. Zero LLM calls. Establishes all imports, config, reference data, and the test dataset.

In [0]:
%pip install spacy==3.8.4 langgraph>=1.1.6 pydantic>=2.0 plotly mlflow polars --quiet
dbutils.library.restartPython()

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/aaronng.dev1@gmail.com/trading-databricks/src")

import trading_databricks
print(f"trading_databricks loaded from: {trading_databricks.__file__}")

# ── Project imports ─────────────────────────────────────────────────────────────────
from trading_databricks.agents.config import MASConfig, GatekeeperConfig
from trading_databricks.agents.gatekeeper import (
    build_nlp,
    extract_ner_features,
    is_crypto_relevant,
    CRYPTO_ENTITY_PATTERNS,
)
from trading_databricks.agents.gold import CRYPTO_RLIKE_PATTERN, _apply_sql_gatekeeper
from trading_databricks.agents.state import MASState
from trading_databricks.agents.schemas import CausalChain, CAUSAL_FEATURES_SCHEMA
from trading_databricks.agents.taxonomy import load_taxonomy, load_whitelist
from trading_databricks.agents.nodes.context_retriever import context_retriever
from trading_databricks.agents.graph import build_mas_graph

# ── Third-party ──────────────────────────────────────────────────────────────────
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import mlflow
from pyspark.sql import functions as F

print("✓ All imports resolved")

In [0]:
config = MASConfig()
taxonomy = load_taxonomy(spark, catalog=config.catalog, schema=config.schema)
whitelist = load_whitelist(spark, catalog=config.catalog, schema=config.schema)

# ── Validate reference data ─────────────────────────────────────────────────────
assert len(taxonomy) == 8, f"Expected 8 L1 categories, got {len(taxonomy)}"
l2_count = sum(len(v) for v in taxonomy.values())
assert l2_count == 34, f"Expected 34 L2 subcategories, got {l2_count}"
assert len(whitelist) == 40, f"Expected 40 whitelist entries, got {len(whitelist)}"

# ── Summary table ────────────────────────────────────────────────────────────
print(f"{'L1 Category':<15} {'L2 Count':>10} {'Mechanisms':>12}")
print("─" * 39)
for l1 in sorted(taxonomy):
    mech_count = sum(1 for (cat, _) in whitelist if cat == l1)
    print(f"{l1:<15} {len(taxonomy[l1]):>10} {mech_count:>12}")

print(f"\n✓ Config loaded: {l2_count} treatments, {len(whitelist)} mechanisms")
print(f"  Catalog: {config.catalog} | Schema: {config.schema}")
print(f"  LLM endpoint: {config.llm_endpoint}")
print(f"  VS endpoint: {config.vs_endpoint_name}")

### Cell 0.4 `[python]` — MLflow Experiment Setup

**Purpose:** Set MLflow experiment for all subsequent metric logging.

```python
mlflow.set_experiment("/Shared/mas-component-eval")
mlflow.start_run(run_name="component_eval_{timestamp}")
```

**Notes:** Single parent run. Each section logs metrics with `section.metric_name` prefix.  
Run is ended in Section 15 (Summary).

In [0]:
from datetime import datetime

mlflow.set_experiment("/Shared/mas-component-eval")
run = mlflow.start_run(run_name=f"component_eval_{datetime.now():%Y%m%d_%H%M%S}")

print(f"✓ MLflow run started: {run.info.run_id}")
print(f"  Experiment: /Shared/mas-component-eval")

In [0]:
import pandas as pd
from pyspark.sql.types import BooleanType

CRYPTO_N, NON_CRYPTO_N = 120, 80

# ── Read source data ──────────────────────────────────────────────────────────
raw_news = spark.table("`smart-trading-dev`.`01_bronze`.raw_news")
print(f"Source rows: {raw_news.count():,}")

# ── Tag all articles with spaCy gatekeeper ───────────────────────────────────

@F.pandas_udf(BooleanType())
def _gatekeeper_tag_udf(headlines: pd.Series, bodies: pd.Series) -> pd.Series:
    cfg = GatekeeperConfig()
    return pd.Series(
        [
            bool(
                is_crypto_relevant(
                    str(h) if h is not None else "",
                    str(b) if b is not None else "",
                    config=cfg,
                )[0]
            )
            for h, b in zip(headlines, bodies, strict=False)
        ],
        dtype=object,
    )

tagged = raw_news.withColumn(
    "is_crypto", _gatekeeper_tag_udf(F.col("headline"), F.col("body"))
)

crypto_count = tagged.filter(F.col("is_crypto")).count()
non_crypto_count = tagged.filter(~F.col("is_crypto")).count()
print(f"Crypto-relevant: {crypto_count:,}  |  Non-crypto: {non_crypto_count:,}")

# ── Sample: 120 crypto (most recent) + 80 non-crypto (random) ──────────────
crypto_sample = (
    tagged.filter(F.col("is_crypto"))
    .orderBy(F.col("published_at").desc())
    .limit(CRYPTO_N)
)
non_crypto_sample = (
    tagged.filter(~F.col("is_crypto"))
    .orderBy(F.rand(seed=42))
    .limit(NON_CRYPTO_N)
)

test_sdf = crypto_sample.unionByName(non_crypto_sample)
test_sdf.createOrReplaceTempView("mas_test_200")
test_pdf = test_sdf.toPandas()

# ── Convert to list[dict] for direct node invocation ─────────────────────
test_articles = test_pdf.to_dict("records")
crypto_articles = [a for a in test_articles if a["is_crypto"]]
non_crypto_articles = [a for a in test_articles if not a["is_crypto"]]

assert len(test_articles) == 200, f"Expected 200, got {len(test_articles)}"
assert len(crypto_articles) == CRYPTO_N, f"Expected {CRYPTO_N} crypto, got {len(crypto_articles)}"

print(f"\nTest set: {len(test_articles)} articles ({len(crypto_articles)} crypto, {len(non_crypto_articles)} non-crypto)")
display(test_sdf.limit(5))

---
## Section 1: Gatekeeper — spaCy EntityRuler NER (6 cells)

**Component:** `gatekeeper.py` (spaCy mode) + `gold.py` (SQL RLIKE mode)  
**What it does:** Filters \~95% of non-crypto articles. First line of defense.  
**No LLM calls.** Pure rule-based NER + regex matching.

**Key questions:**
1. Do spaCy and SQL RLIKE gatekeepers agree? (Parity test)
2. What is the entity category distribution across crypto articles?
3. Is `build_nlp()` properly cached? (Performance)
4. Does the 500-char `text_window` miss entities in long bodies?

### Cell 1.1 `[python]` — Run spaCy Gatekeeper on All 200 Articles

**Logic:**
```
for article in test_articles:
    doc = build_nlp()(headline + " " + body[:config.gatekeeper.text_window])
    ner_features = extract_ner_features(doc)
    is_impactful, confidence = is_crypto_relevant(headline, body, config=config.gatekeeper)
    # collect: {**article, **ner_features, is_impactful, confidence}
```

**Variables created:**
* `gatekeeper_results: pl.DataFrame` with columns: `story_id`, `headline`, `is_impactful`, `confidence`, `entity_count`, `has_crypto_asset`, `has_exchange`, `has_stablecoin`, `has_protocol`, `has_concept`

**Print:** Summary: N impactful, N non-impactful, mean entity count.

In [0]:
nlp = build_nlp()
gk_config = config.gatekeeper

results = []
for article in test_articles:
    headline = str(article["headline"]) if article["headline"] is not None else ""
    body = str(article["body"]) if article["body"] is not None else ""

    text = headline + " " + body[: gk_config.text_window]
    doc = nlp(text)
    ner_features = extract_ner_features(doc)
    is_impactful, confidence = is_crypto_relevant(headline, body, config=gk_config)

    results.append(
        {
            "story_id": article["story_id"],
            "headline": headline[:100],
            "is_impactful": is_impactful,
            "confidence": confidence,
            **ner_features,
        }
    )

gatekeeper_results = pl.DataFrame(results)

# ── Summary ─────────────────────────────────────────────────────────────────
n_impactful = gatekeeper_results.filter(pl.col("is_impactful")).height
n_non_impactful = gatekeeper_results.filter(~pl.col("is_impactful")).height
mean_entities = gatekeeper_results["entity_count"].mean()

print(f"Impactful: {n_impactful} | Non-impactful: {n_non_impactful}")
print(f"Mean entity count: {mean_entities:.2f}")
display(gatekeeper_results.head(10).to_pandas())

### Cell 1.2 `[python]` — SQL RLIKE Gatekeeper Parity Test

**Purpose:** Run the SQL RLIKE pattern from `gold.py` on the same 200 articles. Compare with spaCy results.

**Logic:**
1. Apply `CRYPTO_RLIKE_PATTERN` to test_sdf via `.filter(headline.rlike(...) | body.rlike(...))`
2. Collect story_ids that pass SQL gatekeeper
3. Compare: `spacy_positive_ids ∩ sql_positive_ids`, `spacy_only`, `sql_only`

**Variables created:**
* `parity_df: pl.DataFrame` with columns: `story_id`, `spacy_pass`, `sql_pass`, `agree`
* `agreement_rate: float`

**Metrics logged:** `mlflow.log_metric("gatekeeper.spacy_sql_agreement", agreement_rate)`

**Print:** Agreement rate, count of discrepancies, list first 5 disagreements for inspection.

**Key insight from code:** SQL RLIKE uses `\b` word boundaries + `(?i)` case-insensitive. spaCy uses `{LOWER: ...}` and `{TEXT: ...}` patterns. Multi-word terms differ: RLIKE uses `\s+`, spaCy uses sequential tokens. Expect \~98% agreement; gap comes from tokenization edge cases.

In [0]:
# ── SQL RLIKE gatekeeper on the same test set ─────────────────────────────
sql_positive = test_sdf.filter(
    F.col("headline").rlike(CRYPTO_RLIKE_PATTERN)
    | F.col("body").rlike(CRYPTO_RLIKE_PATTERN)
)
sql_positive_ids = {row.story_id for row in sql_positive.select("story_id").collect()}

# spaCy positive IDs from gatekeeper_results
spacy_positive_ids = set(
    gatekeeper_results.filter(pl.col("is_impactful"))["story_id"].to_list()
)
all_ids = set(gatekeeper_results["story_id"].to_list())

# ── Build parity DataFrame ──────────────────────────────────────────────────
parity_records = []
for sid in all_ids:
    spacy_pass = sid in spacy_positive_ids
    sql_pass = sid in sql_positive_ids
    parity_records.append(
        {"story_id": sid, "spacy_pass": spacy_pass, "sql_pass": sql_pass, "agree": spacy_pass == sql_pass}
    )

parity_df = pl.DataFrame(parity_records)
agreement_rate = parity_df["agree"].mean()

mlflow.log_metric("gatekeeper.spacy_sql_agreement", agreement_rate)

# ── Summary ─────────────────────────────────────────────────────────────────
spacy_only = spacy_positive_ids - sql_positive_ids
sql_only = sql_positive_ids - spacy_positive_ids

print(f"Agreement rate: {agreement_rate:.2%} ({parity_df.filter(pl.col('agree')).height}/{parity_df.height})")
print(f"Disagreements: {parity_df.filter(~pl.col('agree')).height}")
print(f"  spaCy-only positives: {len(spacy_only)}")
print(f"  SQL-only positives:   {len(sql_only)}")

if parity_df.filter(~pl.col("agree")).height > 0:
    print("\nFirst 5 disagreements:")
    display(parity_df.filter(~pl.col("agree")).head(5).to_pandas())

### Cell 1.3 `[python]` — Entity Category Coverage Analysis

**Purpose:** For crypto-relevant articles, which NER categories fire most?

**Logic:** From `gatekeeper_results` where `is_impactful == True`:
* Per-category hit rates: `has_crypto_asset`, `has_exchange`, `has_stablecoin`, `has_protocol`, `has_concept`
* Co-occurrence matrix: which categories appear together?
* `text_window` sensitivity: re-run with `text_window=1000` and compare entity counts

**Metrics logged:**
* `gatekeeper.category_hit_rate.{category}` for each of 5 categories
* `gatekeeper.mean_entity_count`
* `gatekeeper.text_window_miss_rate` (articles where expanding window finds more entities)

**Print:** Category hit rate table, top 5 most entity-rich articles.

In [0]:
# ── Category hit rates for crypto-relevant articles ─────────────────────
crypto_gk = gatekeeper_results.filter(pl.col("is_impactful"))

categories = ["has_crypto_asset", "has_exchange", "has_stablecoin", "has_protocol", "has_concept"]
hit_rates = {}
for cat in categories:
    rate = crypto_gk[cat].sum() / crypto_gk.height
    hit_rates[cat] = rate
    mlflow.log_metric(f"gatekeeper.category_hit_rate.{cat}", rate)

mlflow.log_metric("gatekeeper.mean_entity_count", crypto_gk["entity_count"].mean())

# ── Co-occurrence matrix ───────────────────────────────────────────────────
co_matrix = crypto_gk.select(categories).to_pandas().astype(int)
co_occurrence = co_matrix.T @ co_matrix
print("Category Co-occurrence Matrix:")
display(co_occurrence)

# ── text_window sensitivity: 500 vs 1000 chars ──────────────────────────
nlp = build_nlp()
expanded_results = []
for article in crypto_articles[:30]:  # subsample for speed
    headline = str(article["headline"]) if article["headline"] is not None else ""
    body = str(article["body"]) if article["body"] is not None else ""

    doc_500 = nlp(headline + " " + body[:500])
    doc_1000 = nlp(headline + " " + body[:1000])

    expanded_results.append(
        {
            "story_id": article["story_id"],
            "entities_500": len(doc_500.ents),
            "entities_1000": len(doc_1000.ents),
            "gained": len(doc_1000.ents) > len(doc_500.ents),
        }
    )

expanded_df = pl.DataFrame(expanded_results)
miss_rate = expanded_df["gained"].sum() / expanded_df.height
mlflow.log_metric("gatekeeper.text_window_miss_rate", miss_rate)

# ── Print summary ───────────────────────────────────────────────────────────
print("\nCategory Hit Rates (crypto articles):")
print(f"{'Category':<25} {'Hit Rate':>10}")
print("─" * 37)
for cat, rate in sorted(hit_rates.items(), key=lambda x: -x[1]):
    print(f"{cat:<25} {rate:>9.1%}")

print(f"\nMean entity count (crypto): {crypto_gk['entity_count'].mean():.2f}")
print(f"text_window miss rate (500→1000): {miss_rate:.1%}")

# Top 5 most entity-rich articles
top5 = gatekeeper_results.sort("entity_count", descending=True).head(5)
print("\nTop 5 entity-rich articles:")
display(top5.select("story_id", "headline", "entity_count", "is_impactful").to_pandas())

### Cell 1.4 `[python]` — Gatekeeper Visualizations

**3 Plotly charts (dark theme):**

1. **Entity count histogram** — `px.histogram(gatekeeper_results, x='entity_count', color='is_impactful')` with Plotly dark theme. Shows distribution of NER entity counts, split by impactful/non-impactful.

2. **Category coverage bar** — Horizontal bar chart showing hit rate (%) for each of 5 NER categories among crypto articles. Use bright colors: `#58A6FF` (asset), `#52C41A` (exchange), `#F0883E` (stablecoin), `#FF4D4F` (protocol), `#A371F7` (concept).

3. **spaCy vs RLIKE agreement matrix** — 2x2 confusion matrix heatmap (`px.imshow`). Rows = spaCy (pass/fail), Cols = RLIKE (pass/fail).

**All charts:** `log_plotly()` to MLflow.

In [0]:
# ── 1. Entity count histogram ─────────────────────────────────────────────────
gk_pd = gatekeeper_results.to_pandas()

fig1 = px.histogram(
    gk_pd,
    x="entity_count",
    color="is_impactful",
    title="NER Entity Count Distribution by Gatekeeper Decision",
    labels={"entity_count": "Entity Count", "is_impactful": "Crypto-Relevant"},
    color_discrete_map={True: "#52C41A", False: "#FF4D4F"},
    barmode="overlay",
    opacity=0.7,
    template="plotly_dark",
)
fig1.update_layout(
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
)
mlflow.log_figure(fig1, "gatekeeper_entity_histogram.html")
fig1.show()

# ── 2. Category coverage bar ──────────────────────────────────────────────────
cats = list(hit_rates.keys())
rates = [hit_rates[c] for c in cats]
cat_colors = ["#58A6FF", "#52C41A", "#F0883E", "#FF4D4F", "#A371F7"]

fig2 = go.Figure(
    go.Bar(
        y=[c.replace("has_", "").replace("_", " ").title() for c in cats],
        x=[r * 100 for r in rates],
        orientation="h",
        marker_color=cat_colors,
        text=[f"{r:.0%}" for r in rates],
        textposition="auto",
    )
)
fig2.update_layout(
    title="NER Category Hit Rates Among Crypto Articles",
    xaxis_title="Hit Rate (%)",
    template="plotly_dark",
    plot_bgcolor="#0D1117",
    paper_bgcolor="#161B22",
    font_color="#E6EDF3",
)
mlflow.log_figure(fig2, "gatekeeper_category_coverage.html")
fig2.show()

# ── 3. spaCy vs RLIKE confusion matrix ──────────────────────────────────────
parity_pd = parity_df.to_pandas()
confusion = pd.crosstab(
    parity_pd["spacy_pass"].map({True: "spaCy: Pass", False: "spaCy: Fail"}),
    parity_pd["sql_pass"].map({True: "RLIKE: Pass", False: "RLIKE: Fail"}),
)

fig3 = px.imshow(
    confusion.values,
    x=confusion.columns.tolist(),
    y=confusion.index.tolist(),
    text_auto=True,
    color_continuous_scale=[[0, "#0D1117"], [1, "#58A6FF"]],
    title="spaCy vs SQL RLIKE Gatekeeper Agreement Matrix",
    template="plotly_dark",
)
fig3.update_layout(
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
)
mlflow.log_figure(fig3, "gatekeeper_confusion_matrix.html")
fig3.show()

### Cell 1.5 `[python]` — Gatekeeper Assertions

**Hard gates:**
```python
assert build_nlp() is build_nlp(), "build_nlp must be cached (functools.cache)"
assert all(0.0 <= r['confidence'] <= 1.0 for r in results), "Confidence must be in [0, 1]"
assert agreement_rate >= 0.90, f"spaCy/RLIKE agreement {agreement_rate:.2%} < 90% threshold"
assert gatekeeper_results.filter(pl.col('is_impactful'))['entity_count'].min() >= 1, "Impactful articles must have >= 1 entity"
assert gatekeeper_results.filter(~pl.col('is_impactful'))['entity_count'].max() == 0, "Non-impactful must have 0 entities"
```

**Rationale:** Gatekeeper is the foundation — if it misclassifies, everything downstream is wrong.

In [0]:
# ── Gatekeeper Hard Gates ────────────────────────────────────────────────────

# 1. build_nlp() is cached (singleton)
assert build_nlp() is build_nlp(), "build_nlp must be cached (functools.cache)"

# 2. Confidence bounds
for r in gatekeeper_results.iter_rows(named=True):
    assert 0.0 <= r["confidence"] <= 1.0, f"Confidence {r['confidence']} out of [0, 1]"

# 3. spaCy/RLIKE agreement >= 90%
assert agreement_rate >= 0.90, f"spaCy/RLIKE agreement {agreement_rate:.2%} < 90% threshold"

# 4. Impactful articles must have >= 1 entity
impactful_min = gatekeeper_results.filter(pl.col("is_impactful"))["entity_count"].min()
assert impactful_min >= 1, f"Impactful articles must have >= 1 entity, got min={impactful_min}"

# 5. Non-impactful articles must have 0 entities
non_impactful_max = gatekeeper_results.filter(~pl.col("is_impactful"))["entity_count"].max()
assert non_impactful_max == 0, f"Non-impactful must have 0 entities, got max={non_impactful_max}"

print("✓ All gatekeeper assertions passed")
print(f"  • build_nlp() singleton cache: OK")
print(f"  • Confidence ∈ [0, 1]: OK")
print(f"  • spaCy/RLIKE agreement: {agreement_rate:.2%} ≥ 90%: OK")
print(f"  • Impactful min entities: {impactful_min} ≥ 1: OK")
print(f"  • Non-impactful max entities: {non_impactful_max} == 0: OK")

---
## Section 2: Context Retriever — V1 Stub Validation (2 cells)

**Component:** `context_retriever.py`  
**What it does:** V1 stub returns NULL for all 3 market context fields (`market_vol_at_publish`, `market_trend_at_publish`, `dollar_bar_close`). Full implementation blocked on TRD-5 (dollar bars).

**Key questions:**
1. Does the stub preserve all input state fields?
2. Are all 3 output fields exactly None (not empty string, not 0.0)?

### Cell 2.1 `[python]` — Context Retriever Stub Validation

**Logic:**
```python
for article in crypto_articles[:5]:
    state_in = {"headline": article["headline"], "body": article["body"], "published_at": str(article["published_at"])}
    state_out = context_retriever(state_in, config=config)
    # Verify: state_out contains all input keys + 3 new NULL keys
```

**Assertions:**
```python
assert state_out["market_vol_at_publish"] is None
assert state_out["market_trend_at_publish"] is None
assert state_out["dollar_bar_close"] is None
assert state_out["headline"] == state_in["headline"]  # passthrough
assert state_out["body"] == state_in["body"]           # passthrough
```

**Print:** "✓ Context retriever V1 stub: all 3 market fields = None, state passthrough correct"

In [0]:
# ── Context Retriever V1 Stub Validation ───────────────────────────────────
for i, article in enumerate(crypto_articles[:5]):
    state_in: MASState = {
        "headline": article["headline"],
        "body": article["body"],
        "published_at": str(article["published_at"]),
    }
    state_out = context_retriever(state_in, config=config)

    # Verify: 3 market fields are exactly None (not empty string, not 0.0)
    assert state_out["market_vol_at_publish"] is None, f"Article {i}: market_vol_at_publish should be None"
    assert state_out["market_trend_at_publish"] is None, f"Article {i}: market_trend_at_publish should be None"
    assert state_out["dollar_bar_close"] is None, f"Article {i}: dollar_bar_close should be None"

    # Verify: input state fields preserved (passthrough)
    assert state_out["headline"] == state_in["headline"], f"Article {i}: headline not passed through"
    assert state_out["body"] == state_in["body"], f"Article {i}: body not passed through"
    assert state_out["published_at"] == state_in["published_at"], f"Article {i}: published_at not passed through"

print("✓ Context retriever V1 stub: all 3 market fields = None, state passthrough correct")
print(f"  Validated on {min(5, len(crypto_articles))} crypto articles")
print(f"  Fields verified: market_vol_at_publish, market_trend_at_publish, dollar_bar_close")

---
## Section 3: Few-Shot Retriever — VS Bootstrap Flywheel (7 cells)

**Component:** `few_shot_retriever.py` + VS index `causal_chains_idx`  
**What it does:** Queries Vector Search for similar accepted causal chains. Three operational modes:

| Mode | Condition | Behavior |
| --- | --- | --- |
| **Cold Start** | 0 accepted chains / no gold table | 5 static examples (hard-coded) |
| **Bootstrapping** | 1-49 accepted chains | Static examples (below `min_index_size=50`) |
| **Self-Improving** | 50+ accepted chains | VS query → dynamic top-5, backfilled with static |

**Current status:** SELF-IMPROVING (118 ≥ 50 chains)

**Key questions:**
1. Does cold-start fallback work correctly?
2. Does threshold boundary (49 → 50) switch behavior?
3. Are VS results semantically relevant?
4. Does `merge_examples()` handle all edge cases?

### Cell 3.1 `[python]` — Cold-Start Path Test (Monkeypatch)

**Purpose:** Force cold-start mode and verify static examples are returned.

**Logic:**
1. Monkeypatch `few_shot_retriever.get_accepted_chain_count` to return 0
2. Monkeypatch `few_shot_retriever.get_spark_session` to return `spark` (real session)
3. Run `few_shot_retriever(state, config=config)` on 3 test articles
4. Restore original functions after test

**Assertions:**
* `len(state_out['few_shot_examples']) == 5`
* All 5 examples have `story_id` starting with `"static-example-"`
* `state_out['retrieval_story_ids']` contains exactly the 5 static IDs

**Print:** Table of 5 static examples: story_id, treatment_class, mechanism.

In [0]:
import importlib

# importlib avoids the __init__.py re-export shadowing the module name
fsr_module = importlib.import_module('trading_databricks.agents.nodes.few_shot_retriever')
from trading_databricks.agents.nodes.few_shot_retriever import (
    few_shot_retriever, STATIC_FEW_SHOT_EXAMPLES, merge_examples,
)

_orig_get_spark = fsr_module.get_spark_session
_orig_get_count = fsr_module.get_accepted_chain_count

try:
    fsr_module.get_spark_session = lambda: spark
    fsr_module.get_accepted_chain_count = lambda *a, **kw: 0

    cold_start_results = []
    for article in crypto_articles[:3]:
        state_in = {"headline": str(article["headline"] or ""), "body": str(article["body"] or ""), "published_at": str(article["published_at"])}
        state_out = few_shot_retriever(state_in, config=config)
        cold_start_results.append(state_out)

        assert len(state_out["few_shot_examples"]) == 5, f"Expected 5 examples, got {len(state_out['few_shot_examples'])}"
        assert all(ex["story_id"].startswith("static-example-") for ex in state_out["few_shot_examples"])
        assert state_out["retrieval_story_ids"] == []
        assert state_out["headline"] == state_in["headline"]
finally:
    fsr_module.get_spark_session = _orig_get_spark
    fsr_module.get_accepted_chain_count = _orig_get_count

print("✓ Cold-start path test passed (3 articles)")
print(f"\nStatic examples returned:")
for ex in cold_start_results[0]["few_shot_examples"]:
    print(f"  {ex['story_id']:<35} {ex['treatment_class']:<30} {ex['mechanism']}")

### Cell 3.2 `[python]` — Bootstrapping Boundary Test

**Purpose:** Test the 49 → 50 threshold boundary.

**Logic:**
1. Monkeypatch `get_accepted_chain_count` to return 49 → verify static examples
2. Monkeypatch `get_accepted_chain_count` to return 50 → verify VS query attempted
3. Monkeypatch `get_accepted_chain_count` to return 51 → verify VS query attempted

**Assertions:**
* At count=49: static examples returned (bootstrapping mode)
* At count=50: VS query fires (self-improving mode)

**Notes:** The threshold is `config.router.min_index_size = 50`. This is read by the few_shot_retriever node to decide whether to query VS or use static examples.

In [0]:
# ── Bootstrapping boundary: 49 → 50 threshold ────────────────────────────
# fsr_module already imported via importlib in Cell 3.1
_orig_get_spark = fsr_module.get_spark_session
_orig_get_count = fsr_module.get_accepted_chain_count
_orig_query_vs = fsr_module.query_vector_search

vs_call_log = []

def mock_query_vs(headline, *, index_name, num_results=5):
    vs_call_log.append(headline)
    return [
        {"story_id": f"mock-vs-{i}", "treatment_class": "Macro.rate_decision", "mechanism": "liquidity_contraction",
         "expected_direction": "bearish", "confidence": 0.8, "trigger_event": f"Mock event {i}",
         "causal_chain_json": "{}", "similarity_score": 0.9 - i * 0.1}
        for i in range(min(num_results, 3))
    ]

test_article = {"headline": str(crypto_articles[0]["headline"] or ""), "body": str(crypto_articles[0]["body"] or ""), "published_at": str(crypto_articles[0]["published_at"])}

try:
    fsr_module.get_spark_session = lambda: spark
    fsr_module.query_vector_search = mock_query_vs

    # Test count=49 → bootstrapping (static)
    vs_call_log.clear()
    fsr_module.get_accepted_chain_count = lambda *a, **kw: 49
    out_49 = few_shot_retriever(test_article, config=config)
    assert len(vs_call_log) == 0, "VS should NOT be called at count=49"
    assert all(ex["story_id"].startswith("static-example-") for ex in out_49["few_shot_examples"])

    # Test count=50 → self-improving (VS fires)
    vs_call_log.clear()
    fsr_module.get_accepted_chain_count = lambda *a, **kw: 50
    out_50 = few_shot_retriever(test_article, config=config)
    assert len(vs_call_log) == 1, f"VS should be called once at count=50, got {len(vs_call_log)}"
    assert any(ex["story_id"].startswith("mock-vs-") for ex in out_50["few_shot_examples"])

    # Test count=51 → self-improving (VS fires)
    vs_call_log.clear()
    fsr_module.get_accepted_chain_count = lambda *a, **kw: 51
    out_51 = few_shot_retriever(test_article, config=config)
    assert len(vs_call_log) == 1, "VS should be called once at count=51"

finally:
    fsr_module.get_spark_session = _orig_get_spark
    fsr_module.get_accepted_chain_count = _orig_get_count
    fsr_module.query_vector_search = _orig_query_vs

print("✓ Bootstrapping boundary test passed")
print(f"  count=49: {len(out_49['few_shot_examples'])} examples (all static)")
print(f"  count=50: {len(out_50['few_shot_examples'])} examples ({sum(1 for e in out_50['few_shot_examples'] if e['story_id'].startswith('mock-vs-'))} dynamic + {sum(1 for e in out_50['few_shot_examples'] if e['story_id'].startswith('static-'))} static)")
print(f"  count=51: {len(out_51['few_shot_examples'])} examples (VS fired)")

### Cell 3.3 `[python]` — Live VS Query Quality Test

**Purpose:** Query the real VS index with diverse headlines and measure retrieval quality.

**Logic:**
1. Select 10 diverse crypto headlines from `crypto_articles` (mix of Regulation, Macro, Adoption, Security, Supply themes)
2. For each headline, call VS `similarity_search()` directly via VectorSearchClient:
   ```python
   vsc = VectorSearchClient(disable_notice=True)
   idx = vsc.get_index(endpoint_name="mas-few-shot", index_name="smart-trading-dev.03_gold.causal_chains_idx")
   results = idx.similarity_search(query_text=headline, columns=[...], num_results=5, filters={"validation_status": ["accepted"]})
   ```
3. Collect: per-headline top-5 similarity scores, treatment_classes, mechanisms

**Variables created:**
* `vs_quality_df: pl.DataFrame` with columns: `query_headline`, `rank`, `similarity_score`, `treatment_class`, `mechanism`, `trigger_event`

**Metrics logged:** `retriever.mean_top1_similarity`, `retriever.mean_top5_similarity`, `retriever.unique_treatments_in_top5`

In [0]:
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Select 10 diverse crypto headlines
diverse_indices = list(range(0, min(100, len(crypto_articles)), 10))[:10]
diverse_headlines = [str(crypto_articles[i]["headline"] or "") for i in diverse_indices]

vs_records = []
for headline in diverse_headlines:
    try:
        resp = w.vector_search_indexes.query_index(
            index_name=config.vs_index_name,
            columns=["story_id", "treatment_class", "mechanism", "expected_direction", "confidence", "trigger_event"],
            query_text=headline,
            num_results=5,
            filters_json=json.dumps({"validation_status": ["accepted"]}),
        )
        if resp.result and resp.result.data_array:
            col_names = [c.name for c in resp.manifest.columns]
            for rank, row in enumerate(resp.result.data_array):
                record = dict(zip(col_names, row))
                record["query_headline"] = headline[:80]
                record["rank"] = rank + 1
                # Score is the last column in the manifest
                record["similarity_score"] = float(row[-1]) if row[-1] is not None else 0.0
                vs_records.append(record)
    except Exception as e:
        print(f"  VS query failed for: {headline[:60]}... → {e}")

vs_quality_df = pl.DataFrame(vs_records) if vs_records else pl.DataFrame({"query_headline": [], "rank": [], "similarity_score": []})

# Metrics
if vs_quality_df.height > 0:
    top1_df = vs_quality_df.filter(pl.col("rank") == 1)
    mean_top1 = top1_df["similarity_score"].mean()
    mean_top5 = vs_quality_df["similarity_score"].mean()
    unique_treatments = vs_quality_df["treatment_class"].n_unique() if "treatment_class" in vs_quality_df.columns else 0

    mlflow.log_metric("retriever.mean_top1_similarity", mean_top1)
    mlflow.log_metric("retriever.mean_top5_similarity", mean_top5)
    mlflow.log_metric("retriever.unique_treatments_in_top5", unique_treatments)

    print(f"VS Quality Summary ({vs_quality_df.height} total results across {len(diverse_headlines)} queries):")
    print(f"  Mean top-1 similarity: {mean_top1:.4f}")
    print(f"  Mean top-5 similarity: {mean_top5:.4f}")
    print(f"  Unique treatment classes: {unique_treatments}")
    display(vs_quality_df.head(15).to_pandas())
else:
    print("⚠ No VS results returned — VS endpoint may be unavailable")

### Cell 3.4 `[python]` — merge_examples() Unit Tests

**Purpose:** Validate the merging logic for dynamic + static examples.

**Test cases:**
1. `dynamic=7, static=5, max_total=5` → returns 5 dynamic (truncated), 0 static
2. `dynamic=3, static=5, max_total=5` → returns 3 dynamic + 2 static = 5
3. `dynamic=0, static=5, max_total=5` → returns 0 dynamic + 5 static = 5
4. `dynamic=0, static=0, max_total=5` → returns empty list
5. `dynamic=2, static=1, max_total=5` → returns 2 dynamic + 1 static = 3 (under max)

**Uses:** `from trading_databricks.agents.nodes.few_shot_retriever import merge_examples, STATIC_FEW_SHOT_EXAMPLES`

**Assertions:** All 5 test cases pass.

In [0]:
# ── merge_examples() unit tests ─────────────────────────────────────────────
def make_examples(n, prefix="dyn"):
    return [{"story_id": f"{prefix}-{i}", "treatment_class": f"Test.{i}", "mechanism": f"mech_{i}"} for i in range(n)]

static_5 = make_examples(5, prefix="static")

# Test 1: dynamic=7, max=5 → 5 dynamic (truncated)
result = merge_examples(make_examples(7), static_5, max_total=5)
assert len(result) == 5 and all(r["story_id"].startswith("dyn-") for r in result), f"Test 1 failed: {[r['story_id'] for r in result]}"

# Test 2: dynamic=3, max=5 → 3 dynamic + 2 static = 5
result = merge_examples(make_examples(3), static_5, max_total=5)
assert len(result) == 5, f"Test 2: expected 5, got {len(result)}"
assert sum(1 for r in result if r["story_id"].startswith("dyn-")) == 3
assert sum(1 for r in result if r["story_id"].startswith("static-")) == 2

# Test 3: dynamic=0, max=5 → 5 static
result = merge_examples([], static_5, max_total=5)
assert len(result) == 5 and all(r["story_id"].startswith("static-") for r in result), f"Test 3 failed"

# Test 4: dynamic=0, static=0, max=5 → empty
result = merge_examples([], [], max_total=5)
assert len(result) == 0, f"Test 4: expected 0, got {len(result)}"

# Test 5: dynamic=2, static=1, max=5 → 3 (under max)
result = merge_examples(make_examples(2), make_examples(1, prefix="static"), max_total=5)
assert len(result) == 3, f"Test 5: expected 3, got {len(result)}"

print("✓ All 5 merge_examples() tests passed")
print("  Test 1: dynamic overflow → truncated to max_total")
print("  Test 2: dynamic+static fill → exact max_total")
print("  Test 3: no dynamic → all static")
print("  Test 4: both empty → empty result")
print("  Test 5: total under max → returns all available")

### Cell 3.5 `[python]` — Retriever Visualizations

**2 Plotly charts (dark theme):**

1. **Similarity score box plot** — One box per query headline (top 10). Y-axis = cosine similarity. Shows spread of top-5 scores per query. Helps identify queries where VS struggles.

2. **Treatment class heatmap** — Rows = query headlines (truncated to 40 chars), Cols = unique treatment_classes from results. Cell color = similarity score. Shows whether VS retrieves semantically appropriate categories.

**Both:** `log_plotly()` to MLflow.

In [0]:
if vs_quality_df.height > 0:
    # 1. Similarity score box plot per query headline
    fig1 = px.box(
        vs_quality_df.to_pandas(), x="query_headline", y="similarity_score",
        title="VS Similarity Score Distribution per Query Headline",
        labels={"query_headline": "Query Headline", "similarity_score": "Cosine Similarity"},
        template="plotly_dark",
    )
    fig1.update_layout(
        plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
        xaxis_tickangle=-45, xaxis_tickfont_size=9,
    )
    mlflow.log_figure(fig1, "retriever_similarity_boxplot.html")
    fig1.show()

    # 2. Treatment class heatmap
    if "treatment_class" in vs_quality_df.columns:
        pivot = vs_quality_df.to_pandas().pivot_table(
            index="query_headline", columns="treatment_class", values="similarity_score", aggfunc="max",
        ).fillna(0)
        fig2 = px.imshow(
            pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
            title="Treatment Class × Query Headline Similarity Heatmap",
            color_continuous_scale=[[0, "#0D1117"], [0.5, "#58A6FF"], [1, "#52C41A"]],
            template="plotly_dark", aspect="auto",
        )
        fig2.update_layout(
            plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
            xaxis_tickangle=-45, xaxis_tickfont_size=9, yaxis_tickfont_size=9,
        )
        mlflow.log_figure(fig2, "retriever_treatment_heatmap.html")
        fig2.show()
else:
    print("⚠ Skipping visualizations — no VS results available")

### Cell 3.6 `[python]` — Few-Shot Retriever Assertions

**Hard gates:**
```python
assert STATIC_FEW_SHOT_EXAMPLES is not None and len(STATIC_FEW_SHOT_EXAMPLES) == 5
assert all('story_id' in ex and 'treatment_class' in ex for ex in STATIC_FEW_SHOT_EXAMPLES)
assert vs_quality_df['similarity_score'].mean() > 0.5, "Mean VS similarity too low"
assert vs_quality_df['similarity_score'].max() <= 1.0
assert vs_quality_df.filter(pl.col('rank') == 1)['similarity_score'].min() > 0.3, "Top-1 similarity below minimum threshold"
```

**Rationale:** If VS returns garbage (similarity < 0.3), few-shot examples won't help the LLM.

In [0]:
# ── Hard gates ──────────────────────────────────────────────────────────────────────
assert STATIC_FEW_SHOT_EXAMPLES is not None and len(STATIC_FEW_SHOT_EXAMPLES) == 5, f"Expected 5 static examples, got {len(STATIC_FEW_SHOT_EXAMPLES) if STATIC_FEW_SHOT_EXAMPLES else 'None'}"
assert all("story_id" in ex and "treatment_class" in ex for ex in STATIC_FEW_SHOT_EXAMPLES), "Static examples missing required fields"

if vs_quality_df.height > 0:
    assert vs_quality_df["similarity_score"].mean() > 0.5, f"Mean VS similarity {vs_quality_df['similarity_score'].mean():.4f} ≤ 0.5"
    assert vs_quality_df["similarity_score"].max() <= 1.0, f"Max similarity {vs_quality_df['similarity_score'].max():.4f} > 1.0"
    top1_min = vs_quality_df.filter(pl.col("rank") == 1)["similarity_score"].min()
    assert top1_min > 0.3, f"Top-1 min similarity {top1_min:.4f} ≤ 0.3"
    print(f"✓ All few-shot retriever assertions passed")
    print(f"  • Static examples: 5 valid entries")
    print(f"  • Mean VS similarity: {vs_quality_df['similarity_score'].mean():.4f} > 0.5")
    print(f"  • Top-1 min similarity: {top1_min:.4f} > 0.3")
else:
    print("⚠ VS assertions skipped (no VS results) — only static example assertions checked")
    print("✓ Static example assertions passed")

---
### Section 3.7: Formal Retrieval Metrics (P0 Enhancement)

**Objective 2.6:** *Use tools and metrics to evaluate retrieval performance*

**New module:** `eval/retrieval_scorers.py` — precision@k, recall@k, MRR, NDCG@k

**Approach:**
1. Build labeled query-document pairs from the VS quality data in Cell 3.3
2. For each query headline, define "relevant" results as those with matching L1 treatment category
3. Compute formal IR metrics using `evaluate_retrieval()`
4. Compare against a random-retrieval baseline
5. Log all metrics to MLflow

**Why this matters:** Raw similarity scores (Cell 3.5) tell us *how similar* results are, but not *how relevant*. A high-similarity result with the wrong treatment class is a false positive. Formal IR metrics quantify this.

In [0]:
# ── Formal retrieval metrics using eval/retrieval_scorers.py ─────────────
import importlib.util

# Direct file-based import — autoreload's custom finder blocks discovery of
# new modules added after notebook start. spec_from_file_location bypasses it.
_rs_path = (
    "/Workspace/Users/aaronng.dev1@gmail.com/trading-databricks"
    "/src/trading_databricks/agents/eval/retrieval_scorers.py"
)
_spec = importlib.util.spec_from_file_location(
    "trading_databricks.agents.eval.retrieval_scorers", _rs_path
)
_rs_mod = importlib.util.module_from_spec(_spec)
import sys as _sys
_sys.modules[_spec.name] = _rs_mod
_spec.loader.exec_module(_rs_mod)

from trading_databricks.agents.eval.retrieval_scorers import (
    evaluate_retrieval,
    summarize_retrieval,
    RetrievalMetricsConfig,
)
print(f"✓ retrieval_scorers loaded from {_rs_path}")

# ── Build labeled query-document pairs from VS quality data (Cell 3.3) ────
# Strategy: For each query headline, define "relevant" as VS results whose
# treatment_class shares the same L1 category as the majority of results.
# This is a proxy for relevance when we lack human labels.

labeled_queries: list[dict] = []

if vs_quality_df.height > 0:
    query_headlines = vs_quality_df["query_headline"].unique().to_list()

    for qh in query_headlines:
        q_results = vs_quality_df.filter(pl.col("query_headline") == qh).sort(
            "similarity_score", descending=True
        )

        # Extract L1 category from treatment_class (e.g., "Regulation.etf_ruling" → "Regulation")
        if "treatment_class" in q_results.columns:
            l1_categories = (
                q_results.filter(pl.col("treatment_class").is_not_null())
                .with_columns(
                    pl.col("treatment_class")
                    .str.split(".")
                    .list.first()
                    .alias("l1_category")
                )
            )

            # Majority L1 = the "expected" category for this query
            if l1_categories.height > 0:
                majority_l1 = (
                    l1_categories.group_by("l1_category")
                    .len()
                    .sort("len", descending=True)
                    .row(0)[0]
                )

                # Relevant = results with matching L1 category
                relevant_ids = (
                    l1_categories.filter(pl.col("l1_category") == majority_l1)["story_id"]
                    .to_list()
                )
            else:
                relevant_ids = []
        else:
            relevant_ids = q_results["story_id"].to_list()[:1]  # fallback: top-1 is relevant

        retrieved_ids = q_results["story_id"].to_list()
        sim_scores = q_results["similarity_score"].to_list()

        labeled_queries.append({
            "query_id": qh[:40],
            "query_text": qh,
            "retrieved_ids": retrieved_ids,
            "relevant_ids": relevant_ids,
            "similarity_scores": sim_scores,
        })

    # ── Compute formal IR metrics ────────────────────────────────────────────
    ir_config = RetrievalMetricsConfig(k=5, similarity_threshold=0.3)
    ir_metrics_df = evaluate_retrieval(labeled_queries, config=ir_config)
    ir_summary = summarize_retrieval(ir_metrics_df)

    # ── Log to MLflow ────────────────────────────────────────────────────────
    for metric_name, value in ir_summary.items():
        if value is not None:
            mlflow.log_metric(f"retrieval.{metric_name}", value)

    # ── Print summary ────────────────────────────────────────────────────────
    print("\n" + "═" * 60)
    print("  FORMAL RETRIEVAL METRICS (k=5)")
    print("═" * 60)
    for k, v in ir_summary.items():
        if v is not None:
            print(f"  {k:<25} {v:.4f}")
        else:
            print(f"  {k:<25} N/A")
    print("─" * 60)

    # ── Per-query detail ─────────────────────────────────────────────────────
    print("\nPer-Query Metrics:")
    print("-" * 90)
    print(f"{'Query':<42} {'P@5':>6} {'R@5':>6} {'MRR':>6} {'NDCG@5':>8}")
    print("-" * 90)
    for row in ir_metrics_df.iter_rows(named=True):
        print(
            f"{row['query_id']:<42} "
            f"{row['precision_at_k']:>6.3f} "
            f"{row['recall_at_k']:>6.3f} "
            f"{row['mrr']:>6.3f} "
            f"{row['ndcg_at_k']:>8.4f}"
        )
    print("-" * 90)

    # ── Random baseline comparison ───────────────────────────────────────────
    import random
    random.seed(42)
    all_story_ids = vs_quality_df["story_id"].unique().to_list()

    random_queries = []
    for q in labeled_queries:
        random_retrieved = random.sample(all_story_ids, min(5, len(all_story_ids)))
        random_queries.append({
            "query_id": q["query_id"],
            "retrieved_ids": random_retrieved,
            "relevant_ids": q["relevant_ids"],
        })

    random_metrics = summarize_retrieval(evaluate_retrieval(random_queries, config=ir_config))
    print("\nVS vs Random Baseline:")
    print(f"  {'Metric':<25} {'VS':>8} {'Random':>8} {'Lift':>8}")
    print("  " + "-" * 52)
    for key in ["mean_precision_at_k", "mean_recall_at_k", "mean_mrr", "mean_ndcg_at_k"]:
        vs_val = ir_summary.get(key, 0) or 0
        rand_val = random_metrics.get(key, 0) or 0
        lift = ((vs_val - rand_val) / rand_val * 100) if rand_val > 0 else float("inf")
        print(f"  {key:<25} {vs_val:>8.4f} {rand_val:>8.4f} {lift:>+7.1f}%")

    mlflow.log_metric("retrieval.random_baseline_mrr", random_metrics.get("mean_mrr", 0) or 0)
else:
    print("⚠ No VS quality data available — skipping formal retrieval metrics")
    ir_metrics_df = pl.DataFrame()
    ir_summary = {}

In [0]:
# ── Retrieval metrics visualization ──────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if ir_metrics_df.height > 0:
    ir_pd = ir_metrics_df.to_pandas()
    metric_cols = ["precision_at_k", "recall_at_k", "mrr", "ndcg_at_k"]

    # ── 1. Per-query radar chart ─────────────────────────────────────────────
    mean_vals = [ir_summary.get(f"mean_{c}", 0) or 0 for c in metric_cols]
    labels = ["Precision@5", "Recall@5", "MRR", "NDCG@5"]
    radar_labels = labels + [labels[0]]
    radar_vals = mean_vals + [mean_vals[0]]

    fig1 = go.Figure()
    fig1.add_trace(go.Scatterpolar(
        r=radar_vals,
        theta=radar_labels,
        fill="toself",
        name="VS Retrieval",
        line_color="#58A6FF",
        fillcolor="rgba(88,166,255,0.2)",
    ))

    # Add random baseline
    if random_metrics:
        rand_vals = [random_metrics.get(f"mean_{c}", 0) or 0 for c in metric_cols]
        rand_radar = rand_vals + [rand_vals[0]]
        fig1.add_trace(go.Scatterpolar(
            r=rand_radar,
            theta=radar_labels,
            fill="toself",
            name="Random Baseline",
            line_color="#FF4D4F",
            fillcolor="rgba(255,77,79,0.1)",
        ))

    fig1.update_layout(
        title="Retrieval Quality: VS vs Random Baseline",
        polar=dict(
            bgcolor="#0D1117",
            radialaxis=dict(visible=True, range=[0, 1], gridcolor="#21262D"),
            angularaxis=dict(gridcolor="#21262D"),
        ),
        plot_bgcolor="#0D1117",
        paper_bgcolor="#161B22",
        font_color="#E6EDF3",
        showlegend=True,
    )
    mlflow.log_figure(fig1, "retrieval_radar_vs_baseline.html")
    fig1.show()

    # ── 2. Per-query grouped bar chart ───────────────────────────────────────
    fig2 = make_subplots(rows=1, cols=1)
    colors = {"precision_at_k": "#58A6FF", "recall_at_k": "#52C41A", "mrr": "#F0883E", "ndcg_at_k": "#A371F7"}

    for col, color in colors.items():
        label = col.replace("_at_k", "@5").replace("_", " ").upper()
        fig2.add_trace(go.Bar(
            x=ir_pd["query_id"],
            y=ir_pd[col],
            name=label,
            marker_color=color,
            opacity=0.85,
        ))

    fig2.update_layout(
        title="Per-Query Retrieval Metrics",
        barmode="group",
        xaxis_title="Query",
        yaxis_title="Score",
        xaxis_tickangle=-45,
        xaxis_tickfont_size=8,
        plot_bgcolor="#0D1117",
        paper_bgcolor="#161B22",
        font_color="#E6EDF3",
        yaxis=dict(gridcolor="#21262D", range=[0, 1.05]),
    )
    mlflow.log_figure(fig2, "retrieval_per_query_metrics.html")
    fig2.show()
else:
    print("⚠ No retrieval metrics to visualize")

In [0]:
# ── Retrieval metrics assertions ─────────────────────────────────────────
if ir_metrics_df.height > 0:
    mean_mrr = ir_summary.get("mean_mrr", 0) or 0
    mean_ndcg = ir_summary.get("mean_ndcg_at_k", 0) or 0
    mean_precision = ir_summary.get("mean_precision_at_k", 0) or 0

    # VS must beat random baseline
    random_mrr = random_metrics.get("mean_mrr", 0) or 0
    assert mean_mrr > random_mrr, (
        f"VS MRR ({mean_mrr:.4f}) must beat random baseline ({random_mrr:.4f})"
    )

    # Minimum quality thresholds
    assert mean_mrr > 0.3, f"Mean MRR {mean_mrr:.4f} below 0.3 minimum"
    assert mean_ndcg > 0.3, f"Mean NDCG@5 {mean_ndcg:.4f} below 0.3 minimum"

    print("✓ Retrieval metrics assertions passed")
    print(f"  MRR: {mean_mrr:.4f} (random: {random_mrr:.4f}, lift: {(mean_mrr-random_mrr)/max(random_mrr,0.001)*100:+.1f}%)")
    print(f"  NDCG@5: {mean_ndcg:.4f}")
    print(f"  Precision@5: {mean_precision:.4f}")
else:
    print("⚠ Retrieval assertions skipped — no VS data")

---
## Section 4: Uncertainty Router — Extraction Path Selection (4 cells)

**Component:** `uncertainty_router.py`  
**What it does:** Reads `few_shot_examples` from state, examines top similarity score, routes to:
* `fast` (similarity ≥ 0.85) — high-confidence, minimal LLM compute
* `standard` (0.50 ≤ similarity < 0.85) — default path
* `careful` (similarity < 0.50 or insufficient examples) — max compute budget

**Config thresholds:**
* `high_similarity_threshold = 0.85`
* `low_similarity_threshold = 0.50`
* `min_vs_results = 2`

**Key questions:**
1. Do boundary values route correctly?
2. What is the path distribution for real articles?
3. Does empty/insufficient examples correctly route to `careful`?

### Cell 4.1 `[python]` — Synthetic Boundary Threshold Tests

**Purpose:** Test every boundary condition in the routing logic.

**Test cases:**
| # | Input | Expected Path | Why |
| --- | --- | --- | --- |
| 1 | `few_shot_examples = []` | `careful` | Empty examples |
| 2 | `[{similarity_score: 0.9}]` (1 example) | `careful` | Below `min_vs_results=2` |
| 3 | `[{0.9}, {0.8}]` (2 examples) | `fast` | Top=0.9 ≥ 0.85 |
| 4 | `[{0.85}, {0.7}]` | `fast` | Top=0.85 ≥ 0.85 (boundary) |
| 5 | `[{0.84}, {0.7}]` | `standard` | Top=0.84 < 0.85, ≥ 0.50 |
| 6 | `[{0.50}, {0.4}]` | `standard` | Top=0.50 ≥ 0.50 (boundary) |
| 7 | `[{0.49}, {0.3}]` | `careful` | Top=0.49 < 0.50 |
| 8 | `[{0.0}, {0.0}]` | `careful` | Zero similarity |

**Logic:** For each test case, construct `MASState` with synthetic `few_shot_examples`, call `uncertainty_router(state, config=config)`, verify `extraction_path`.

**Assertions:** All 8 test cases pass exactly.

In [0]:
from trading_databricks.agents.nodes.uncertainty_router import uncertainty_router

def make_state_with_examples(scores):
    examples = [{"story_id": f"test-{i}", "similarity_score": s} for i, s in enumerate(scores)]
    return {"headline": "test", "body": "test", "published_at": "2026-01-01", "few_shot_examples": examples}

test_cases = [
    ([], "careful", "empty examples"),
    ([0.9], "careful", "1 example < min_vs_results=2"),
    ([0.9, 0.8], "fast", "top=0.9 ≥ 0.85"),
    ([0.85, 0.7], "fast", "top=0.85 ≥ 0.85 (boundary)"),
    ([0.84, 0.7], "standard", "top=0.84 < 0.85, ≥ 0.50"),
    ([0.50, 0.4], "standard", "top=0.50 ≥ 0.50 (boundary)"),
    ([0.49, 0.3], "careful", "top=0.49 < 0.50"),
    ([0.0, 0.0], "careful", "zero similarity"),
]

print(f"{'#':<4} {'Scores':<20} {'Expected':<12} {'Actual':<12} {'Status':<8} {'Reason'}")
print("─" * 80)
for i, (scores, expected, reason) in enumerate(test_cases, 1):
    state = make_state_with_examples(scores)
    result = uncertainty_router(state, config=config)
    actual = result["extraction_path"]
    status = "✓ PASS" if actual == expected else "✗ FAIL"
    print(f"{i:<4} {str(scores):<20} {expected:<12} {actual:<12} {status:<8} {reason}")
    assert actual == expected, f"Test {i} failed: {actual} != {expected} ({reason})"

print(f"\n✓ All 8 boundary threshold tests passed")

### Cell 4.2 `[python]` — Router Distribution on Real Articles

**Purpose:** Run the full outer pipeline (context_retriever → few_shot_retriever → uncertainty_router) on all 120 crypto articles to see the real routing distribution.

**Logic:**
1. For each crypto article, pipe through the 3 outer nodes sequentially
2. Collect `extraction_path` for each
3. Build `router_results: pl.DataFrame` with `story_id`, `extraction_path`, `top_similarity`

**Metrics logged:**
* `router.fast_rate`, `router.standard_rate`, `router.careful_rate`

**Note:** This cell makes VS calls (one per article) but NO LLM calls.

In [0]:
# ── Monkeypatch for live pipeline ──────────────────────────────────────
# The module's query_vs_index is a placeholder. Replace query_vector_search
# with a real implementation using the Databricks SDK.

def _real_query_vs(headline, *, index_name, num_results=5):
    """Query VS via Databricks SDK (replaces the MCP placeholder)."""
    try:
        resp = w.vector_search_indexes.query_index(
            index_name=index_name,
            columns=["story_id", "treatment_class", "mechanism", "expected_direction",
                     "confidence", "trigger_event", "causal_chain_json"],
            query_text=headline,
            num_results=num_results,
            filters_json=json.dumps({"validation_status": ["accepted"]}),
        )
        if resp.result and resp.result.data_array:
            col_names = [c.name for c in resp.manifest.columns]
            results = []
            for row in resp.result.data_array:
                record = dict(zip(col_names, row))
                record["similarity_score"] = float(row[-1]) if row[-1] is not None else 0.0
                results.append(record)
            return results
    except Exception as e:
        import logging
        logging.getLogger(__name__).warning(f"VS SDK query failed: {e}")
    return []

_orig_get_spark = fsr_module.get_spark_session
_orig_query_vs = fsr_module.query_vector_search

try:
    fsr_module.get_spark_session = lambda: spark
    fsr_module.query_vector_search = _real_query_vs

    router_records = []
    total = len(crypto_articles)
    for i, article in enumerate(crypto_articles):
        if i % 20 == 0:
            print(f"  Processing {i}/{total}...")
        state = {
            "headline": str(article["headline"] or ""),
            "body": str(article["body"] or ""),
            "published_at": str(article["published_at"]),
        }
        # Pipe through outer pipeline: context → few_shot → router
        state = context_retriever(state, config=config)
        state = few_shot_retriever(state, config=config)
        router_out = uncertainty_router(state, config=config)

        top_sim = 0.0
        examples = state.get("few_shot_examples", [])
        if examples:
            top_sim = examples[0].get("similarity_score", 0.0)

        router_records.append({
            "story_id": article["story_id"],
            "headline": str(article["headline"] or "")[:80],
            "extraction_path": router_out["extraction_path"],
            "top_similarity": top_sim,
            "n_examples": len(examples),
        })
finally:
    fsr_module.get_spark_session = _orig_get_spark
    fsr_module.query_vector_search = _orig_query_vs

router_results = pl.DataFrame(router_records)

# Metrics
for path in ["fast", "standard", "careful"]:
    rate = router_results.filter(pl.col("extraction_path") == path).height / router_results.height
    mlflow.log_metric(f"router.{path}_rate", rate)

print(f"\nRouter distribution ({router_results.height} crypto articles):")
for path in ["fast", "standard", "careful"]:
    n = router_results.filter(pl.col("extraction_path") == path).height
    print(f"  {path:<10}: {n:>4} ({n/router_results.height:.1%})")
print(f"\nMean top similarity: {router_results['top_similarity'].mean():.4f}")
display(router_results.head(10).to_pandas())

### Cell 4.3 `[python]` — Router Visualizations + Assertions

**2 Plotly charts:**

1. **Extraction path distribution** — `px.pie(router_results, names='extraction_path')`. Shows what fraction of articles go through each path.

2. **Similarity vs Path scatter** — `px.strip(router_results, x='extraction_path', y='top_similarity')`. Jittered strip plot showing the similarity scores that led to each routing decision. Overlay threshold lines at 0.50 and 0.85.

**Assertions:**
```python
assert set(router_results['extraction_path'].unique().to_list()) <= {'fast', 'standard', 'careful'}
assert router_results.height == len(crypto_articles)
```

In [0]:
# 1. Extraction path pie chart
path_counts = router_results.group_by("extraction_path").len().to_pandas()
fig1 = px.pie(
    path_counts, names="extraction_path", values="len",
    title="Extraction Path Distribution (120 Crypto Articles)",
    color_discrete_map={"fast": "#52C41A", "standard": "#58A6FF", "careful": "#FF4D4F"},
    template="plotly_dark",
)
fig1.update_layout(plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3")
mlflow.log_figure(fig1, "router_path_distribution.html")
fig1.show()

# 2. Similarity vs path strip plot with threshold lines
fig2 = px.strip(
    router_results.to_pandas(), x="extraction_path", y="top_similarity",
    title="Top Similarity Score by Extraction Path",
    color="extraction_path",
    color_discrete_map={"fast": "#52C41A", "standard": "#58A6FF", "careful": "#FF4D4F"},
    template="plotly_dark",
)
fig2.add_hline(y=0.85, line_dash="dash", line_color="#F0883E", annotation_text="high threshold (0.85)")
fig2.add_hline(y=0.50, line_dash="dash", line_color="#A371F7", annotation_text="low threshold (0.50)")
fig2.update_layout(plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3")
mlflow.log_figure(fig2, "router_similarity_vs_path.html")
fig2.show()

# Assertions
valid_paths = {"fast", "standard", "careful"}
assert set(router_results["extraction_path"].unique().to_list()) <= valid_paths, "Invalid extraction paths found"
assert router_results.height == len(crypto_articles), f"Expected {len(crypto_articles)} rows, got {router_results.height}"
print(f"\n✓ Router assertions passed")
print(f"  • All paths valid: {set(router_results['extraction_path'].unique().to_list())}")
print(f"  • Row count: {router_results.height} == {len(crypto_articles)}")

---
## Section 5: Evidence Extractor — LLM Structured Extraction (5 cells)

**Component:** `evidence_extractor.py`  
**What it does:** First LLM call in the causality sub-pipeline. Extracts structured `(actor, action, object, span, timestamp_hint)` tuples from article text. Parse failures cascade into empty evidence for downstream nodes.

**LLM calls:** \~20 (one per article in the sample)  
**Prompt:** `prompts/evidence_extractor.yaml`  
**Temperature:** 0.0 (deterministic)  
**Max tokens:** 800

**Key questions:**
1. What fraction of articles parse successfully into valid JSON arrays?
2. How many evidence tuples per article on average?
3. Are `span` fields actually verbatim text from the article?
4. What does failure look like (malformed JSON, empty response)?

### Cell 5.1 `[python]` — Run Evidence Extractor on 20 Crypto Articles

**Purpose:** Isolate and evaluate the evidence extraction LLM call.

**Logic:**
1. Select first 20 crypto articles (sorted by published_at desc for recency)
2. For each article, construct minimal state: `{"headline": ..., "body": ...}`
3. Call `evidence_extractor(state, config=config)` directly
4. Collect: `evidence` (list[dict]), `evidence_status` (ok/empty/failed)
5. Measure wall-clock time per call

**Variables created:**
* `evidence_results: list[dict]` — each has: `story_id`, `headline`, `evidence`, `evidence_status`, `evidence_count`, `latency_ms`
* `evidence_df: pl.DataFrame`

**Print:** Summary: N ok / N empty / N failed, mean evidence count, mean latency.

In [0]:
import time
from trading_databricks.agents.nodes.evidence_extractor import evidence_extractor

# ── Run evidence extractor on first 20 crypto articles ─────────────────────
test_articles_20 = crypto_articles[:20]

evidence_results: list[dict] = []
for i, article in enumerate(test_articles_20):
    state_in: MASState = {
        "headline": str(article["headline"] or ""),
        "body": str(article["body"] or ""),
        "published_at": str(article["published_at"]),
    }

    t0 = time.perf_counter()
    try:
        state_out = evidence_extractor(state_in, config=config)
    except Exception as exc:
        state_out = {"evidence": [], "evidence_status": "failed"}
        print(f"  ⚠ Article {i} failed: {exc}")
    latency_ms = (time.perf_counter() - t0) * 1000

    evidence_results.append({
        "article_idx": i,
        "headline": state_in["headline"][:80],
        "evidence_status": state_out["evidence_status"],
        "evidence_count": len(state_out.get("evidence", [])),
        "evidence": state_out.get("evidence", []),
        "latency_ms": latency_ms,
    })

    if (i + 1) % 5 == 0:
        print(f"  Processed {i + 1}/20...")

# Build summary DataFrame (exclude nested evidence list for tabular view)
evidence_df = pl.DataFrame(
    [{k: v for k, v in r.items() if k != "evidence"} for r in evidence_results]
)

print(f"\n✓ Evidence extraction complete: {len(evidence_results)} articles")
status_counts = evidence_df.group_by("evidence_status").len().sort("evidence_status")
for row in status_counts.iter_rows(named=True):
    print(f"  {row['evidence_status']:<10} {row['len']:>3} ({row['len']/len(evidence_results):.0%})")
print(f"\nMedian latency: {evidence_df['latency_ms'].median():.0f}ms")
display(evidence_df.to_pandas())

In [0]:
# ── Evidence Quality Metrics ────────────────────────────────────────────────
ok_count = evidence_df.filter(pl.col("evidence_status") == "ok").height
total = evidence_df.height
parse_success_rate = ok_count / total if total > 0 else 0.0

# Mean evidence count (ok articles only)
ok_df = evidence_df.filter(pl.col("evidence_status") == "ok")
mean_evidence_count = ok_df["evidence_count"].mean() if ok_df.height > 0 else 0.0
max_evidence_count = ok_df["evidence_count"].max() if ok_df.height > 0 else 0
min_evidence_count = ok_df["evidence_count"].min() if ok_df.height > 0 else 0

# Field coverage across all extracted evidence tuples
all_evidence = [e for r in evidence_results for e in r["evidence"]]
required_fields = ["actor", "action", "object", "span", "timestamp_hint"]
field_coverage: dict[str, float] = {}
for field_name in required_fields:
    present = sum(1 for e in all_evidence if field_name in e and e[field_name])
    field_coverage[field_name] = present / len(all_evidence) if all_evidence else 0.0

# Latency stats
mean_latency = evidence_df["latency_ms"].mean()
p95_latency = evidence_df["latency_ms"].quantile(0.95)
p50_latency = evidence_df["latency_ms"].median()

# ── Log to MLflow ───────────────────────────────────────────────────────────
mlflow.log_metric("evidence.parse_success_rate", parse_success_rate)
mlflow.log_metric("evidence.mean_evidence_count", mean_evidence_count)
mlflow.log_metric("evidence.total_evidence_tuples", len(all_evidence))
mlflow.log_metric("evidence.mean_latency_ms", mean_latency)
mlflow.log_metric("evidence.p95_latency_ms", p95_latency)
for field_name, rate in field_coverage.items():
    mlflow.log_metric(f"evidence.field_coverage.{field_name}", rate)

# ── Print summary ───────────────────────────────────────────────────────────
print(f"Evidence Quality Metrics (n={total})")
print(f"{'─' * 50}")
print(f"  Parse success rate:  {parse_success_rate:.2%} ({ok_count}/{total})")
print(f"  Mean evidence count: {mean_evidence_count:.2f} (min={min_evidence_count}, max={max_evidence_count})")
print(f"  Total evidence tuples: {len(all_evidence)}")
print(f"  Latency — mean: {mean_latency:.0f}ms | p50: {p50_latency:.0f}ms | p95: {p95_latency:.0f}ms")
print(f"\nField coverage ({len(all_evidence)} tuples):")
for field_name in required_fields:
    bar = '█' * int(field_coverage[field_name] * 20)
    print(f"  {field_name:<18} {field_coverage[field_name]:>6.1%} {bar}")

In [0]:
# ── Display 3 ok + 2 non-ok articles for human review ──────────────────────
ok_articles = [r for r in evidence_results if r["evidence_status"] == "ok"][:3]
non_ok_articles = [r for r in evidence_results if r["evidence_status"] != "ok"][:2]
inspection_set = ok_articles + non_ok_articles

if not non_ok_articles:
    # All articles succeeded — just show 5 ok articles
    inspection_set = ok_articles[:5] if len(ok_articles) >= 5 else ok_articles
    print(f"Note: All articles parsed successfully. Showing {len(inspection_set)} ok examples.\n")

for r in inspection_set:
    print(f"{'═' * 80}")
    print(f"  Headline:  {r['headline']}")
    print(f"  Status:    {r['evidence_status']} | Count: {r['evidence_count']} | Latency: {r['latency_ms']:.0f}ms")
    if r["evidence"]:
        for j, ev in enumerate(r["evidence"]):
            print(f"  ── Evidence #{j + 1} {'─' * 40}")
            for k, v in ev.items():
                v_str = str(v)[:120] if v else "(empty)"
                print(f"     {k:<18} {v_str}")
    else:
        print("  (no evidence extracted)")
print(f"{'═' * 80}")

In [0]:
# ── 1. Evidence count histogram ─────────────────────────────────────────────
evidence_pd = evidence_df.to_pandas()

fig1 = px.histogram(
    evidence_pd, x="evidence_count", nbins=10,
    color="evidence_status",
    title="Evidence Count Distribution per Article",
    labels={"evidence_count": "Evidence Tuples per Article", "evidence_status": "Status"},
    color_discrete_map={"ok": "#52C41A", "empty": "#F0883E", "failed": "#FF4D4F"},
    barmode="overlay", opacity=0.7, template="plotly_dark",
)
fig1.update_layout(
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
)
mlflow.log_figure(fig1, "evidence_count_histogram.html")
fig1.show()

# ── 2. Evidence status pie ──────────────────────────────────────────────────
status_pd = evidence_df.group_by("evidence_status").len().to_pandas()
fig2 = px.pie(
    status_pd, names="evidence_status", values="len",
    title="Evidence Extraction Status Distribution",
    color="evidence_status",
    color_discrete_map={"ok": "#52C41A", "empty": "#F0883E", "failed": "#FF4D4F"},
    template="plotly_dark",
)
fig2.update_layout(
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
)
mlflow.log_figure(fig2, "evidence_status_pie.html")
fig2.show()

# ── Hard assertions ─────────────────────────────────────────────────────────
assert parse_success_rate >= 0.80, (
    f"Parse success rate {parse_success_rate:.2%} < 80% threshold"
)
assert mean_evidence_count >= 1.0, (
    f"Mean evidence count {mean_evidence_count:.2f} < 1.0 threshold"
)

print("✓ All evidence extractor assertions passed")
print(f"  • Parse success rate: {parse_success_rate:.2%} ≥ 80%")
print(f"  • Mean evidence count: {mean_evidence_count:.2f} ≥ 1.0")

### Cell 5.2 `[python]` — Evidence Quality Metrics

**Metrics to compute:**
1. **Parse success rate:** `evidence_status == 'ok'` / total
2. **Mean evidence count:** average `len(evidence)` for ok articles
3. **Field completeness:** For each evidence tuple, check presence of actor, action, object, span — compute per-field fill rate
4. **Span verbatim rate:** For each `span` field, check `span.casefold() in body.casefold()` — fraction of spans that are truly verbatim
5. **Span coverage:** What fraction of article body is covered by evidence spans (total span chars / body length)

**Metrics logged:**
* `evidence.parse_success_rate`, `evidence.mean_count`, `evidence.field_completeness.{field}`, `evidence.span_verbatim_rate`, `evidence.mean_latency_ms`

### Cell 5.3 `[python]` — Evidence Sample Inspection

**Purpose:** Display 5 raw evidence extractions for human review.

**Logic:**
1. Pick 3 articles with `evidence_status == 'ok'` and 2 with `evidence_status != 'ok'`
2. For each, display: headline (truncated 80 chars), body (truncated 200 chars), full evidence JSON (pretty-printed)
3. Highlight any `span` fields NOT found verbatim in the body (red text)

**Output format:** Structured print, not a DataFrame — evidence tuples are nested JSON.

**Notes:** This is a qualitative checkpoint. No metrics logged.

### Cell 5.4 `[python]` — Evidence Visualizations + Assertions

**2 Plotly charts:**

1. **Evidence count histogram** — `px.histogram(evidence_df, x='evidence_count', nbins=10)`. Shows how many tuples the LLM extracts per article.

2. **Evidence status pie** — `px.pie(evidence_df, names='evidence_status')`. ok vs empty vs failed.

**Assertions:**
```python
assert evidence_df.filter(pl.col('evidence_status') == 'ok').height / evidence_df.height >= 0.80, "Parse success rate < 80%"
assert evidence_df.filter(pl.col('evidence_status') == 'ok')['evidence_count'].mean() >= 1.0, "Mean evidence count < 1"
```

---
## Section 6: RAG Retriever — Precedent News + Market Outcomes (3 cells)

**Component:** `rag_retriever.py`  
**What it does:** Queries `news_firehose_idx` VS index for precedent articles, then joins each hit to OHLC outcomes (±24 hours). Provides historical market context to the hypothesis generator.

**Current status:** Both `_query_firehose()` and `_join_market_outcomes()` raise `NotImplementedError` — the firehose VS index is not yet provisioned. The node must degrade gracefully.

**No LLM calls.** Testing graceful degradation only.

**Key questions:**
1. Does the node return `precedent_status='unavailable'` without crashing?
2. Does `_build_query_text()` correctly combine headline + evidence span?
3. Is `precedents = []` on failure (not None or missing key)?

### Cell 6.1 `[python]` — RAG Retriever Graceful Degradation

**Logic:**
1. For 5 crypto articles, construct state with evidence from Section 5
2. Call `rag_retriever(state, config=config)` directly
3. Verify graceful fallback: no exceptions, correct return structure

**Assertions:**
```python
for result in rag_results:
    assert result['precedent_status'] == 'unavailable'
    assert result['precedents'] == []
    assert isinstance(result['precedents'], list)
```

**Print:** "✓ RAG retriever gracefully degrades: 5/5 articles returned precedent_status='unavailable'"

In [0]:
from trading_databricks.agents.nodes.rag_retriever import rag_retriever

# ── Test graceful degradation on 5 crypto articles ───────────────────────
rag_results = []
for i, article in enumerate(crypto_articles[:5]):
    # Construct state with evidence from Section 5 (reuse first article's evidence)
    sample_evidence = evidence_results[i]["evidence"] if i < len(evidence_results) else []
    state_in: MASState = {
        "headline": str(article["headline"] or ""),
        "body": str(article["body"] or ""),
        "published_at": str(article["published_at"]),
        "evidence": sample_evidence,
    }
    state_out = rag_retriever(state_in, config=config)
    rag_results.append(state_out)

    # Verify graceful fallback: no exceptions, correct return structure
    assert state_out["precedent_status"] == "unavailable", (
        f"Article {i}: expected 'unavailable', got '{state_out['precedent_status']}'"
    )
    assert state_out["precedents"] == [], (
        f"Article {i}: expected empty list, got {state_out['precedents']}"
    )
    assert isinstance(state_out["precedents"], list), (
        f"Article {i}: precedents should be list, got {type(state_out['precedents'])}"
    )

print("✓ RAG retriever gracefully degrades: 5/5 articles returned precedent_status='unavailable'")
print(f"  All returned: precedents=[], precedent_status='unavailable'")
print(f"  No exceptions raised despite NotImplementedError in _query_firehose()")

In [0]:
from trading_databricks.agents.nodes.rag_retriever import _build_query_text

# ── 6 test cases for query construction logic ───────────────────────────

# Test 1: no evidence → just headline
assert _build_query_text("headline", []) == "headline", "Test 1 failed: no evidence"

# Test 2: with span → headline\nspan
assert _build_query_text("headline", [{"span": "key event"}]) == "headline\nkey event", "Test 2 failed: with span"

# Test 3: empty span → just headline
assert _build_query_text("headline", [{"span": ""}]) == "headline", "Test 3 failed: empty span"

# Test 4: whitespace span → just headline
assert _build_query_text("headline", [{"span": "  "}]) == "headline", "Test 4 failed: whitespace span"

# Test 5: missing span key → just headline
assert _build_query_text("headline", [{}]) == "headline", "Test 5 failed: missing span key"

# Test 6: empty headline → \nevent
assert _build_query_text("", [{"span": "event"}]) == "\nevent", "Test 6 failed: empty headline"

print("✓ All 6 _build_query_text() tests passed")
print("  Test 1: no evidence → headline only")
print("  Test 2: with span → headline + newline + span")
print("  Test 3: empty span → headline only")
print("  Test 4: whitespace span → headline only")
print("  Test 5: missing span key → headline only")
print("  Test 6: empty headline → newline + span")

### Cell 6.2 `[python]` — _build_query_text() Unit Tests

**Purpose:** Validate the query construction logic.

**Test cases:**
```python
from trading_databricks.agents.nodes.rag_retriever import _build_query_text

assert _build_query_text("headline", []) == "headline"  # no evidence
assert _build_query_text("headline", [{"span": "key event"}]) == "headline\nkey event"  # with span
assert _build_query_text("headline", [{"span": ""}]) == "headline"  # empty span
assert _build_query_text("headline", [{"span": "  "}]) == "headline"  # whitespace span
assert _build_query_text("headline", [{}]) == "headline"  # missing span key
assert _build_query_text("", [{"span": "event"}]) == "\nevent"  # empty headline
```

---
## Section 7: Hypothesis Generator — Causal Chain Extraction (7 cells)

**Component:** `hypothesis_generator.py`  
**What it does:** Core LLM extraction. Consumes evidence tuples (from evidence_extractor) + precedents (from rag_retriever) + few-shot examples, produces a `CausalChain` extraction dict.

**LLM calls:** \~20-40 (one per article + potential parse retries)  
**Prompt:** `prompts/hypothesis_generator.yaml` (system + user template with taxonomy/whitelist/few-shot injection)  
**Temperature:** 0.0  
**Max tokens:** 1000

**Critical dependencies:**
* Taxonomy (34 L2 categories) injected into system prompt via `format_full_taxonomy()`
* Whitelist (40 L1,mechanism pairs) injected via `format_whitelist()`
* Few-shot examples injected via `format_few_shot()` (up to 5)
* Evidence block from evidence_extractor
* Precedent block from rag_retriever (currently empty)

**Key questions:**
1. What fraction of extractions parse into valid JSON dicts?
2. Do extractions conform to the CausalChain Pydantic schema?
3. Are `treatment_class` values found in the taxonomy?
4. Are `(L1, mechanism)` pairs found in the whitelist?
5. Are `evidence_spans` verbatim from the article body?
6. Does the parse-retry mechanism (stricter suffix) recover failures?

### Cell 7.1 `[python]` — Run Hypothesis Generator on 20 Crypto Articles

**Purpose:** Isolate hypothesis generator with real evidence + few-shot context.

**Logic:**
1. Use the 20 crypto articles from Section 5 (reuse `evidence_results`)
2. For each article, construct full state:
   ```python
   state = {
       "headline": article["headline"],
       "body": article["body"],
       "evidence": evidence_results[i]["evidence"],  # from Section 5
       "evidence_status": evidence_results[i]["evidence_status"],
       "precedents": [],  # rag_retriever unavailable
       "precedent_status": "unavailable",
       "few_shot_examples": few_shot_examples,  # from live VS or static
       "extraction_path": "standard",
       "reflection_count": 0,
       "judge_failures": None,
   }
   ```
3. Call `hypothesis_generator(state, config=config, taxonomy=taxonomy, whitelist=whitelist)`
4. Measure latency per call

**Variables created:**
* `hypothesis_results: list[dict]` — each has: `story_id`, `extraction`, `extraction_status`, `latency_ms`
* `hypothesis_df: pl.DataFrame`

In [0]:
import time
from trading_databricks.agents.nodes.few_shot_retriever import STATIC_FEW_SHOT_EXAMPLES
from trading_databricks.agents.nodes.hypothesis_generator import hypothesis_generator

# ── Run hypothesis generator on the same 20 crypto articles from Section 5 ──
hypothesis_results: list[dict] = []
for i, article in enumerate(test_articles_20):
    headline = str(article["headline"] or "")
    body = str(article["body"] or "")

    state_in: MASState = {
        "headline": headline,
        "body": body,
        "published_at": str(article["published_at"]),
        "evidence": evidence_results[i]["evidence"],
        "evidence_status": evidence_results[i]["evidence_status"],
        "precedents": [],  # RAG unavailable (Section 6)
        "precedent_status": "unavailable",
        "few_shot_examples": STATIC_FEW_SHOT_EXAMPLES,
        "extraction_path": "standard",
        "reflection_count": 0,
        "judge_failures": None,
    }

    t0 = time.perf_counter()
    result = hypothesis_generator(state_in, config, taxonomy=taxonomy, whitelist=whitelist)
    latency_ms = (time.perf_counter() - t0) * 1000

    hypothesis_results.append(
        {
            "article_idx": i,
            "headline": headline[:100],
            "body": body,  # keep full body for downstream checks
            "extraction": result.get("extraction"),
            "extraction_status": result.get("extraction_status"),
            "latency_ms": latency_ms,
        }
    )

    if (i + 1) % 5 == 0:
        print(f"  Processed {i + 1}/20...")

hypothesis_df = pl.DataFrame(
    [
        {k: v for k, v in r.items() if k not in ("extraction", "body")}
        for r in hypothesis_results
    ]
)

# ── Summary ─────────────────────────────────────────────────────────────────
status_counts = hypothesis_df.group_by("extraction_status").len()
print(f"\n✓ Hypothesis generation complete: {len(hypothesis_results)} articles")
for row in status_counts.iter_rows(named=True):
    print(f"  {row['extraction_status']:>25} {row['len']:>3} ({row['len'] / len(hypothesis_results):.0%})")

print(f"\nMedian latency: {hypothesis_df['latency_ms'].median():.0f}ms")
display(hypothesis_df.head(10).to_pandas())

### Cell 7.2 `[python]` — Parse Success + Schema Conformance

**Metrics:**
1. **Parse success rate:** `extraction_status in ('ok', 'ok_after_reflection')` / total
2. **Parse failure breakdown:** `ok` vs `parse_failed` vs `failed` counts
3. **CausalChain Pydantic conformance:** For each extraction, try `CausalChain.model_validate(extraction)` — what fraction pass?
4. **Per-field fill rate:** For ok extractions, what fraction have non-null values for each of: `treatment_class`, `trigger_event`, `mechanism`, `asset`, `direction`, `evidence_spans`, `reasoning`

**Metrics logged:** `hypothesis.parse_success_rate`, `hypothesis.schema_valid_rate`, `hypothesis.field_fill_rate.{field}`

In [0]:
# ── Parse success rate ───────────────────────────────────────────────────────────────
ok_statuses = {"ok", "ok_after_reflection"}
parse_ok = hypothesis_df.filter(pl.col("extraction_status").is_in(ok_statuses)).height
parse_rate = parse_ok / hypothesis_df.height
mlflow.log_metric("hypothesis.parse_success_rate", parse_rate)

# ── CausalChain Pydantic validation ─────────────────────────────────────────
# The hypothesis_generator LLM output uses different field names than
# the CausalChain Pydantic schema. Map before validation:
#   expected_direction -> direction
#   is_impactful       -> True (gatekeeper already filtered these)
#   asset              -> None (not produced by hypothesis_generator)
#   confidence, mechanism_candidate -> stripped (extra fields)
from pydantic import ValidationError

def _map_extraction_to_causal_chain(ext: dict) -> dict:
    """Normalize hypothesis_generator output to CausalChain field names."""
    return {
        "is_impactful": True,  # hypothesis_generator only runs on gatekeeper-passed articles
        "treatment_class": ext.get("treatment_class"),
        "trigger_event": ext.get("trigger_event"),
        "mechanism": ext.get("mechanism"),
        "asset": ext.get("asset"),  # None if absent
        "direction": ext.get("expected_direction") or ext.get("direction"),
        "evidence_spans": ext.get("evidence_spans"),
        "reasoning": ext.get("reasoning"),
    }

valid_count = 0
validation_errors: list[dict] = []
for r in hypothesis_results:
    ext = r.get("extraction")
    if ext is None:
        continue
    try:
        mapped = _map_extraction_to_causal_chain(ext)
        CausalChain.model_validate(mapped)
        valid_count += 1
    except ValidationError as e:
        validation_errors.append(
            {"headline": r["headline"][:60], "error": str(e)[:200]}
        )

schema_valid_rate = valid_count / max(parse_ok, 1)
mlflow.log_metric("hypothesis.schema_valid_rate", schema_valid_rate)

# ── Per-field fill rate ───────────────────────────────────────────────────────
# Check both native LLM field names and CausalChain-mapped names
fields = [
    "treatment_class", "trigger_event", "mechanism",
    "expected_direction", "evidence_spans", "reasoning",
    "confidence", "asset",
]
field_rates: dict[str, float] = {}
for field in fields:
    present = sum(
        1 for r in hypothesis_results
        if r.get("extraction") and r["extraction"].get(field) is not None
    )
    rate = present / max(parse_ok, 1)
    field_rates[field] = rate
    mlflow.log_metric(f"hypothesis.field_fill.{field}", rate)

# ── Print summary ───────────────────────────────────────────────────────────
print(f"Parse success rate: {parse_rate:.1%} ({parse_ok}/{hypothesis_df.height})")
print(f"Schema conformance: {schema_valid_rate:.1%} ({valid_count}/{parse_ok})")

if validation_errors:
    print(f"\nSchema validation errors ({len(validation_errors)}):")
    for ve in validation_errors[:5]:
        print(f"  {ve['headline']}")
        print(f"    {ve['error']}")

print(f"\n{'Field':<22} {'Fill Rate':>10}")
print("─" * 34)
for field, rate in sorted(field_rates.items(), key=lambda x: -x[1]):
    bar = "█" * int(rate * 20)
    print(f"{field:<22} {rate:>9.1%} {bar}")

### Cell 7.3 `[python]` — Taxonomy Compliance Check

**Purpose:** Does the LLM respect the 34 allowed treatment classes?

**Logic:**
1. For each extraction with non-null `treatment_class`:
   * Split into L1.L2
   * Check L1 in taxonomy keys
   * Check L2 in taxonomy[L1]
2. Log: valid count, invalid count, list of invalid treatment_classes

**Variables created:**
* `taxonomy_compliance_rate: float`
* `invalid_treatments: list[str]` — treatments the LLM hallucinated outside taxonomy

**Metrics logged:** `hypothesis.taxonomy_compliance_rate`

**Diagnostic:** Print the invalid treatments sorted by frequency — this reveals systematic LLM misunderstanding.

In [0]:
# ── Taxonomy compliance: treatment_class ∈ 34 allowed categories ──────────
taxonomy_valid = 0
taxonomy_invalid: list[dict] = []

for r in hypothesis_results:
    ext = r.get("extraction")
    if ext is None or not ext.get("treatment_class"):
        continue

    tc = ext["treatment_class"]
    parts = tc.split(".", maxsplit=1)

    if len(parts) == 2:
        l1, l2 = parts
        if l1 in taxonomy and l2 in taxonomy[l1]:
            taxonomy_valid += 1
        else:
            taxonomy_invalid.append(
                {"headline": r["headline"][:60], "treatment_class": tc, "l1": l1, "l2": l2}
            )
    else:
        taxonomy_invalid.append(
            {"headline": r["headline"][:60], "treatment_class": tc, "l1": tc, "l2": "(missing)"}
        )

total_with_tc = taxonomy_valid + len(taxonomy_invalid)
taxonomy_compliance = taxonomy_valid / max(total_with_tc, 1)
mlflow.log_metric("hypothesis.taxonomy_compliance", taxonomy_compliance)

# ── Summary ─────────────────────────────────────────────────────────────────
print(f"Taxonomy compliance: {taxonomy_compliance:.1%} ({taxonomy_valid}/{total_with_tc})")

if taxonomy_invalid:
    print(f"\nInvalid treatment classes ({len(taxonomy_invalid)}):")
    for inv in taxonomy_invalid[:10]:
        print(f"  {inv['treatment_class']:<40} | {inv['headline']}")
        # Show closest valid L2 for the given L1
        if inv["l1"] in taxonomy:
            valid_l2s = taxonomy[inv["l1"]]
            print(f"    Valid L2 options for '{inv['l1']}': {valid_l2s}")
        else:
            print(f"    '{inv['l1']}' is not a valid L1. Valid L1s: {sorted(taxonomy.keys())}")

# ── L1 category distribution ─────────────────────────────────────────────
l1_counts: dict[str, int] = {}
for r in hypothesis_results:
    ext = r.get("extraction")
    if ext and ext.get("treatment_class"):
        l1 = ext["treatment_class"].split(".", maxsplit=1)[0]
        l1_counts[l1] = l1_counts.get(l1, 0) + 1

print(f"\n{'L1 Category':<20} {'Count':>7}")
print("─" * 29)
for l1, count in sorted(l1_counts.items(), key=lambda x: -x[1]):
    print(f"{l1:<20} {count:>7}")

### Cell 7.4 `[python]` — Whitelist Compliance Check

**Purpose:** Does the LLM pick mechanisms from the 40 allowed (L1, mechanism) pairs?

**Logic:**
1. For each extraction: extract L1 from `treatment_class`, check `(L1, mechanism)` in whitelist
2. Log: valid count, invalid count
3. For invalid pairs, show: what the LLM produced vs what was available for that L1

**Variables created:**
* `whitelist_compliance_rate: float`
* `invalid_mechanisms: list[dict]` — each has: `treatment_class`, `mechanism_produced`, `valid_mechanisms_for_l1`

**Metrics logged:** `hypothesis.whitelist_compliance_rate`

**Cross-reference:** This directly predicts `mechanism_validator` rejection rate for `rejected_mechanism`.

In [0]:
# ── Whitelist compliance: (L1, mechanism) ∈ 40 allowed pairs ───────────
wl_valid = 0
wl_invalid: list[dict] = []

for r in hypothesis_results:
    ext = r.get("extraction")
    if ext is None or not ext.get("treatment_class") or not ext.get("mechanism"):
        continue

    l1 = ext["treatment_class"].split(".", maxsplit=1)[0]
    mechanism = ext["mechanism"]
    key = (l1, mechanism)

    if key in whitelist:
        wl_valid += 1
    else:
        wl_invalid.append(
            {
                "headline": r["headline"][:60],
                "l1": l1,
                "mechanism_produced": mechanism,
                "valid_mechanisms": [
                    m for (cat, m) in whitelist if cat == l1
                ],
            }
        )

total_with_mech = wl_valid + len(wl_invalid)
wl_compliance = wl_valid / max(total_with_mech, 1)
mlflow.log_metric("hypothesis.whitelist_compliance", wl_compliance)

# ── Summary ─────────────────────────────────────────────────────────────────
print(f"Whitelist compliance: {wl_compliance:.1%} ({wl_valid}/{total_with_mech})")

if wl_invalid:
    print(f"\nInvalid (L1, mechanism) pairs ({len(wl_invalid)}):")
    for inv in wl_invalid[:10]:
        print(f"  L1={inv['l1']!r}  mechanism={inv['mechanism_produced']!r}")
        print(f"    Valid mechanisms for '{inv['l1']}': {inv['valid_mechanisms']}")
        print(f"    Headline: {inv['headline']}")

### Cell 7.5 `[python]` — Evidence Spans Verbatim Check

**Purpose:** Are the `evidence_spans` in the CausalChain extraction actually present in the article body?

**Logic:**
1. For each extraction with non-null `evidence_spans`:
   * For each span: `span.casefold() in body.casefold()`
   * Article passes if ALL spans are verbatim
2. Also measure: per-span pass rate (more granular than per-article)

**Variables created:**
* `evidence_verbatim_article_rate: float` — fraction of articles where ALL spans are verbatim
* `evidence_verbatim_span_rate: float` — fraction of individual spans that are verbatim

**Metrics logged:** `hypothesis.evidence_verbatim_article_rate`, `hypothesis.evidence_verbatim_span_rate`

**Notes:** The LLM often paraphrases slightly. The casefold+strip normalization from `mechanism_validator.py` is applied here for consistency.

In [0]:
# ── Evidence spans verbatim check ──────────────────────────────────────────
evidence_article_results: list[dict] = []
total_spans = 0
verbatim_spans = 0

for r in hypothesis_results:
    ext = r.get("extraction")
    body = r.get("body", "")
    if ext is None or not ext.get("evidence_spans"):
        continue

    spans = ext["evidence_spans"]
    body_lower = body.casefold()
    article_total = len(spans)
    article_verbatim = sum(1 for s in spans if s and s.casefold() in body_lower)

    total_spans += article_total
    verbatim_spans += article_verbatim

    evidence_article_results.append(
        {
            "headline": r["headline"][:60],
            "span_count": article_total,
            "verbatim_count": article_verbatim,
            "verbatim_rate": article_verbatim / max(article_total, 1),
            "non_verbatim": [
                s[:80] for s in spans if s and s.casefold() not in body_lower
            ],
        }
    )

span_verbatim_rate = verbatim_spans / max(total_spans, 1)
article_perfect_rate = (
    sum(1 for a in evidence_article_results if a["verbatim_rate"] == 1.0)
    / max(len(evidence_article_results), 1)
)

mlflow.log_metric("hypothesis.span_verbatim_rate", span_verbatim_rate)
mlflow.log_metric("hypothesis.article_all_spans_verbatim", article_perfect_rate)

# ── Summary ─────────────────────────────────────────────────────────────────
print(f"Span-level verbatim rate: {span_verbatim_rate:.1%} ({verbatim_spans}/{total_spans})")
print(f"Article-level all-verbatim rate: {article_perfect_rate:.1%}")

# Show articles with non-verbatim spans
non_verbatim_articles = [a for a in evidence_article_results if a["verbatim_rate"] < 1.0]
if non_verbatim_articles:
    print(f"\nArticles with non-verbatim spans ({len(non_verbatim_articles)}):")
    for a in non_verbatim_articles[:5]:
        print(f"  {a['headline']} — {a['verbatim_count']}/{a['span_count']} verbatim")
        for span in a["non_verbatim"][:3]:
            print(f"    ✗ \"{span}\"")
else:
    print("\n✓ All evidence spans are verbatim from article bodies")

### Cell 7.6 `[python]` — Hypothesis Visualizations + Assertions

**3 Plotly charts:**

1. **Treatment class distribution** — Horizontal bar chart of treatment_class frequency. Group by L1 category, color by L1. Shows which causal categories the LLM favors.

2. **Extraction status pie** — ok vs parse_failed vs failed.

3. **Compliance rates bar** — Grouped bar: `[schema_valid, taxonomy_compliant, whitelist_compliant, evidence_verbatim]`. Each bar shows the rate. Overlay 0.5 and 0.9 threshold lines.

**Assertions:**
```python
assert hypothesis_df.filter(pl.col('extraction_status').is_in(['ok', 'ok_after_reflection'])).height / hypothesis_df.height >= 0.70, "Parse success < 70%"
```

In [0]:
# ── 1. Treatment class distribution ─────────────────────────────────────────
tc_data = []
for r in hypothesis_results:
    ext = r.get("extraction")
    if ext and ext.get("treatment_class"):
        parts = ext["treatment_class"].split(".", maxsplit=1)
        l1 = parts[0]
        l2 = parts[1] if len(parts) == 2 else "(missing)"
        tc_data.append({"L1": l1, "treatment_class": ext["treatment_class"]})

if tc_data:
    tc_df = pl.DataFrame(tc_data).group_by("treatment_class", "L1").len().sort("len", descending=True)
    fig1 = px.bar(
        tc_df.to_pandas(), y="treatment_class", x="len", color="L1",
        orientation="h", title="Treatment Class Distribution (Hypothesis Generator)",
        labels={"len": "Count", "treatment_class": "Treatment Class"},
        template="plotly_dark",
    )
    fig1.update_layout(
        plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
        yaxis={"categoryorder": "total ascending"},
        height=max(400, len(tc_data) * 25),
    )
    mlflow.log_figure(fig1, "hypothesis_treatment_distribution.html")
    fig1.show()

# ── 2. Field fill rate heatmap ─────────────────────────────────────────────
import numpy as np

fill_fields = list(field_rates.keys())
fill_values = [field_rates[f] for f in fill_fields]

fig2 = go.Figure(
    go.Heatmap(
        z=[fill_values],
        x=fill_fields,
        y=["Fill Rate"],
        colorscale=[[0, "#FF4D4F"], [0.5, "#F0883E"], [1.0, "#52C41A"]],
        text=[[f"{v:.0%}" for v in fill_values]],
        texttemplate="%{text}",
        textfont={"size": 14, "color": "#E6EDF3"},
        zmin=0, zmax=1,
    )
)
fig2.update_layout(
    title="CausalChain Field Fill Rates",
    template="plotly_dark",
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
    height=200,
)
mlflow.log_figure(fig2, "hypothesis_field_fill_rates.html")
fig2.show()

# ── 3. Latency box plot ─────────────────────────────────────────────────
fig3 = px.box(
    hypothesis_df.to_pandas(), y="latency_ms", color="extraction_status",
    title="Hypothesis Generator Latency by Status",
    labels={"latency_ms": "Latency (ms)"},
    color_discrete_map={"ok": "#52C41A", "ok_after_reflection": "#58A6FF", "parse_failed": "#FF4D4F", "failed": "#F0883E"},
    template="plotly_dark",
)
fig3.update_layout(
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
)
mlflow.log_figure(fig3, "hypothesis_latency.html")
fig3.show()

# ── Hard assertions ────────────────────────────────────────────────────
assert parse_rate >= 0.70, f"Parse success {parse_rate:.1%} < 70% threshold"
assert schema_valid_rate >= 0.80, f"Schema conformance {schema_valid_rate:.1%} < 80% threshold"
assert taxonomy_compliance >= 0.60, f"Taxonomy compliance {taxonomy_compliance:.1%} < 60% threshold"
assert span_verbatim_rate >= 0.50, f"Span verbatim rate {span_verbatim_rate:.1%} < 50% threshold"

print("\n✓ Section 7 assertions passed")
print(f"  Parse: {parse_rate:.1%} | Schema: {schema_valid_rate:.1%} | Taxonomy: {taxonomy_compliance:.1%} | Verbatim: {span_verbatim_rate:.1%}")

---
## Section 8: Judge Panel — 3 LLM Judges (6 cells)

**Component:** `judge_panel.py`  
**What it does:** Runs 3 independent LLM judges in sequence, aggregates via strict-AND:

| Judge | Prompt | Output | Pass Condition |
| --- | --- | --- | --- |
| `judge_direction` | `judge_direction.yaml` | "up" / "down" / "neutral" | Matches extraction.expected_direction (mapping: bullish→up, bearish→down, volatile→neutral) |
| `judge_mechanism` | `judge_mechanism.yaml` | "match" / "mismatch" | Starts with "match" |
| `judge_grounding` | `judge_grounding.yaml` | float [0,1] | ≥ `config.judge_panel.grounding_threshold` (0.5) |

**LLM calls:** \~60 (3 judges x 20 articles)  
**Strict-AND:** Panel passes only if ALL 3 judges pass.

**Key questions:**
1. What are the individual judge pass rates?
2. Which judge is the bottleneck (most frequent failure)?
3. Are the grounding scores calibrated well? (distribution vs threshold)
4. Do judges agree with each other (independence check)?

### Cell 8.1 `[python]` — Run Judge Panel on Hypothesis Outputs

**Purpose:** Run all 3 judges on every extraction from Section 7.

**Logic:**
1. For each hypothesis result with `extraction is not None`:
   ```python
   state = {
       "headline": article["headline"],
       "body": article["body"],
       "extraction": hypothesis_results[i]["extraction"],
   }
   judge_output = judge_panel(state, config=config)
   ```
2. Collect: `judge_results` (per-judge verdicts), `judge_failures` (failure reasons)
3. Measure latency per panel invocation

**Variables created:**
* `judge_results_list: list[dict]` — each has: `story_id`, `direction_passed`, `direction_score`, `mechanism_passed`, `mechanism_score`, `grounding_passed`, `grounding_score`, `all_passed`, `judge_failures`, `latency_ms`
* `judge_df: pl.DataFrame`

In [0]:
import time
from trading_databricks.agents.nodes.judge_panel import judge_panel

# ── Run judge panel on all articles with non-null extractions ─────────
judge_results_list: list[dict] = []
articles_with_extraction = [
    r for r in hypothesis_results if r.get("extraction") is not None
]

for i, r in enumerate(articles_with_extraction):
    state_in: MASState = {
        "headline": r["headline"],
        "body": r["body"],
        "extraction": r["extraction"],
    }

    t0 = time.perf_counter()
    judge_output = judge_panel(state_in, config)
    latency_ms = (time.perf_counter() - t0) * 1000

    jr = judge_output.get("judge_results", {})
    direction = jr.get("direction", {})
    mechanism = jr.get("mechanism", {})
    grounding = jr.get("grounding", {})

    all_passed = (
        direction.get("passed", False)
        and mechanism.get("passed", False)
        and grounding.get("passed", False)
    )

    judge_results_list.append(
        {
            "article_idx": r["article_idx"],
            "headline": r["headline"][:80],
            "direction_passed": direction.get("passed", False),
            "direction_score": direction.get("score"),
            "direction_reason": direction.get("reason", ""),
            "mechanism_passed": mechanism.get("passed", False),
            "mechanism_score": mechanism.get("score"),
            "mechanism_reason": mechanism.get("reason", ""),
            "grounding_passed": grounding.get("passed", False),
            "grounding_score": grounding.get("score"),
            "grounding_reason": grounding.get("reason", ""),
            "all_passed": all_passed,
            "judge_failures": judge_output.get("judge_failures"),
            "latency_ms": latency_ms,
        }
    )

    if (i + 1) % 5 == 0:
        print(f"  Processed {i + 1}/{len(articles_with_extraction)}...")

judge_df = pl.DataFrame(
    [{k: v for k, v in r.items() if k != "judge_failures"} for r in judge_results_list]
)

# ── Summary ─────────────────────────────────────────────────────────────────
panel_pass = judge_df.filter(pl.col("all_passed")).height
print(f"\n✓ Judge panel complete: {len(judge_results_list)} articles evaluated")
print(f"  Panel pass (strict-AND): {panel_pass}/{len(judge_results_list)} ({panel_pass / max(len(judge_results_list), 1):.1%})")
print(f"  Median latency: {judge_df['latency_ms'].median():.0f}ms")
display(judge_df.head(10).to_pandas())

### Cell 8.2 `[python]` — Per-Judge Pass Rates

**Metrics:**
1. `direction_pass_rate` — fraction where direction judge passed
2. `mechanism_pass_rate` — fraction where mechanism judge passed
3. `grounding_pass_rate` — fraction where grounding judge passed (score ≥ 0.5)
4. `panel_pass_rate` — fraction where ALL 3 passed (strict-AND)
5. `panel_fail_rate_by_judge` — when panel fails, which judge was responsible?

**Metrics logged:** `judge.direction_pass_rate`, `judge.mechanism_pass_rate`, `judge.grounding_pass_rate`, `judge.panel_pass_rate`

**Print:** Formatted table with rates + absolute counts.

In [0]:
# ── Per-judge pass rates ───────────────────────────────────────────────────
total_judged = judge_df.height

direction_pass_rate = judge_df["direction_passed"].sum() / max(total_judged, 1)
mechanism_pass_rate = judge_df["mechanism_passed"].sum() / max(total_judged, 1)
grounding_pass_rate = judge_df["grounding_passed"].sum() / max(total_judged, 1)
panel_pass_rate = judge_df["all_passed"].sum() / max(total_judged, 1)

mlflow.log_metric("judge.direction_pass_rate", direction_pass_rate)
mlflow.log_metric("judge.mechanism_pass_rate", mechanism_pass_rate)
mlflow.log_metric("judge.grounding_pass_rate", grounding_pass_rate)
mlflow.log_metric("judge.panel_pass_rate", panel_pass_rate)

# ── Identify bottleneck judge ───────────────────────────────────────────
judge_rates = {
    "direction": direction_pass_rate,
    "mechanism": mechanism_pass_rate,
    "grounding": grounding_pass_rate,
}
bottleneck = min(judge_rates, key=judge_rates.get)

print(f"Per-Judge Pass Rates (n={total_judged})")
print("─" * 45)
print(f"  {'Judge':<15} {'Pass Rate':>10} {'Passed':>8}")
print("─" * 45)
for judge, rate in sorted(judge_rates.items(), key=lambda x: -x[1]):
    passed_count = int(rate * total_judged)
    bar = "█" * int(rate * 20)
    print(f"  {judge:<15} {rate:>9.1%} {passed_count:>7}  {bar}")
print("─" * 45)
print(f"  {'panel (AND)':<15} {panel_pass_rate:>9.1%} {int(panel_pass_rate * total_judged):>7}")

print(f"\n⚠ Bottleneck judge: {bottleneck} ({judge_rates[bottleneck]:.1%})")

# ── Failure reason breakdown ───────────────────────────────────────────
print("\nDirection failure reasons:")
dir_failures = judge_df.filter(~pl.col("direction_passed"))
if dir_failures.height > 0:
    for row in dir_failures.select("headline", "direction_score", "direction_reason").head(5).iter_rows(named=True):
        print(f"  {row['headline'][:50]} | score={row['direction_score']} | {row['direction_reason'][:60]}")
else:
    print("  (none)")

print("\nMechanism failure reasons:")
mech_failures = judge_df.filter(~pl.col("mechanism_passed"))
if mech_failures.height > 0:
    for row in mech_failures.select("headline", "mechanism_score", "mechanism_reason").head(5).iter_rows(named=True):
        print(f"  {row['headline'][:50]} | score={row['mechanism_score']} | {row['mechanism_reason'][:60]}")
else:
    print("  (none)")

### Cell 8.3 `[python]` — Judge Disagreement Analysis

**Purpose:** Identify articles where judges disagree — these are the most informative for calibration.

**Logic:**
1. For each article, count how many judges passed (0, 1, 2, or 3)
2. Articles with 1 or 2 passing judges = disagreement cases
3. For disagreement cases, display: headline, which judges passed/failed, failure reasons
4. Compute pairwise judge agreement rates: direction↔mechanism, direction↔grounding, mechanism↔grounding

**Variables created:**
* `disagreement_articles: pl.DataFrame` — filtered to articles with mixed judge verdicts
* `pairwise_agreement: dict[str, float]` — 3 pairwise agreement rates

**Metrics logged:** `judge.pairwise_agreement.dir_mech`, `judge.pairwise_agreement.dir_ground`, `judge.pairwise_agreement.mech_ground`

In [0]:
# ── Count judges passed per article ───────────────────────────────────────
judge_df_ext = judge_df.with_columns(
    (pl.col("direction_passed").cast(pl.Int8)
     + pl.col("mechanism_passed").cast(pl.Int8)
     + pl.col("grounding_passed").cast(pl.Int8)).alias("judges_passed")
)

# Disagreement = 1 or 2 judges passed (mixed verdict)
disagreement_df = judge_df_ext.filter(
    pl.col("judges_passed").is_between(1, 2)
)

print(f"Judge verdict distribution:")
for count in range(4):
    n = judge_df_ext.filter(pl.col("judges_passed") == count).height
    label = {0: "All failed", 1: "1 passed (disagreement)", 2: "2 passed (disagreement)", 3: "All passed"}[count]
    print(f"  {count}/3 passed: {n:>3} articles — {label}")

print(f"\nDisagreement articles: {disagreement_df.height}/{judge_df_ext.height}")

# ── Pairwise judge agreement ───────────────────────────────────────────
pairs = [
    ("direction", "mechanism"),
    ("direction", "grounding"),
    ("mechanism", "grounding"),
]
pairwise_agreement: dict[str, float] = {}

for j1, j2 in pairs:
    agree = judge_df_ext.filter(
        pl.col(f"{j1}_passed") == pl.col(f"{j2}_passed")
    ).height
    rate = agree / max(judge_df_ext.height, 1)
    key = f"{j1[:3]}_{j2[:3]}"
    pairwise_agreement[key] = rate
    mlflow.log_metric(f"judge.pairwise_agreement.{key}", rate)

print(f"\nPairwise Agreement Rates:")
for pair_key, rate in pairwise_agreement.items():
    print(f"  {pair_key:<12} {rate:.1%}")

# ── Show disagreement details ─────────────────────────────────────────
if disagreement_df.height > 0:
    print(f"\nFirst 5 disagreement articles:")
    for row in disagreement_df.head(5).iter_rows(named=True):
        d = "✓" if row["direction_passed"] else "✗"
        m = "✓" if row["mechanism_passed"] else "✗"
        g = "✓" if row["grounding_passed"] else "✗"
        print(f"  [{d}{m}{g}] {row['headline'][:60]}")
        if not row["direction_passed"]:
            print(f"       dir: {row['direction_reason'][:80]}")
        if not row["mechanism_passed"]:
            print(f"       mech: {row['mechanism_reason'][:80]}")
        if not row["grounding_passed"]:
            print(f"       grnd: {row['grounding_reason'][:80]}")

### Cell 8.4 `[python]` — Grounding Score Calibration

**Purpose:** The grounding judge returns a float [0,1]. Is the threshold (0.5) well-calibrated?

**Logic:**
1. Extract all grounding scores from `judge_df`
2. Compute: mean, median, std, min, max
3. Count: scores in [0, 0.25), [0.25, 0.50), [0.50, 0.75), [0.75, 1.0]
4. Check: is there a natural gap around the threshold, or are scores clustered near it?

**Variables created:**
* `grounding_scores: pl.Series`

**Diagnostic insight:** If most scores cluster near the threshold (0.4-0.6), the judge is uncertain and the threshold may need adjustment. If there's a clear bimodal distribution, the threshold is well-placed.

In [0]:
# ── Extract grounding scores ─────────────────────────────────────────────
grounding_scores = judge_df.filter(
    pl.col("grounding_score").is_not_null()
)["grounding_score"].cast(pl.Float64)

threshold = config.judge_panel.grounding_threshold  # 0.5

if grounding_scores.len() > 0:
    mean_score = grounding_scores.mean()
    median_score = grounding_scores.median()
    std_score = grounding_scores.std()
    min_score = grounding_scores.min()
    max_score = grounding_scores.max()

    # Bucket distribution
    buckets = {
        "[0.00, 0.25)": grounding_scores.filter((grounding_scores >= 0) & (grounding_scores < 0.25)).len(),
        "[0.25, 0.50)": grounding_scores.filter((grounding_scores >= 0.25) & (grounding_scores < 0.50)).len(),
        "[0.50, 0.75)": grounding_scores.filter((grounding_scores >= 0.50) & (grounding_scores < 0.75)).len(),
        "[0.75, 1.00]": grounding_scores.filter((grounding_scores >= 0.75) & (grounding_scores <= 1.0)).len(),
    }

    # Near-threshold cluster (0.4–0.6)
    near_threshold = grounding_scores.filter(
        (grounding_scores >= 0.4) & (grounding_scores <= 0.6)
    ).len()
    near_threshold_pct = near_threshold / grounding_scores.len()

    mlflow.log_metric("judge.grounding_mean", mean_score)
    mlflow.log_metric("judge.grounding_median", median_score)
    mlflow.log_metric("judge.grounding_near_threshold_pct", near_threshold_pct)

    print(f"Grounding Score Distribution (n={grounding_scores.len()}, threshold={threshold})")
    print("─" * 50)
    print(f"  Mean:   {mean_score:.3f}")
    print(f"  Median: {median_score:.3f}")
    print(f"  Std:    {std_score:.3f}")
    print(f"  Range:  [{min_score:.3f}, {max_score:.3f}]")
    print(f"\nBucket distribution:")
    for bucket, count in buckets.items():
        bar = "█" * int(count / max(grounding_scores.len(), 1) * 30)
        print(f"  {bucket:<15} {count:>3} ({count / grounding_scores.len():.0%}) {bar}")
    print(f"\nNear threshold (0.4–0.6): {near_threshold}/{grounding_scores.len()} ({near_threshold_pct:.0%})")

    if near_threshold_pct > 0.5:
        print("⚠ WARNING: >50% scores near threshold — judge is uncertain, threshold may need tuning")
    else:
        print("✓ Scores show reasonable separation from threshold")
else:
    print("⚠ No grounding scores available (all judges may have been unavailable)")

### Cell 8.5 `[python]` — Judge Visualizations + Assertions

**3 Plotly charts:**

1. **Per-judge pass rate bar** — Grouped bar with 3 judges + panel (strict-AND). Color: pass=`#52C41A`, fail=`#FF4D4F`.

2. **Grounding score histogram** — `px.histogram(grounding_scores, nbins=20)`. Overlay vertical line at threshold=0.5. Shows calibration.

3. **Judge agreement heatmap** — 3x3 symmetric heatmap of pairwise agreement rates. Diagonal = 1.0.

**Assertions:**
```python
assert judge_df['all_passed'].mean() >= 0.20, "Panel pass rate < 20% — judges may be too strict"
assert judge_df['all_passed'].mean() <= 0.95, "Panel pass rate > 95% — judges may be too lenient"
```

In [0]:
import numpy as np
from plotly.subplots import make_subplots

# ── 1. Per-judge pass rate bar chart ──────────────────────────────────────
judge_names = ["direction", "mechanism", "grounding", "panel (AND)"]
pass_rates = [direction_pass_rate, mechanism_pass_rate, grounding_pass_rate, panel_pass_rate]
colors = ["#58A6FF", "#F0883E", "#A371F7", "#52C41A" if panel_pass_rate >= 0.3 else "#FF4D4F"]

fig1 = go.Figure(
    go.Bar(
        x=judge_names, y=pass_rates,
        marker_color=colors,
        text=[f"{r:.0%}" for r in pass_rates],
        textposition="outside",
        textfont={"color": "#E6EDF3", "size": 14},
    )
)
fig1.update_layout(
    title="Judge Pass Rates (Strict-AND Panel)",
    yaxis_title="Pass Rate", yaxis_range=[0, 1.1],
    template="plotly_dark",
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
)
mlflow.log_figure(fig1, "judge_pass_rates.html")
fig1.show()

# ── 2. Grounding score histogram ────────────────────────────────────────
if grounding_scores.len() > 0:
    fig2 = px.histogram(
        x=grounding_scores.to_list(), nbins=20,
        title="Grounding Score Distribution vs Threshold",
        labels={"x": "Grounding Score", "y": "Count"},
        template="plotly_dark",
        color_discrete_sequence=["#A371F7"],
    )
    fig2.add_vline(
        x=threshold, line_dash="dash", line_color="#FF4D4F",
        annotation_text=f"threshold={threshold}",
        annotation_font_color="#FF4D4F",
    )
    fig2.update_layout(
        plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
    )
    mlflow.log_figure(fig2, "judge_grounding_histogram.html")
    fig2.show()

# ── 3. Judge agreement heatmap ──────────────────────────────────────────
judge_labels = ["direction", "mechanism", "grounding"]
agreement_matrix = np.ones((3, 3))
for i, j1 in enumerate(judge_labels):
    for j, j2 in enumerate(judge_labels):
        if i != j:
            key = f"{j1[:3]}_{j2[:3]}"
            agreement_matrix[i][j] = pairwise_agreement.get(key, pairwise_agreement.get(f"{j2[:3]}_{j1[:3]}", 0.0))

fig3 = go.Figure(
    go.Heatmap(
        z=agreement_matrix,
        x=judge_labels, y=judge_labels,
        colorscale=[[0, "#FF4D4F"], [0.5, "#F0883E"], [1.0, "#52C41A"]],
        text=[[f"{v:.0%}" for v in row] for row in agreement_matrix],
        texttemplate="%{text}",
        textfont={"size": 16, "color": "#E6EDF3"},
        zmin=0, zmax=1,
    )
)
fig3.update_layout(
    title="Judge Pairwise Agreement Matrix",
    template="plotly_dark",
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
    height=400,
)
mlflow.log_figure(fig3, "judge_agreement_matrix.html")
fig3.show()

# ── Hard assertions ────────────────────────────────────────────────────
assert panel_pass_rate >= 0.20, f"Panel pass rate {panel_pass_rate:.1%} < 20% — judges may be too strict"
assert panel_pass_rate <= 0.95, f"Panel pass rate {panel_pass_rate:.1%} > 95% — judges may be too lenient"

print("\n✓ Section 8 assertions passed")
print(f"  Panel pass rate: {panel_pass_rate:.1%} (within [20%, 95%] bounds)")

---
## Section 9: Reflector — Bounded Reflection Loop (4 cells)

**Component:** `reflector.py`  
**What it does:** Pure routing logic. Reads `judge_results` + `reflection_count`, decides:
* All judges passed → `mechanism_validator` (proceed)
* Judges failed + budget available (`reflection_count < max_reflections=1`) → `hypothesis_generator` (retry)
* Judges failed + budget exhausted → `mechanism_validator` + `extraction_status='judge_failed_accepted'`
* All judges unavailable (`extraction_status='unjudged'`) → `mechanism_validator` (proceed without judgment)

**No LLM calls.** Pure conditional routing.

**Key questions:**
1. Does every routing scenario produce the correct `_route` value?
2. From the live data: what fraction of articles triggered reflection?
3. Did reflection actually improve extraction quality?

### Cell 9.1 `[python]` — Exhaustive Routing Scenario Tests

**Purpose:** Test every code path in the reflector with synthetic state.

**Test cases:**
| # | judge_results | judge_failures | reflection_count | extraction_status | Expected _route | Expected status |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | All passed | None | 0 | ok | mechanism_validator | (unchanged) |
| 2 | direction failed | {direction: ...} | 0 | ok | hypothesis_generator | (unchanged) |
| 3 | direction failed | {direction: ...} | 1 | ok | mechanism_validator | judge_failed_accepted |
| 4 | All failed | {dir, mech, ground} | 0 | ok | hypothesis_generator | (unchanged) |
| 5 | All failed | {dir, mech, ground} | 1 | ok | mechanism_validator | judge_failed_accepted |
| 6 | Empty (unavailable) | None | 0 | unjudged | mechanism_validator | (unchanged) |
| 7 | All passed | None | 1 | ok_after_reflection | mechanism_validator | (unchanged) |

**Logic:** For each case, construct `MASState` dict, call `reflector(state, config=config)`, verify `_route` and `extraction_status`.

**Assertions:** All 7 test cases pass exactly.

**Implementation note:** Use `REFLECTOR_ROUTE_VALIDATOR` and `REFLECTOR_ROUTE_RETRY` constants from `nodes/__init__.py` for comparison.

In [0]:
from trading_databricks.agents.nodes.reflector import (
    reflector,
    REFLECTOR_ROUTE_VALIDATOR,
    REFLECTOR_ROUTE_RETRY,
)

# ── 7 test cases covering every reflector code path ──────────────────
test_cases = [
    {
        "name": "1. All judges passed, first attempt",
        "state": {
            "judge_results": {
                "direction": {"passed": True, "score": "up", "reason": ""},
                "mechanism": {"passed": True, "score": "match", "reason": ""},
                "grounding": {"passed": True, "score": 0.8, "reason": ""},
            },
            "judge_failures": None,
            "reflection_count": 0,
            "extraction_status": "ok",
        },
        "expected_route": REFLECTOR_ROUTE_VALIDATOR,
        "expected_status": None,  # not overwritten
    },
    {
        "name": "2. Direction failed, budget available",
        "state": {
            "judge_results": {
                "direction": {"passed": False, "score": "down", "reason": "mismatch"},
                "mechanism": {"passed": True, "score": "match", "reason": ""},
                "grounding": {"passed": True, "score": 0.7, "reason": ""},
            },
            "judge_failures": {"direction": "direction mismatch"},
            "reflection_count": 0,
            "extraction_status": "ok",
        },
        "expected_route": REFLECTOR_ROUTE_RETRY,
        "expected_status": None,
    },
    {
        "name": "3. Direction failed, budget exhausted",
        "state": {
            "judge_results": {
                "direction": {"passed": False, "score": "down", "reason": "mismatch"},
                "mechanism": {"passed": True, "score": "match", "reason": ""},
                "grounding": {"passed": True, "score": 0.7, "reason": ""},
            },
            "judge_failures": {"direction": "direction mismatch"},
            "reflection_count": 1,
            "extraction_status": "ok",
        },
        "expected_route": REFLECTOR_ROUTE_VALIDATOR,
        "expected_status": "judge_failed_accepted",
    },
    {
        "name": "4. All judges failed, budget available",
        "state": {
            "judge_results": {
                "direction": {"passed": False, "score": "neutral", "reason": "mismatch"},
                "mechanism": {"passed": False, "score": "mismatch", "reason": "mismatch"},
                "grounding": {"passed": False, "score": 0.2, "reason": "low score"},
            },
            "judge_failures": {"direction": "mismatch", "mechanism": "mismatch", "grounding": "low"},
            "reflection_count": 0,
            "extraction_status": "ok",
        },
        "expected_route": REFLECTOR_ROUTE_RETRY,
        "expected_status": None,
    },
    {
        "name": "5. All judges failed, budget exhausted",
        "state": {
            "judge_results": {
                "direction": {"passed": False, "score": "neutral", "reason": "mismatch"},
                "mechanism": {"passed": False, "score": "mismatch", "reason": "mismatch"},
                "grounding": {"passed": False, "score": 0.2, "reason": "low score"},
            },
            "judge_failures": {"direction": "mismatch", "mechanism": "mismatch", "grounding": "low"},
            "reflection_count": 1,
            "extraction_status": "ok",
        },
        "expected_route": REFLECTOR_ROUTE_VALIDATOR,
        "expected_status": "judge_failed_accepted",
    },
    {
        "name": "6. All judges unavailable (unjudged)",
        "state": {
            "judge_results": {},
            "judge_failures": None,
            "reflection_count": 0,
            "extraction_status": "unjudged",
        },
        "expected_route": REFLECTOR_ROUTE_VALIDATOR,
        "expected_status": None,
    },
    {
        "name": "7. All passed after reflection",
        "state": {
            "judge_results": {
                "direction": {"passed": True, "score": "up", "reason": ""},
                "mechanism": {"passed": True, "score": "match", "reason": ""},
                "grounding": {"passed": True, "score": 0.9, "reason": ""},
            },
            "judge_failures": None,
            "reflection_count": 1,
            "extraction_status": "ok_after_reflection",
        },
        "expected_route": REFLECTOR_ROUTE_VALIDATOR,
        "expected_status": None,
    },
]

# ── Run all test cases ─────────────────────────────────────────────────
all_passed = True
for tc in test_cases:
    result = reflector(tc["state"], config)
    route = result.get("_route")
    status = result.get("extraction_status")

    route_ok = route == tc["expected_route"]
    status_ok = (
        tc["expected_status"] is None  # no status expectation
        or status == tc["expected_status"]
    )

    passed = route_ok and status_ok
    if not passed:
        all_passed = False
    icon = "✓" if passed else "✗"
    print(f"  {icon} {tc['name']}")
    if not route_ok:
        print(f"    ROUTE: expected={tc['expected_route']!r}, got={route!r}")
    if not status_ok:
        print(f"    STATUS: expected={tc['expected_status']!r}, got={status!r}")

assert all_passed, "Not all reflector routing tests passed"
print(f"\n✓ All {len(test_cases)} reflector routing scenarios passed")

### Cell 9.2 `[python]` — Reflection Rate + Quality Impact

**Purpose:** Using judge outputs from Section 8, compute how many articles would trigger reflection.

**Logic:**
1. For each article in judge_results:
   * If `judge_failures is not None` (any judge failed) → would trigger reflection
   * Count: `reflection_needed`, `no_reflection_needed`, `unjudged`
2. **Quality impact** (requires re-running hypothesis_generator on reflected articles):
   * For articles where reflection was triggered, compare first-attempt extraction vs re-attempted
   * Metrics: did taxonomy compliance improve? Did whitelist compliance improve?
   * **Note:** This cell makes LLM calls only if reflection actually fires (\~N articles)

**Metrics logged:** `reflector.reflection_rate`, `reflector.quality_improvement_rate` (fraction of reflected articles that improved)

**Print:** Summary table: N total, N reflected, N improved after reflection.

In [0]:
# ── Compute reflection rate from live judge outputs ───────────────────
reflection_needed = 0
no_reflection_needed = 0
unjudged_count = 0

for r in judge_results_list:
    if r.get("judge_failures") is not None and len(r["judge_failures"]) > 0:
        reflection_needed += 1
    elif not r.get("all_passed") and r.get("judge_failures") is None:
        # All judges unavailable
        unjudged_count += 1
    else:
        no_reflection_needed += 1

total_live = len(judge_results_list)
reflection_rate = reflection_needed / max(total_live, 1)

mlflow.log_metric("reflector.reflection_rate", reflection_rate)
mlflow.log_metric("reflector.unjudged_rate", unjudged_count / max(total_live, 1))

print(f"Reflection Analysis (n={total_live} articles from live judge data)")
print("─" * 55)
print(f"  No reflection needed (all passed):  {no_reflection_needed:>3} ({no_reflection_needed / max(total_live, 1):.0%})")
print(f"  Reflection would trigger:           {reflection_needed:>3} ({reflection_rate:.0%})")
print(f"  Unjudged (judges unavailable):      {unjudged_count:>3} ({unjudged_count / max(total_live, 1):.0%})")

# ── Failure cause breakdown for reflected articles ───────────────────
reflection_causes: dict[str, int] = {}
for r in judge_results_list:
    failures = r.get("judge_failures")
    if failures and len(failures) > 0:
        for judge_name in failures:
            reflection_causes[judge_name] = reflection_causes.get(judge_name, 0) + 1

if reflection_causes:
    print(f"\nReflection trigger causes:")
    for cause, count in sorted(reflection_causes.items(), key=lambda x: -x[1]):
        print(f"  {cause:<15} {count:>3} ({count / reflection_needed:.0%} of reflected)")

# ── Simulate reflection loop routing ───────────────────────────────
print(f"\nRouting simulation (max_reflections={config.reflection.max_reflections}):")
route_validator = 0
route_retry = 0

for r in judge_results_list:
    state_sim: MASState = {
        "judge_results": {  # reconstruct minimal judge_results dict
            "direction": {"passed": r["direction_passed"]},
            "mechanism": {"passed": r["mechanism_passed"]},
            "grounding": {"passed": r["grounding_passed"]},
        },
        "judge_failures": r.get("judge_failures"),
        "reflection_count": 0,  # simulate first pass
        "extraction_status": "ok",
    }
    result = reflector(state_sim, config)
    if result["_route"] == REFLECTOR_ROUTE_VALIDATOR:
        route_validator += 1
    else:
        route_retry += 1

print(f"  → mechanism_validator: {route_validator} ({route_validator / max(total_live, 1):.0%})")
print(f"  → hypothesis_generator (retry): {route_retry} ({route_retry / max(total_live, 1):.0%})")

### Cell 9.3 `[python]` — Reflector Assertions

**Assertions:**
```python
assert config.reflection.max_reflections == 1, "Expected max_reflections=1 for V1"
assert reflection_rate <= 0.80, f"Reflection rate {reflection_rate:.2%} is too high — judges may be miscalibrated"
```

**Rationale:** If > 80% of articles need reflection, the hypothesis generator prompt or the judges need tuning — reflection should be the exception, not the norm.

In [0]:
# ── Reflector hard gates ─────────────────────────────────────────────────
assert config.reflection.max_reflections == 1, (
    f"Expected max_reflections=1 for V1, got {config.reflection.max_reflections}"
)
assert reflection_rate <= 0.80, (
    f"Reflection rate {reflection_rate:.2%} is too high (>80%) — judges may be miscalibrated"
)

print("✓ Section 9 assertions passed")
print(f"  max_reflections: {config.reflection.max_reflections} (expected 1)")
print(f"  reflection_rate: {reflection_rate:.1%} (threshold: ≤80%)")
print(f"  routing tests:   7/7 passed")

# ── Log final section metrics ──────────────────────────────────────────
mlflow.log_metric("reflector.max_reflections", config.reflection.max_reflections)
mlflow.log_metric("reflector.routing_tests_passed", 7)

print("\n" + "═" * 60)
print("SECTIONS 7–9 COMPLETE")
print("═" * 60)
print(f"  Hypothesis Generator:  parse={parse_rate:.0%} schema={schema_valid_rate:.0%} taxonomy={taxonomy_compliance:.0%}")
print(f"  Judge Panel:           panel_pass={panel_pass_rate:.0%} bottleneck={bottleneck}")
print(f"  Reflector:             reflection_rate={reflection_rate:.0%} routing=7/7")
print("═" * 60)

In [0]:
import time
import mlflow.deployments

from trading_databricks.agents.nodes.mechanism_validator import validate_extraction

# ── Set up embedding function via MLflow deployments client ─────────────────
client = mlflow.deployments.get_deploy_client("databricks")

def embed_fn(text: str) -> list[float]:
    resp = client.predict(
        endpoint=config.validator.embedding_model,
        inputs={"input": [text]},
    )
    return resp["data"][0]["embedding"]

# ── Run mechanism validator on all articles with non-null extractions ────────
validator_results: list[dict] = []
articles_with_extraction = [
    r for r in hypothesis_results if r.get("extraction") is not None
]

for i, r in enumerate(articles_with_extraction):
    ext = r["extraction"]
    body = r.get("body", "")

    t0 = time.perf_counter()
    result = validate_extraction(
        ext,
        body,
        whitelist,
        config=config.validator,
        embed_fn=embed_fn,
    )
    latency_ms = (time.perf_counter() - t0) * 1000

    validator_results.append(
        {
            "article_idx": r["article_idx"],
            "headline": r["headline"][:80],
            "treatment_class": ext.get("treatment_class", ""),
            "mechanism": ext.get("mechanism", ""),
            "expected_direction": ext.get("expected_direction", ""),
            "evidence_spans_count": len(ext.get("evidence_spans") or []),
            **result,
            "latency_ms": latency_ms,
        }
    )

    if (i + 1) % 5 == 0:
        print(f"  Validated {i + 1}/{len(articles_with_extraction)}...")

validator_df = pl.DataFrame(validator_results)

# ── Summary ─────────────────────────────────────────────────────────────────
total_validated = validator_df.height
status_counts = validator_df.group_by("validation_status").len().sort("len", descending=True)

accepted_count = validator_df.filter(pl.col("validation_status") == "accepted").height
acceptance_rate = accepted_count / max(total_validated, 1)

mlflow.log_metric("validator.acceptance_rate", acceptance_rate)
mlflow.log_metric("validator.total_validated", total_validated)

print(f"\n✓ Mechanism Validator complete: {total_validated} extractions validated")
print(f"  Acceptance rate: {accepted_count}/{total_validated} ({acceptance_rate:.1%})")
print(f"  Median latency: {validator_df['latency_ms'].median():.0f}ms")
print(f"\nStatus breakdown:")
for row in status_counts.iter_rows(named=True):
    pct = row["len"] / total_validated
    print(f"  {row['validation_status']:<25} {row['len']:>3} ({pct:.0%})")

display(validator_df.head(10).to_pandas())

In [0]:
from difflib import get_close_matches

# ── Split by rejection type ─────────────────────────────────────────────────────
rej_mechanism = validator_df.filter(pl.col("validation_status") == "rejected_mechanism")
rej_direction = validator_df.filter(pl.col("validation_status") == "rejected_direction")
rej_evidence = validator_df.filter(pl.col("validation_status") == "rejected_evidence")

# Compute per-type rejection rates
rej_rates = {
    "mechanism": rej_mechanism.height / max(total_validated, 1),
    "direction": rej_direction.height / max(total_validated, 1),
    "evidence": rej_evidence.height / max(total_validated, 1),
}
for key, rate in rej_rates.items():
    mlflow.log_metric(f"validator.rejection_rate.{key}", rate)

print(f"Rejection Root Cause Analysis (n={total_validated})")
print("─" * 60)

# ── 1. Rejected Mechanism ────────────────────────────────────────────────
print(f"\n✖ rejected_mechanism: {rej_mechanism.height} ({rej_rates['mechanism']:.0%})")
if rej_mechanism.height > 0:
    # Build reverse lookup: L1 -> valid mechanisms
    l1_mechanisms: dict[str, list[str]] = {}
    for (l1, mech), _dirs in whitelist.items():
        l1_mechanisms.setdefault(l1, []).append(mech)

    for row in rej_mechanism.head(10).iter_rows(named=True):
        tc = row["treatment_class"]
        l1 = tc.split(".", maxsplit=1)[0] if "." in tc else tc
        produced_mech = row.get("rejected_mechanism_raw") or row["mechanism"]
        valid_mechs = l1_mechanisms.get(l1, [])
        close = get_close_matches(produced_mech, valid_mechs, n=2, cutoff=0.6)
        close_str = f" (close: {close})" if close else " (no close match)"
        print(f"  {row['headline'][:50]}")
        print(f"    L1={l1} | produced='{produced_mech}'{close_str}")
        print(f"    valid: {valid_mechs[:5]}")

# ── 2. Rejected Direction ────────────────────────────────────────────────
print(f"\n✖ rejected_direction: {rej_direction.height} ({rej_rates['direction']:.0%})")
if rej_direction.height > 0:
    for row in rej_direction.head(10).iter_rows(named=True):
        tc = row["treatment_class"]
        l1 = tc.split(".", maxsplit=1)[0] if "." in tc else tc
        mech = row["mechanism"]
        direction = row["expected_direction"]
        valid_dirs = whitelist.get((l1, mech), [])
        print(f"  {row['headline'][:50]}")
        print(f"    ({l1}, {mech}) produced='{direction}' valid={valid_dirs}")

# ── 3. Rejected Evidence ─────────────────────────────────────────────────
print(f"\n✖ rejected_evidence: {rej_evidence.height} ({rej_rates['evidence']:.0%})")
if rej_evidence.height > 0:
    # Look up original articles to show evidence mismatch
    for row in rej_evidence.head(5).iter_rows(named=True):
        idx = row["article_idx"]
        original = next((r for r in hypothesis_results if r["article_idx"] == idx), None)
        if original and original.get("extraction"):
            spans = original["extraction"].get("evidence_spans", [])
            body = original.get("body", "")
            print(f"  {row['headline'][:60]}")
            for span in spans[:2]:
                found = span.casefold().strip() in body.casefold().strip()
                mark = "✓" if found else "✗"
                print(f"    {mark} \"{span[:80]}\"")

# ── Summary dict for Section 10.4 ───────────────────────────────────────
rejection_analysis = {
    "mechanism": rej_mechanism,
    "direction": rej_direction,
    "evidence": rej_evidence,
}
print(f"\nRejection analysis stored: {', '.join(f'{k}={v.height}' for k, v in rejection_analysis.items())}")

In [0]:
# ── Extract evidence relevance scores segmented by validation status ────────
relevance_df = validator_df.filter(
    pl.col("evidence_relevance_score").is_not_null()
).select("validation_status", "evidence_relevance_score", "headline")

print(f"Evidence Relevance Score Analysis (n={relevance_df.height} with scores)")
print("─" * 60)

def _fmt(val, width=6):
    """Format a numeric value safely, returning 'N/A' for None."""
    if val is None:
        return f"{'N/A':>{width}}"
    return f"{val:>{width}.3f}"

if relevance_df.height > 0:
    # Grouped stats by validation status
    relevance_by_status = (
        relevance_df
        .group_by("validation_status")
        .agg(
            pl.col("evidence_relevance_score").mean().alias("mean"),
            pl.col("evidence_relevance_score").median().alias("median"),
            pl.col("evidence_relevance_score").std().alias("std"),
            pl.col("evidence_relevance_score").min().alias("min"),
            pl.col("evidence_relevance_score").max().alias("max"),
            pl.col("evidence_relevance_score").len().alias("count"),
        )
        .sort("validation_status")
    )

    print(f"\n{'Status':<25} {'Mean':>6} {'Median':>7} {'Std':>6} {'Min':>5} {'Max':>5} {'N':>4}")
    print("─" * 65)
    for row in relevance_by_status.iter_rows(named=True):
        print(
            f"  {row['validation_status']:<23} "
            f"{_fmt(row['mean'])} {_fmt(row['median'], 7)} "
            f"{_fmt(row['std'])} {_fmt(row['min'], 5)} {_fmt(row['max'], 5)} "
            f"{row['count']:>4}"
        )

    # Log key metrics
    accepted_scores = relevance_df.filter(
        pl.col("validation_status") == "accepted"
    )["evidence_relevance_score"]
    rejected_scores = relevance_df.filter(
        pl.col("validation_status") != "accepted"
    )["evidence_relevance_score"]

    if accepted_scores.len() > 0:
        mlflow.log_metric("validator.evidence_relevance.accepted_mean", accepted_scores.mean())
    if rejected_scores.len() > 0:
        mlflow.log_metric("validator.evidence_relevance.rejected_mean", rejected_scores.mean())

    # Discriminative power check
    if accepted_scores.len() > 0 and rejected_scores.len() > 0:
        diff = accepted_scores.mean() - rejected_scores.mean()
        print(f"\nDiscriminative power: accepted_mean - rejected_mean = {diff:+.3f}")
        if abs(diff) < 0.05:
            print("⚠ Module B is NOT discriminative — scores are similar across statuses")
            print("  Recommendation: Do NOT activate evidence_grounding_reject_threshold")
        else:
            print(f"✓ Module B shows {'positive' if diff > 0 else 'negative'} discrimination")
    else:
        print("\n⚠ Cannot assess discrimination — insufficient data in one or both groups")

    # Hypothetical threshold analysis
    all_scores = relevance_df["evidence_relevance_score"]
    print(f"\nHypothetical threshold impact:")
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
        would_reject = all_scores.filter(all_scores < threshold).len()
        print(f"  If threshold={threshold:.1f}: {would_reject}/{all_scores.len()} additional rejections ({would_reject / max(all_scores.len(), 1):.0%})")
else:
    print("\n⚠ No evidence relevance scores available (embedding failures?)")
    relevance_by_status = pl.DataFrame()

In [0]:
# ── 1. Validation Funnel ───────────────────────────────────────────────────────
# Show how articles progress through validation steps
funnel_stages = [
    ("Total extractions", total_validated),
    ("Passed step 2 (mechanism)", total_validated - rej_mechanism.height),
    ("Passed step 3 (direction)", total_validated - rej_mechanism.height - rej_direction.height),
    ("Passed step 4 (evidence)", accepted_count),
]

fig1 = go.Figure(
    go.Funnel(
        y=[s[0] for s in funnel_stages],
        x=[s[1] for s in funnel_stages],
        textinfo="value+percent initial",
        marker_color=["#58A6FF", "#A371F7", "#F0883E", "#52C41A"],
    )
)
fig1.update_layout(
    title="Mechanism Validator — Validation Funnel",
    template="plotly_dark",
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
    height=350,
)
mlflow.log_figure(fig1, "validator_funnel.html")
fig1.show()

# ── 2. Evidence Relevance Histogram ───────────────────────────────────────
if relevance_df.height > 0:
    fig2 = px.histogram(
        relevance_df.to_pandas(),
        x="evidence_relevance_score",
        color="validation_status",
        nbins=20,
        title="Evidence Relevance Score Distribution by Validation Status",
        template="plotly_dark",
        color_discrete_map={
            "accepted": "#52C41A",
            "rejected_mechanism": "#FF4D4F",
            "rejected_direction": "#F0883E",
            "rejected_evidence": "#A371F7",
        },
        barmode="overlay",
        opacity=0.7,
    )
    # Add threshold line if configured
    if config.validator.evidence_grounding_reject_threshold is not None:
        fig2.add_vline(
            x=config.validator.evidence_grounding_reject_threshold,
            line_dash="dash", line_color="#FF4D4F",
            annotation_text=f"threshold={config.validator.evidence_grounding_reject_threshold}",
        )
    fig2.update_layout(
        plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
    )
    mlflow.log_figure(fig2, "validator_evidence_relevance.html")
    fig2.show()

# ── 3. Rejection Cause Pie ─────────────────────────────────────────────────
status_pd = validator_df.group_by("validation_status").len().sort("len", descending=True).to_pandas()
fig3 = px.pie(
    status_pd,
    names="validation_status",
    values="len",
    title="Validation Status Distribution",
    template="plotly_dark",
    color="validation_status",
    color_discrete_map={
        "accepted": "#52C41A",
        "rejected_mechanism": "#FF4D4F",
        "rejected_direction": "#F0883E",
        "rejected_evidence": "#A371F7",
    },
)
fig3.update_layout(
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
)
mlflow.log_figure(fig3, "validator_status_pie.html")
fig3.show()

# ── Hard assertions ─────────────────────────────────────────────────────
assert acceptance_rate >= 0.40, f"Acceptance rate {acceptance_rate:.1%} < 40% threshold"
assert validator_df.filter(
    pl.col("evidence_relevance_score").is_not_null()
)["evidence_relevance_score"].min() >= 0.0, "Relevance score < 0"
assert validator_df.filter(
    pl.col("evidence_relevance_score").is_not_null()
)["evidence_relevance_score"].max() <= 1.0, "Relevance score > 1"

print("\n✓ Section 10 assertions passed")
print(f"  Acceptance: {acceptance_rate:.1%} | Rejections: mech={rej_rates['mechanism']:.0%} dir={rej_rates['direction']:.0%} evid={rej_rates['evidence']:.0%}")

In [0]:
import time
from trading_databricks.agents.graph import build_mas_graph

# ── Configuration ──────────────────────────────────────────────────────────
E2E_SAMPLE_SIZE = 30  # Set to 120 for full run (expensive: ~360+ LLM calls)

# ── Build the graph ────────────────────────────────────────────────────────
graph = build_mas_graph(config=config, taxonomy=taxonomy, whitelist=whitelist)
print(f"✓ Graph compiled: {len(graph.nodes)} nodes")

# ── Select crypto articles for E2E test ─────────────────────────────────
# Reuse crypto articles from the test set (first E2E_SAMPLE_SIZE)
e2e_articles = test_articles[:E2E_SAMPLE_SIZE]
print(f"  Running E2E on {len(e2e_articles)} articles...")

# ── Run full graph on each article ──────────────────────────────────────
e2e_results: list[dict] = []

for i, article in enumerate(e2e_articles):
    state_in = {
        "headline": article["headline"],
        "body": article["body"],
        "published_at": str(article.get("published_at", "")),
    }

    t0 = time.perf_counter()
    try:
        output = graph.invoke(state_in)
        latency_ms = (time.perf_counter() - t0) * 1000

        # Extract key fields from output state
        extraction = output.get("extraction")
        e2e_results.append({
            "article_idx": i,
            "story_id": article.get("story_id", f"article_{i}"),
            "headline": article["headline"][:80],
            "extraction_status": output.get("extraction_status", "unknown"),
            "extraction_path": output.get("extraction_path"),
            "validation_status": output.get("validation_status"),
            "treatment_class": extraction.get("treatment_class") if extraction else None,
            "mechanism": extraction.get("mechanism") if extraction else None,
            "expected_direction": extraction.get("expected_direction") if extraction else None,
            "evidence_spans": extraction.get("evidence_spans") if extraction else None,
            "evidence_relevance_score": output.get("evidence_relevance_score"),
            "reflection_count": output.get("reflection_count", 0),
            "latency_ms": latency_ms,
            "error": None,
        })
    except Exception as exc:
        latency_ms = (time.perf_counter() - t0) * 1000
        e2e_results.append({
            "article_idx": i,
            "story_id": article.get("story_id", f"article_{i}"),
            "headline": article["headline"][:80],
            "extraction_status": "error",
            "extraction_path": None,
            "validation_status": None,
            "treatment_class": None,
            "mechanism": None,
            "expected_direction": None,
            "evidence_spans": None,
            "evidence_relevance_score": None,
            "reflection_count": 0,
            "latency_ms": latency_ms,
            "error": repr(exc)[:200],
        })

    if (i + 1) % 5 == 0:
        elapsed = sum(r["latency_ms"] for r in e2e_results) / 1000
        print(f"  [{i + 1}/{len(e2e_articles)}] elapsed={elapsed:.1f}s")

# ── Build DataFrame (exclude list columns for Polars) ────────────────────
e2e_df = pl.DataFrame([
    {k: v for k, v in r.items() if k != "evidence_spans"}
    for r in e2e_results
])

# ── Summary ─────────────────────────────────────────────────────────────────
error_count = e2e_df.filter(pl.col("extraction_status") == "error").height
accepted_e2e = e2e_df.filter(pl.col("validation_status") == "accepted").height
median_latency = e2e_df["latency_ms"].median()

print(f"\n✓ Full graph E2E complete: {e2e_df.height} articles processed")
print(f"  Errors: {error_count} | Accepted: {accepted_e2e} ({accepted_e2e / max(e2e_df.height, 1):.1%})")
print(f"  Median latency: {median_latency:.0f}ms | Total time: {e2e_df['latency_ms'].sum() / 1000:.1f}s")

# Show errors if any
if error_count > 0:
    print(f"\n⚠ {error_count} articles failed with errors:")
    for row in e2e_df.filter(pl.col("extraction_status") == "error").head(5).iter_rows(named=True):
        print(f"  {row['headline'][:50]} | {row['error'][:80]}")

display(e2e_df.head(10).to_pandas())

In [0]:
from pydantic import ValidationError
import importlib
import trading_databricks.agents.schemas
importlib.reload(trading_databricks.agents.schemas)
from trading_databricks.agents.schemas import CausalChain

# ── Score each E2E result with 5 deterministic scorers ────────────────────
e2e_scores_list: list[dict] = []

for r in e2e_results:
    scores: dict[str, float] = {"story_id": r["story_id"]}

    # 1. schema_valid: Does extraction conform to CausalChain Pydantic model?
    # Reconstruct minimal extraction dict for Pydantic validation
    ext_for_pydantic = {
        "is_impactful": True,
        "treatment_class": r.get("treatment_class"),
        "trigger_event": "(not stored in e2e_df)",
        "mechanism": r.get("mechanism"),
        "asset": "BTC",
        "expected_direction": r.get("expected_direction"),
        "evidence_spans": r.get("evidence_spans") if "evidence_spans" in r else None,
    }
    try:
        CausalChain(**{k: v for k, v in ext_for_pydantic.items() if v is not None})
        scores["schema_valid"] = 1.0
    except (ValidationError, TypeError):
        scores["schema_valid"] = 0.0

    # 2. treatment_in_taxonomy: treatment_class L1 exists in taxonomy
    tc = r.get("treatment_class") or ""
    l1 = tc.split(".", maxsplit=1)[0] if "." in tc else tc
    scores["treatment_in_taxonomy"] = 1.0 if l1 in taxonomy else 0.0

    # 3. mechanism_in_whitelist: (L1, mechanism) exists in whitelist
    mech = r.get("mechanism") or ""
    scores["mechanism_in_whitelist"] = 1.0 if (l1, mech) in whitelist else 0.0

    # 4. evidence_spans_verbatim: all spans are substrings of body
    e2e_article = e2e_articles[r["article_idx"]] if r["article_idx"] < len(e2e_articles) else {}
    body = e2e_article.get("body", "")
    # For the flat e2e_results, evidence_spans was stored in the full result
    orig_result = e2e_results[r["article_idx"]] if r["article_idx"] < len(e2e_results) else {}
    actual_spans = orig_result.get("evidence_spans")
    if actual_spans and body:
        body_lower = body.casefold()
        all_verbatim = all(s.casefold().strip() in body_lower for s in actual_spans if s)
        scores["evidence_spans_verbatim"] = 1.0 if all_verbatim else 0.0
    else:
        scores["evidence_spans_verbatim"] = 0.0

    # 5. validator_accepted: validation_status == 'accepted'
    scores["validator_accepted"] = 1.0 if r.get("validation_status") == "accepted" else 0.0

    e2e_scores_list.append(scores)

e2e_scores = pl.DataFrame(e2e_scores_list)

# ── Compute aggregate metrics ────────────────────────────────────────────
scorer_cols = ["schema_valid", "treatment_in_taxonomy", "mechanism_in_whitelist",
               "evidence_spans_verbatim", "validator_accepted"]

e2e_metrics: dict[str, float] = {}
print("E2E Deterministic Scorer Results")
print("─" * 50)
print(f"  {'Scorer':<28} {'Mean':>6} {'Pass':>5} {'Total':>6}")
print("─" * 50)
for col in scorer_cols:
    mean_val = e2e_scores[col].mean()
    pass_count = int(e2e_scores[col].sum())
    e2e_metrics[col] = mean_val
    mlflow.log_metric(f"e2e.{col}", mean_val)
    bar = "█" * int(mean_val * 20)
    print(f"  {col:<28} {mean_val:>5.1%} {pass_count:>5} {e2e_scores.height:>6}  {bar}")

print(f"\n✓ E2E scoring complete ({e2e_scores.height} articles)")

In [0]:
# ── Quality Gate (from deploy_mas_agent Phase 4) ─────────────────────────
gates = {
    "schema_valid": {"value": e2e_metrics["schema_valid"], "threshold": 0.9},
    "validator_accepted": {"value": e2e_metrics["validator_accepted"], "threshold": 0.5},
}

print("═" * 50)
print("  QUALITY GATE CHECK")
print("═" * 50)

all_passed = True
for gate_name, gate_info in gates.items():
    value = gate_info["value"]
    threshold = gate_info["threshold"]
    passed = value > threshold
    status = "✓ PASS" if passed else "✗ FAIL"
    if not passed:
        all_passed = False
    print(f"  {gate_name:<25} {value:.3f} (threshold: {threshold})     [{status}]")

print("─" * 50)
quality_gate_passed = all_passed
overall_status = "✓ PASSED" if quality_gate_passed else "✗ FAILED"
print(f"  Overall: {overall_status}")
print("═" * 50)

mlflow.log_metric("e2e.quality_gate_passed", 1.0 if quality_gate_passed else 0.0)

# Do NOT hard-assert — let notebook continue to capture all diagnostics
if not quality_gate_passed:
    print("\n⚠ Quality gate FAILED — review scorer details above")
    print("  Notebook continues to gather full diagnostics.")
else:
    print("\n✓ Quality gate PASSED — system is ready for deployment")

In [0]:
# ── Latency statistics ──────────────────────────────────────────────────────
latencies = e2e_df["latency_ms"]

p50 = latencies.quantile(0.50)
p90 = latencies.quantile(0.90)
p99 = latencies.quantile(0.99)
mean_lat = latencies.mean()
max_lat = latencies.max()

mlflow.log_metric("e2e.latency_p50_ms", p50)
mlflow.log_metric("e2e.latency_p90_ms", p90)
mlflow.log_metric("e2e.latency_mean_ms", mean_lat)

print("E2E Latency Profile")
print("─" * 45)
print(f"  p50:  {p50:>8,.0f} ms")
print(f"  p90:  {p90:>8,.0f} ms")
print(f"  p99:  {p99:>8,.0f} ms")
print(f"  mean: {mean_lat:>8,.0f} ms")
print(f"  max:  {max_lat:>8,.0f} ms")
print(f"  total: {latencies.sum() / 1000:>7,.1f} s ({latencies.sum() / 60_000:.1f} min)")

# ── Identify outliers (> 2x p90) ───────────────────────────────────────
outlier_threshold = 2.0 * p90
outliers = e2e_df.filter(pl.col("latency_ms") > outlier_threshold)

if outliers.height > 0:
    print(f"\n⚠ Outlier articles (> {outlier_threshold:.0f}ms = 2×p90): {outliers.height}")
    for row in outliers.sort("latency_ms", descending=True).head(5).iter_rows(named=True):
        print(f"  {row['headline'][:50]} | {row['latency_ms']:.0f}ms | reflections={row['reflection_count']}")
else:
    print(f"\n✓ No latency outliers (all < {outlier_threshold:.0f}ms)")

# ── Latency by extraction path ────────────────────────────────────────
path_latency = (
    e2e_df.filter(pl.col("extraction_path").is_not_null())
    .group_by("extraction_path")
    .agg(
        pl.col("latency_ms").median().alias("median_ms"),
        pl.col("latency_ms").mean().alias("mean_ms"),
        pl.col("latency_ms").len().alias("count"),
    )
    .sort("median_ms")
)
if path_latency.height > 0:
    print(f"\nLatency by extraction path:")
    for row in path_latency.iter_rows(named=True):
        print(f"  {row['extraction_path']:<12} median={row['median_ms']:.0f}ms mean={row['mean_ms']:.0f}ms (n={row['count']})")

In [0]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 1. Scorer Radar Chart ────────────────────────────────────────────────────
radar_labels = scorer_cols + [scorer_cols[0]]  # close the polygon
radar_values = [e2e_metrics[c] for c in scorer_cols] + [e2e_metrics[scorer_cols[0]]]

fig1 = go.Figure()
fig1.add_trace(go.Scatterpolar(
    r=radar_values, theta=radar_labels,
    fill="toself", name="E2E Scores",
    line_color="#58A6FF", fillcolor="rgba(88, 166, 255, 0.3)",
))
# Threshold lines
fig1.add_trace(go.Scatterpolar(
    r=[0.9] * len(radar_labels), theta=radar_labels,
    name="0.9 threshold", line=dict(dash="dash", color="#FF4D4F"),
))
fig1.add_trace(go.Scatterpolar(
    r=[0.5] * len(radar_labels), theta=radar_labels,
    name="0.5 threshold", line=dict(dash="dot", color="#F0883E"),
))
fig1.update_layout(
    title="E2E Quality Profile — 5 Deterministic Scorers",
    template="plotly_dark",
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1]),
        bgcolor="#0D1117",
    ),
    plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
    height=450,
)
mlflow.log_figure(fig1, "e2e_scorer_radar.html")
fig1.show()

# ── 2. Treatment Class Distribution (accepted only) ─────────────────────
accepted_df = e2e_df.filter(pl.col("validation_status") == "accepted")
if accepted_df.height > 0:
    tc_counts = (
        accepted_df
        .filter(pl.col("treatment_class").is_not_null())
        .with_columns(
            pl.col("treatment_class").str.split(".").list.first().alias("L1")
        )
        .group_by("treatment_class", "L1")
        .len()
        .sort("len", descending=True)
        .head(15)
    )
    fig2 = px.bar(
        tc_counts.to_pandas(), y="treatment_class", x="len", color="L1",
        orientation="h", title="E2E Top-15 Treatment Classes (Accepted)",
        labels={"len": "Count", "treatment_class": "Treatment Class"},
        template="plotly_dark",
    )
    fig2.update_layout(
        plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
        yaxis={"categoryorder": "total ascending"},
        height=max(350, tc_counts.height * 25),
    )
    mlflow.log_figure(fig2, "e2e_treatment_distribution.html")
    fig2.show()

# ── 3. Validation Status Pie ─────────────────────────────────────────────
status_dist = e2e_df.group_by("validation_status").len().sort("len", descending=True)
fig3 = px.pie(
    status_dist.to_pandas(), names="validation_status", values="len",
    title="E2E Validation Status Distribution",
    template="plotly_dark",
    color="validation_status",
    color_discrete_map={
        "accepted": "#52C41A", "rejected_mechanism": "#FF4D4F",
        "rejected_direction": "#F0883E", "rejected_evidence": "#A371F7",
    },
)
fig3.update_layout(plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3")
mlflow.log_figure(fig3, "e2e_validation_status.html")
fig3.show()

# ── 4. Latency Histogram ─────────────────────────────────────────────────
fig4 = px.histogram(
    e2e_df.to_pandas(), x="latency_ms", nbins=30,
    title="E2E Per-Article Latency Distribution",
    labels={"latency_ms": "Latency (ms)"},
    template="plotly_dark",
    color_discrete_sequence=["#58A6FF"],
)
fig4.add_vline(x=p50, line_dash="solid", line_color="#52C41A", annotation_text=f"p50={p50:.0f}ms")
fig4.add_vline(x=p90, line_dash="dash", line_color="#F0883E", annotation_text=f"p90={p90:.0f}ms")
fig4.update_layout(plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3")
mlflow.log_figure(fig4, "e2e_latency_histogram.html")
fig4.show()

# ── Hard assertions ─────────────────────────────────────────────────────
assert error_count == 0, f"Unhandled exceptions in graph: {error_count} articles"
assert e2e_df.height >= E2E_SAMPLE_SIZE * 0.95, (
    f"Too many articles lost: {e2e_df.height}/{E2E_SAMPLE_SIZE}"
)

print("\n✓ Section 11 assertions passed")
print(f"  Gate: {'PASSED' if quality_gate_passed else 'FAILED'} | "
      f"Errors: {error_count} | Articles: {e2e_df.height}/{E2E_SAMPLE_SIZE}")

In [0]:
from trading_databricks.agents.schemas import CAUSAL_FEATURES_SCHEMA

# ── Read gold table ──────────────────────────────────────────────────────────
GOLD_TABLE = "`smart-trading-dev`.`03_gold`.`news_causal_features`"

try:
    gold_spark_df = spark.table(GOLD_TABLE)
    gold_exists = True
except Exception as e:
    gold_exists = False
    print(f"⚠ Gold table not accessible: {e}")

if gold_exists:
    # ── Schema comparison ─────────────────────────────────────────────────
    expected_fields = {f.name for f in CAUSAL_FEATURES_SCHEMA.fields}
    actual_fields = set(gold_spark_df.columns)

    missing = expected_fields - actual_fields
    extra = actual_fields - expected_fields
    common = expected_fields & actual_fields

    print(f"Gold Table Schema Check")
    print("─" * 50)
    print(f"  Expected fields (CAUSAL_FEATURES_SCHEMA): {len(expected_fields)}")
    print(f"  Actual fields in gold table:             {len(actual_fields)}")
    print(f"  Common:  {len(common)}")

    if missing:
        print(f"  ✗ Missing ({len(missing)}): {sorted(missing)}")
    if extra:
        print(f"  + Extra ({len(extra)}): {sorted(extra)}")

    # Type comparison for common fields
    expected_types = {f.name: f.dataType.simpleString() for f in CAUSAL_FEATURES_SCHEMA.fields}
    actual_types = {f.name: f.dataType.simpleString() for f in gold_spark_df.schema.fields}

    type_mismatches = []
    for field_name in sorted(common):
        exp_t = expected_types[field_name]
        act_t = actual_types[field_name]
        if exp_t != act_t:
            type_mismatches.append((field_name, exp_t, act_t))

    if type_mismatches:
        print(f"\n  Type mismatches ({len(type_mismatches)}):")
        for name, exp, act in type_mismatches:
            print(f"    {name}: expected={exp}, actual={act}")
    else:
        print(f"\n  ✓ All {len(common)} common field types match")

    # Assertion: all expected fields present
    schema_conformant = len(missing) == 0
    mlflow.log_metric("gold.schema_conformant", 1.0 if schema_conformant else 0.0)
    mlflow.log_metric("gold.field_count", len(actual_fields))

    if schema_conformant:
        print(f"\n✓ Gold table schema is CONFORMANT ({len(expected_fields)} fields)")
    else:
        print(f"\n⚠ Gold table schema is NON-CONFORMANT (missing: {sorted(missing)})")

    assert schema_conformant, f"Schema mismatch — missing fields: {sorted(missing)}"

In [0]:
from pyspark.sql import functions as F

if gold_exists:
    # ── Basic metrics ────────────────────────────────────────────────────────
    gold_row_count = gold_spark_df.count()

    # Date range and freshness
    date_stats = gold_spark_df.agg(
        F.min("published_at").alias("min_published"),
        F.max("published_at").alias("max_published"),
        F.max("processed_at").alias("last_processed"),
    ).first()

    min_published = date_stats["min_published"]
    max_published = date_stats["max_published"]
    last_processed = date_stats["last_processed"]

    # Hours since last processed
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc)
    hours_since_last = (
        (now - last_processed.replace(tzinfo=timezone.utc)).total_seconds() / 3600
        if last_processed else float("inf")
    )

    # Validation status distribution
    status_dist_gold = (
        gold_spark_df.groupBy("validation_status")
        .count()
        .orderBy(F.desc("count"))
        .collect()
    )

    accepted_gold = sum(r["count"] for r in status_dist_gold if r["validation_status"] == "accepted")
    gold_acceptance_rate = accepted_gold / max(gold_row_count, 1)

    # Unique treatments and mechanisms
    unique_treatments = gold_spark_df.select("treatment_class").distinct().count()
    unique_mechanisms = gold_spark_df.select("mechanism").distinct().count()

    # Per-column null rates
    null_rates = {}
    for col_name in gold_spark_df.columns:
        null_count = gold_spark_df.filter(F.col(col_name).isNull()).count()
        null_rates[col_name] = null_count / max(gold_row_count, 1)

    # Log metrics
    mlflow.log_metric("gold.total_rows", gold_row_count)
    mlflow.log_metric("gold.accepted_count", accepted_gold)
    mlflow.log_metric("gold.acceptance_rate", gold_acceptance_rate)
    mlflow.log_metric("gold.hours_since_last", hours_since_last)
    mlflow.log_metric("gold.unique_treatments", unique_treatments)

    # Print quality report
    print(f"Gold Table Data Quality Report")
    print("═" * 60)
    print(f"  Total rows:          {gold_row_count:,}")
    print(f"  Date range:          {min_published} → {max_published}")
    print(f"  Last processed:      {last_processed}")
    print(f"  Hours since last:    {hours_since_last:.1f}h")
    print(f"  Accepted:            {accepted_gold:,} ({gold_acceptance_rate:.1%})")
    print(f"  Unique treatments:   {unique_treatments}")
    print(f"  Unique mechanisms:   {unique_mechanisms}")

    print(f"\nValidation Status Distribution:")
    for row in status_dist_gold:
        pct = row["count"] / gold_row_count
        print(f"  {row['validation_status'] or '(null)':<25} {row['count']:>5} ({pct:.1%})")

    # High null rate columns (> 50% null, excluding expected nullables)
    always_nullable = {"market_vol_at_publish", "market_trend_at_publish",
                       "dollar_bar_close", "confounding_cluster_id",
                       "rejected_mechanism_raw"}
    print(f"\nColumn Null Rates (showing > 10%, excluding expected nullables):")
    for col_name, rate in sorted(null_rates.items(), key=lambda x: -x[1]):
        if rate > 0.10 and col_name not in always_nullable:
            bar = "█" * int(rate * 20)
            print(f"  {col_name:<30} {rate:>5.0%} {bar}")

    print(f"\n✓ Gold table quality report complete")
else:
    gold_row_count = 0
    hours_since_last = float("inf")
    print("⚠ Skipping data quality check — gold table not accessible")

In [0]:
if gold_exists and e2e_df.height > 0:
    # ── Get story_ids from E2E run ─────────────────────────────────────────
    e2e_story_ids = e2e_df["story_id"].to_list()

    # Query gold table for matching articles
    gold_matching = (
        gold_spark_df
        .filter(F.col("story_id").isin(e2e_story_ids))
        .select(
            "story_id",
            F.col("treatment_class").alias("gold_treatment_class"),
            F.col("mechanism").alias("gold_mechanism"),
            F.col("validation_status").alias("gold_validation_status"),
            F.col("expected_direction").alias("gold_expected_direction"),
        )
        .toPandas()
    )

    if len(gold_matching) > 0:
        gold_pl = pl.from_pandas(gold_matching)

        # Join with local E2E results
        local_subset = e2e_df.select(
            "story_id",
            pl.col("treatment_class").alias("local_treatment_class"),
            pl.col("mechanism").alias("local_mechanism"),
            pl.col("validation_status").alias("local_validation_status"),
            pl.col("expected_direction").alias("local_expected_direction"),
        )

        comparison_df = local_subset.join(gold_pl, on="story_id", how="inner")

        if comparison_df.height > 0:
            # Compute field-level agreement
            fields_to_compare = [
                ("treatment_class", "local_treatment_class", "gold_treatment_class"),
                ("mechanism", "local_mechanism", "gold_mechanism"),
                ("validation_status", "local_validation_status", "gold_validation_status"),
                ("expected_direction", "local_expected_direction", "gold_expected_direction"),
            ]

            field_agreement_rates: dict[str, float] = {}
            print(f"Gold vs Local Graph Comparison (n={comparison_df.height} overlapping articles)")
            print("─" * 55)
            print(f"  {'Field':<22} {'Agreement':>10} {'Match':>6} {'Total':>6}")
            print("─" * 55)

            for field_name, local_col, gold_col in fields_to_compare:
                # Handle nulls: both null = agree, one null = disagree
                agree = comparison_df.filter(
                    (pl.col(local_col) == pl.col(gold_col))
                    | (pl.col(local_col).is_null() & pl.col(gold_col).is_null())
                ).height
                rate = agree / max(comparison_df.height, 1)
                field_agreement_rates[field_name] = rate
                mlflow.log_metric(f"gold.agreement_rate.{field_name}", rate)
                print(f"  {field_name:<22} {rate:>9.1%} {agree:>6} {comparison_df.height:>6}")

            overall_agreement = sum(field_agreement_rates.values()) / len(field_agreement_rates)
            print(f"\n  Overall field agreement: {overall_agreement:.1%}")

            # Show disagreements
            disagreements = comparison_df.filter(
                (pl.col("local_treatment_class") != pl.col("gold_treatment_class"))
                | (pl.col("local_mechanism") != pl.col("gold_mechanism"))
                | (pl.col("local_validation_status") != pl.col("gold_validation_status"))
            )
            if disagreements.height > 0:
                print(f"\n  Disagreements ({disagreements.height} articles):")
                for row in disagreements.head(5).iter_rows(named=True):
                    print(f"    story_id={row['story_id'][:30]}")
                    if row["local_treatment_class"] != row["gold_treatment_class"]:
                        print(f"      treatment: local={row['local_treatment_class']} vs gold={row['gold_treatment_class']}")
                    if row["local_mechanism"] != row["gold_mechanism"]:
                        print(f"      mechanism: local={row['local_mechanism']} vs gold={row['gold_mechanism']}")
                    if row["local_validation_status"] != row["gold_validation_status"]:
                        print(f"      status: local={row['local_validation_status']} vs gold={row['gold_validation_status']}")

                print("\n  Note: Disagreements are expected due to:")
                print("    1. Gatekeeper parity (SQL RLIKE vs spaCy)")
                print("    2. LLM non-determinism (even at temperature=0.0)")
                print("    3. Model version differences (endpoint vs local)")
            else:
                print("\n  ✓ Perfect agreement on all fields!")
        else:
            print("No overlapping articles between E2E run and gold table")
            field_agreement_rates = {}
    else:
        print(f"No matching story_ids found in gold table (checked {len(e2e_story_ids)} IDs)")
        print("  This may mean the gold table covers a different time window")
        field_agreement_rates = {}
        comparison_df = pl.DataFrame()
else:
    print("⚠ Skipping comparison — gold table not accessible or no E2E results")
    field_agreement_rates = {}
    comparison_df = pl.DataFrame()

In [0]:
if gold_exists and gold_row_count > 0:
    # ── 1. Gold Validation Status Pie ───────────────────────────────────────
    gold_status_pd = (
        gold_spark_df.groupBy("validation_status")
        .count()
        .orderBy(F.desc("count"))
        .toPandas()
    )
    fig1 = px.pie(
        gold_status_pd, names="validation_status", values="count",
        title=f"Gold Table Validation Status (n={gold_row_count:,})",
        template="plotly_dark",
        color="validation_status",
        color_discrete_map={
            "accepted": "#52C41A", "rejected_mechanism": "#FF4D4F",
            "rejected_direction": "#F0883E", "rejected_evidence": "#A371F7",
        },
    )
    fig1.update_layout(
        plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
    )
    mlflow.log_figure(fig1, "gold_validation_status.html")
    fig1.show()

    # ── 2. Gold Top-10 Treatments ─────────────────────────────────────────
    gold_tc_pd = (
        gold_spark_df
        .filter(F.col("treatment_class").isNotNull())
        .groupBy("treatment_class")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
        .toPandas()
    )
    if len(gold_tc_pd) > 0:
        gold_tc_pd["L1"] = gold_tc_pd["treatment_class"].str.split(".").str[0]
        fig2 = px.bar(
            gold_tc_pd, y="treatment_class", x="count", color="L1",
            orientation="h",
            title="Gold Table Top-10 Treatment Classes",
            labels={"count": "Count", "treatment_class": "Treatment Class"},
            template="plotly_dark",
        )
        fig2.update_layout(
            plot_bgcolor="#0D1117", paper_bgcolor="#161B22", font_color="#E6EDF3",
            yaxis={"categoryorder": "total ascending"},
            height=400,
        )
        mlflow.log_figure(fig2, "gold_treatment_top10.html")
        fig2.show()

    # ── Assertions ───────────────────────────────────────────────────────
    assert gold_row_count > 0, "Gold table is empty — pipeline has not run"
    assert hours_since_last < 168, (
        f"Gold table is stale: {hours_since_last:.0f} hours since last processed_at"
    )

    print(f"\n✓ Section 12 assertions passed")
    print(f"  Rows: {gold_row_count:,} | Freshness: {hours_since_last:.1f}h | Schema: conformant")
else:
    print("⚠ Skipping gold table visualizations — table is empty or inaccessible")

print("\n" + "═" * 60)
print("SECTIONS 10–12 COMPLETE")
print("═" * 60)
print(f"  Mechanism Validator: acceptance={acceptance_rate:.1%}")
print(f"  E2E Quality Gate:   {'PASSED' if quality_gate_passed else 'FAILED'}")
print(f"  Gold Table:         {'OK' if gold_exists and gold_row_count > 0 else 'UNAVAILABLE'}")
print("═" * 60)

---
## Section 13: Serving Endpoint — Model Serving Validation (4 cells)

**Component:** `serve.py` → `MASPyfuncModel` deployed to `mas-news-causal`  
**What it does:** Production inference path. Serving container loads bundled taxonomy/whitelist JSON artifacts, builds graph without Spark, processes articles via HTTP API.

**LLM calls:** \~5 (via endpoint, which internally makes FMAPI calls).

**Key questions:**
1. Is the endpoint responsive? (May need cold-start wake-up)
2. Does the response schema match the 13-field output contract?
3. Are endpoint results consistent with local graph results?
4. What is the end-to-end latency (including cold start)?

### Cell 13.1 `[python]` — Endpoint Health + Cold Start Latency

**Logic:**
1. Check endpoint state via SDK: `w.serving_endpoints.get(name='mas-news-causal')`
2. If scaled to zero, send a warm-up request and measure cold-start latency
3. Verify: endpoint state is READY, served entity version matches latest model version

**Variables created:**
* `endpoint_state: str` (READY / NOT_READY / SCALED_TO_ZERO)
* `cold_start_latency_s: float | None`
* `served_model_version: str`

**Print:** Endpoint status, served version, cold-start latency if applicable.

### Cell 13.2 `[python]` — Query Endpoint with Test Articles

**Logic:**
1. Select 5 crypto articles from test set
2. For each, call:
   ```python
   response = w.serving_endpoints.query(
       name="mas-news-causal",
       dataframe_records=[{"headline": ..., "body": ..., "published_at": ...}],
   )
   ```
3. Parse response, verify all 13 output fields present
4. Measure per-request latency

**Variables created:**
* `endpoint_results: list[dict]` — parsed responses with latency
* `endpoint_df: pl.DataFrame`

**Assertions:**
```python
for r in endpoint_results:
    assert 'treatment_class' in r, "Missing treatment_class in response"
    assert 'validation_status' in r, "Missing validation_status in response"
```

### Cell 13.3 `[python]` — Endpoint vs Local Graph Consistency

**Purpose:** For the same 5 articles, compare endpoint response vs local graph output.

**Logic:**
1. Run local graph on same 5 articles (reuse from Section 11 if available)
2. Compare field-by-field: treatment_class, mechanism, validation_status, expected_direction
3. Note: temperature=0.0 but non-determinism in multi-request batching can cause differences

**Variables created:**
* `consistency_rate: float` — fraction of fields that match

**Metrics logged:** `endpoint.consistency_rate`

**Print:** Per-article comparison table. Flag any discrepancies with explanation.

**Important note:** Some discrepancy is expected because:
* Serving container uses `_parse_input()` which handles both struct and JSON-string input
* Cold-start may cause different VS query results (if few_shot_retriever returns stale cache)
* LLM endpoints may have slightly different batching behavior

### Cell 13.4 `[python]` — Inference Table Verification

**Purpose:** Verify that AI Gateway inference tables are capturing requests.

**Logic:**
1. Query `smart-trading-dev.03_gold.mas_news_causal_payload` — check row count
2. Query `...payload_request_logs` — check if our test requests appear
3. Verify schema of inference tables

**Assertion:**
```python
# Inference tables may take a few minutes to populate
assert payload_count >= 0, "Payload table should exist (even if empty)"
```

**Print:** Inference table row counts and last insert timestamp.

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateReady

w = WorkspaceClient()

ENDPOINT_NAME = "mas-news-causal"
MODEL_NAME = "smart-trading-dev.03_gold.mas_news_causal"

# ── Check endpoint state ─────────────────────────────────────────────────────
endpoint_ready = False
endpoint_state = "NOT_FOUND"
served_model_version = "unknown"
cold_start_latency_s: float | None = None

try:
    ep = w.serving_endpoints.get(name=ENDPOINT_NAME)
    endpoint_state = ep.state.ready.value if ep.state and ep.state.ready else "UNKNOWN"

    # Extract served entity version
    if ep.config and ep.config.served_entities:
        entity = ep.config.served_entities[0]
        served_model_version = entity.entity_version or "unknown"

    if endpoint_state == EndpointStateReady.READY.value:
        endpoint_ready = True
        print(f"✓ Endpoint '{ENDPOINT_NAME}' is READY")
    else:
        print(f"⚠ Endpoint '{ENDPOINT_NAME}' state: {endpoint_state}")
        # If not ready, attempt a warm-up query to trigger cold start
        print("  Sending warm-up request to trigger cold start...")
        t0 = time.perf_counter()
        try:
            warmup_response = w.serving_endpoints.query(
                name=ENDPOINT_NAME,
                dataframe_records=[{
                    "headline": "Bitcoin price surges past $100k",
                    "body": "Bitcoin reached a new all-time high today.",
                    "published_at": "2026-04-20T00:00:00Z",
                }],
            )
            cold_start_latency_s = time.perf_counter() - t0
            endpoint_ready = True
            endpoint_state = "READY"
            print(f"  ✓ Endpoint warmed up in {cold_start_latency_s:.1f}s")
        except Exception as warmup_err:
            cold_start_latency_s = time.perf_counter() - t0
            print(f"  ✗ Warm-up failed after {cold_start_latency_s:.1f}s: {warmup_err}")

except Exception as e:
    print(f"✗ Endpoint '{ENDPOINT_NAME}' not found or inaccessible: {e}")

# ── Log metrics ──────────────────────────────────────────────────────────────
mlflow.log_metric("endpoint.ready", 1.0 if endpoint_ready else 0.0)
if cold_start_latency_s is not None:
    mlflow.log_metric("endpoint.cold_start_s", cold_start_latency_s)
mlflow.log_param("endpoint.served_version", served_model_version)

print(f"\nEndpoint Health Summary")
print("─" * 45)
print(f"  Endpoint:        {ENDPOINT_NAME}")
print(f"  State:           {endpoint_state}")
print(f"  Served version:  {served_model_version}")
print(f"  Cold start:      {f'{cold_start_latency_s:.1f}s' if cold_start_latency_s else 'N/A (already warm)'}")

In [0]:
# ── Select 5 crypto articles from test set for endpoint testing ──────────────
ENDPOINT_TEST_SIZE = 5
endpoint_test_articles = [a for a in test_articles if a.get("is_crypto", True)][:ENDPOINT_TEST_SIZE]

# ── Expected output fields (from serve.py OUTPUT_SCHEMA) ─────────────────────
EXPECTED_OUTPUT_FIELDS = {
    "treatment_class", "trigger_event", "mechanism", "asset",
    "expected_direction", "confidence", "causal_chain_json",
    "evidence_spans", "validation_status", "rejected_mechanism_raw",
    "retrieval_story_ids", "extraction_path", "evidence_relevance_score",
}

endpoint_results: list[dict] = []
endpoint_latencies: list[float] = []

if endpoint_ready:
    print(f"Querying endpoint with {len(endpoint_test_articles)} articles...")
    print("─" * 60)

    for i, article in enumerate(endpoint_test_articles):
        payload = {
            "headline": article["headline"],
            "body": article.get("body", "")[:2000],  # Truncate for safety
            "published_at": str(article.get("published_at", "2026-04-20T00:00:00Z")),
        }

        t0 = time.perf_counter()
        try:
            response = w.serving_endpoints.query(
                name=ENDPOINT_NAME,
                dataframe_records=[payload],
            )
            latency_ms = (time.perf_counter() - t0) * 1000
            endpoint_latencies.append(latency_ms)

            # Parse response — predictions is a list of dicts
            if hasattr(response, "predictions") and response.predictions:
                result = response.predictions[0] if isinstance(response.predictions, list) else response.predictions
            elif hasattr(response, "dataframe_records") and response.dataframe_records:
                result = response.dataframe_records[0]
            else:
                # Try to access as dict
                result = response.as_dict() if hasattr(response, "as_dict") else {}

            result["story_id"] = article.get("story_id", f"test_{i}")
            result["latency_ms"] = latency_ms
            endpoint_results.append(result)

            # Check output fields
            present_fields = set(result.keys()) & EXPECTED_OUTPUT_FIELDS
            missing_fields = EXPECTED_OUTPUT_FIELDS - set(result.keys())
            status = "✓" if not missing_fields else "⚠"
            print(f"  {status} Article {i+1}: {article['headline'][:50]}...")
            print(f"      Latency: {latency_ms:.0f}ms | Fields: {len(present_fields)}/13")
            if result.get("treatment_class"):
                print(f"      Treatment: {result['treatment_class']} | Status: {result.get('validation_status')}")

        except Exception as e:
            latency_ms = (time.perf_counter() - t0) * 1000
            endpoint_latencies.append(latency_ms)
            print(f"  ✗ Article {i+1} FAILED ({latency_ms:.0f}ms): {e}")
            endpoint_results.append({
                "story_id": article.get("story_id", f"test_{i}"),
                "validation_status": "endpoint_error",
                "latency_ms": latency_ms,
                "error": str(e),
            })

    # ── Build endpoint DataFrame ─────────────────────────────────────────────
    endpoint_df = pl.DataFrame(endpoint_results)

    # ── Schema conformance check ─────────────────────────────────────────────
    successful = [r for r in endpoint_results if "error" not in r]
    if successful:
        schema_checks = []
        for r in successful:
            present = set(r.keys()) & EXPECTED_OUTPUT_FIELDS
            schema_checks.append(len(present) == len(EXPECTED_OUTPUT_FIELDS))
        schema_conformance = sum(schema_checks) / len(schema_checks)
    else:
        schema_conformance = 0.0

    mlflow.log_metric("endpoint.schema_conformance", schema_conformance)
    mlflow.log_metric("endpoint.success_rate", len(successful) / max(len(endpoint_test_articles), 1))
    if endpoint_latencies:
        mlflow.log_metric("endpoint.mean_latency_ms", sum(endpoint_latencies) / len(endpoint_latencies))

    print(f"\nEndpoint Query Summary")
    print("─" * 45)
    print(f"  Successful:         {len(successful)}/{len(endpoint_test_articles)}")
    print(f"  Schema conformance: {schema_conformance:.0%}")
    if endpoint_latencies:
        print(f"  Mean latency:       {sum(endpoint_latencies)/len(endpoint_latencies):.0f}ms")
        print(f"  Max latency:        {max(endpoint_latencies):.0f}ms")

    # Assertions (soft — log but don't block)
    for r in successful:
        assert "treatment_class" in r or "validation_status" in r, (
            f"Missing both treatment_class and validation_status in response for {r.get('story_id')}"
        )
else:
    print("⚠ Skipping endpoint queries — endpoint not ready")
    endpoint_df = pl.DataFrame()
    endpoint_results = []

In [0]:
# ── Compare endpoint results with local graph results ────────────────────────
if endpoint_ready and endpoint_results and e2e_df.height > 0:
    # Find matching story_ids between endpoint results and local E2E results
    endpoint_story_ids = {r["story_id"] for r in endpoint_results if "error" not in r}
    local_matching = e2e_df.filter(pl.col("story_id").is_in(list(endpoint_story_ids)))

    if local_matching.height > 0:
        print(f"Endpoint vs Local Graph Consistency (n={local_matching.height} overlapping articles)")
        print("─" * 65)

        fields_to_compare = ["treatment_class", "mechanism", "validation_status", "expected_direction"]
        field_matches: dict[str, int] = {f: 0 for f in fields_to_compare}
        total_compared = 0

        for row in local_matching.iter_rows(named=True):
            sid = row["story_id"]
            ep_result = next((r for r in endpoint_results if r.get("story_id") == sid), None)
            if ep_result is None:
                continue
            total_compared += 1

            print(f"\n  Article: {sid[:40]}...")
            for field in fields_to_compare:
                local_val = row.get(field)
                ep_val = ep_result.get(field)
                match = (local_val == ep_val) or (local_val is None and ep_val is None)
                if match:
                    field_matches[field] += 1
                symbol = "=" if match else "≠"
                print(f"    {field:<22} local={str(local_val):<25} ep={str(ep_val):<25} [{symbol}]")

        # Compute consistency rate
        if total_compared > 0:
            consistency_rate = sum(field_matches.values()) / (len(fields_to_compare) * total_compared)
        else:
            consistency_rate = 0.0

        mlflow.log_metric("endpoint.consistency_rate", consistency_rate)

        print(f"\nConsistency Summary")
        print("─" * 45)
        for field, matches in field_matches.items():
            rate = matches / max(total_compared, 1)
            print(f"  {field:<22} {rate:.0%} ({matches}/{total_compared})")
        print(f"  {'Overall':<22} {consistency_rate:.0%}")

        # Note about expected non-determinism
        if consistency_rate < 1.0:
            print(f"\n  ℹ Consistency < 100% is expected due to:")
            print(f"    • LLM non-determinism (even at temperature=0.0)")
            print(f"    • Serving container vs local graph patching differences")
            print(f"    • VS index state differences (cold-start cache vs live)")
    else:
        # No overlap — run local graph on same articles used for endpoint
        print("No overlapping story_ids between endpoint test and E2E run.")
        print("Running local graph on endpoint test articles for comparison...")

        local_for_comparison: list[dict] = []
        for article in endpoint_test_articles:
            try:
                state = {
                    "headline": article["headline"],
                    "body": article.get("body", ""),
                    "published_at": str(article.get("published_at", "")),
                }
                output = graph.invoke(state)
                extraction = output.get("extraction") or {}
                local_for_comparison.append({
                    "story_id": article.get("story_id", ""),
                    "treatment_class": extraction.get("treatment_class"),
                    "mechanism": extraction.get("mechanism"),
                    "validation_status": output.get("validation_status"),
                    "expected_direction": extraction.get("direction"),
                })
            except Exception as e:
                local_for_comparison.append({
                    "story_id": article.get("story_id", ""),
                    "treatment_class": None,
                    "mechanism": None,
                    "validation_status": "local_error",
                    "expected_direction": None,
                })

        # Compare
        total_compared = 0
        field_matches = {f: 0 for f in ["treatment_class", "mechanism", "validation_status", "expected_direction"]}

        for local_r in local_for_comparison:
            sid = local_r["story_id"]
            ep_result = next((r for r in endpoint_results if r.get("story_id") == sid), None)
            if ep_result is None or "error" in ep_result:
                continue
            total_compared += 1
            for field in field_matches:
                if local_r.get(field) == ep_result.get(field):
                    field_matches[field] += 1

        consistency_rate = (
            sum(field_matches.values()) / (len(field_matches) * total_compared)
            if total_compared > 0 else 0.0
        )
        mlflow.log_metric("endpoint.consistency_rate", consistency_rate)
        print(f"\n  Consistency rate (local re-run vs endpoint): {consistency_rate:.0%}")
else:
    consistency_rate = 0.0
    if not endpoint_ready:
        print("⚠ Skipping consistency check — endpoint not ready")
    elif not endpoint_results:
        print("⚠ Skipping consistency check — no endpoint results")
    else:
        print("⚠ Skipping consistency check — no local E2E results")

In [0]:
# ── Verify AI Gateway inference tables are capturing requests ─────────────────
INFERENCE_CATALOG = "smart-trading-dev"
INFERENCE_SCHEMA = "03_gold"

# Inference table naming: Databricks uses the endpoint name (hyphens → underscores)
# and appends _payload / _payload_request_logs / _payload_assessment_logs
_ep_sanitized = ENDPOINT_NAME.replace("-", "_")
PAYLOAD_TABLE_CANDIDATES = [
    f"`{INFERENCE_CATALOG}`.`{INFERENCE_SCHEMA}`.`{_ep_sanitized}_payload`",
    f"`{INFERENCE_CATALOG}`.`{INFERENCE_SCHEMA}`.`{ENDPOINT_NAME}_payload`",
]

payload_exists = False
payload_count = 0
request_logs_exists = False
request_logs_count = 0
last_request_time = None

# ── Check payload table (try multiple naming conventions) ───────────────────
for candidate in PAYLOAD_TABLE_CANDIDATES:
    try:
        _df = spark.table(candidate)
        _count = _df.count()  # Force evaluation (spark.table is lazy)
        payload_exists = True
        payload_count = _count
        if payload_count > 0:
            last_ts = _df.agg(F.max("timestamp").alias("latest")).first()["latest"]
            last_request_time = last_ts
        print(f"  Found payload table: {candidate}")
        break
    except Exception:
        continue

if not payload_exists:
    print(f"  ℹ No payload table found (tried {len(PAYLOAD_TABLE_CANDIDATES)} names)")
    print(f"    (Expected if inference tables not yet enabled or no traffic sent)")

# ── Check request logs table ─────────────────────────────────────────────────
LOGS_CANDIDATES = [
    f"`{INFERENCE_CATALOG}`.`{INFERENCE_SCHEMA}`.`{_ep_sanitized}_payload_request_logs`",
    f"`{INFERENCE_CATALOG}`.`{INFERENCE_SCHEMA}`.`{ENDPOINT_NAME}_payload_request_logs`",
]
for candidate in LOGS_CANDIDATES:
    try:
        _df = spark.table(candidate)
        _count = _df.count()
        request_logs_exists = True
        request_logs_count = _count
        print(f"  Found request logs: {candidate}")
        break
    except Exception:
        continue

# ── Log metrics ──────────────────────────────────────────────────────────────
mlflow.log_metric("endpoint.inference_table_exists", 1.0 if payload_exists else 0.0)
mlflow.log_metric("endpoint.payload_count", payload_count)

print(f"\nInference Table Verification")
print("═" * 55)
print(f"  Payload table:      {'EXISTS' if payload_exists else 'NOT FOUND'}")
if payload_exists:
    print(f"    Rows:             {payload_count:,}")
    print(f"    Last request:     {last_request_time}")
print(f"  Request logs table: {'EXISTS' if request_logs_exists else 'NOT FOUND'}")
if request_logs_exists:
    print(f"    Rows:             {request_logs_count:,}")

# ── Soft assertion (inference tables may take minutes to populate) ────────────
if payload_exists:
    print(f"\n✓ Inference tables are active (payload: {payload_count:,} rows)")
else:
    print(f"\nℹ Inference tables not yet populated")
    print(f"  This is expected if:")
    print(f"    • Endpoint was recently created")
    print(f"    • Auto-capture config needs time to create tables (∼5-10 min)")
    print(f"    • No production traffic has been sent yet")

# ── Section 13 Summary ───────────────────────────────────────────────────────
print(f"\n{'═' * 55}")
print(f"SECTION 13 COMPLETE — Serving Endpoint Evaluation")
print(f"{'═' * 55}")
print(f"  Endpoint state:      {endpoint_state}")
print(f"  Served version:      {served_model_version}")
print(f"  Query success:       {len([r for r in endpoint_results if 'error' not in r])}/{len(endpoint_results)}")
print(f"  Consistency rate:    {consistency_rate:.0%}")
print(f"  Inference tables:    {'Active' if payload_exists else 'Pending'}")
print(f"{'═' * 55}")

---
## Section 14: VS Bootstrap Flywheel — End-to-End Validation (4 cells)

**Components:** VS index (`causal_chains_idx`), staging table (`causal_chains_vs`), VS endpoint (`mas-few-shot`)

**Architecture:**
```
news_causal_features (MV)  ───▶  causal_chains_vs (Delta, CDF)  ───▶  causal_chains_idx (VS)
  242 rows (all statuses)          118 rows (accepted only)              118 embeddings (ONLINE)
```

**Key questions:**
1. Is the VS index ONLINE and healthy?
2. Does staging table count match gold table accepted count?
3. Are VS search results semantically relevant for diverse query types?
4. What flywheel stage are we in? (cold-start / bootstrapping / self-improving)

### Cell 14.1 `[python]` — VS Index Health Check

**Logic:**
1. `vsc = VectorSearchClient(disable_notice=True)`
2. `idx = vsc.get_index(endpoint_name='mas-few-shot', index_name='smart-trading-dev.03_gold.causal_chains_idx')`
3. `idx.describe()` → check: status, num_docs, last_sync_time, index_type
4. Query staging table: `SELECT COUNT(*) FROM causal_chains_vs`
5. Query gold MV: `SELECT COUNT(*) FROM news_causal_features WHERE validation_status = 'accepted'`
6. Compare: staging count should match gold accepted count

**Variables created:**
* `vs_status: str`, `vs_doc_count: int`, `staging_count: int`, `gold_accepted_count: int`

**Assertions:**
```python
assert vs_status == 'ONLINE', f"VS index not ONLINE: {vs_status}"
assert staging_count == gold_accepted_count, f"Staging ({staging_count}) != gold accepted ({gold_accepted_count})"
assert vs_doc_count == staging_count, f"VS docs ({vs_doc_count}) != staging ({staging_count})"
```

### Cell 14.2 `[python]` — VS Semantic Quality Test

**Purpose:** Query the VS index with diverse headlines covering all 8 L1 categories.

**Logic:**
1. Craft 8 representative headlines (one per L1 category):
   * Regulation: "SEC approves spot Bitcoin ETF applications"
   * Macro: "Federal Reserve cuts interest rates by 50 basis points"
   * Adoption: "MicroStrategy purchases additional 10,000 BTC"
   * Security: "Major exchange confirms $200M hack"
   * Supply: "Bitcoin halving complete, block reward drops"
   * Sentiment: "Dormant whale moves $500M BTC to Coinbase"
   * DeFi: "Uniswap activates fee switch for token holders"
   * Exchange: "Binance announces spot listing for new token"
2. For each, query VS index (top-5), check if returned treatment_classes match the query L1
3. Compute: `semantic_precision` = fraction of top-1 results matching expected L1

**Variables created:**
* `semantic_quality_df: pl.DataFrame`
* `semantic_precision: float`

**Metrics logged:** `vs.semantic_precision`

### Cell 14.3 `[python]` — Flywheel Stage Detection + Assertions

**Logic:**
1. Read `gold_accepted_count` from Cell 14.1
2. Determine stage:
   * 0 chains → COLD START
   * 1-49 chains → BOOTSTRAPPING
   * 50+ chains → SELF-IMPROVING
3. Verify `config.router.min_index_size == 50` (the threshold)

**Output:**
```
═══ FLYWHEEL STATUS ═══
  Accepted chains:  118
  Threshold:        50
  Current stage:    SELF-IMPROVING ✓
  VS index status:  ONLINE
  VS documents:     118
  Staging table:    118 rows
  Semantic precision: XX%
```

**Assertions:**
```python
assert gold_accepted_count >= config.router.min_index_size, f"Not in self-improving stage: {gold_accepted_count} < {config.router.min_index_size}"
assert semantic_precision >= 0.50, f"Semantic precision {semantic_precision:.2%} < 50%"
```

In [0]:
from databricks.vector_search.client import VectorSearchClient

VS_ENDPOINT_NAME = config.vs_endpoint_name   # "mas-few-shot"
VS_INDEX_NAME = config.vs_index_name          # "smart-trading-dev.03_gold.causal_chains_idx"
STAGING_TABLE = f"`smart-trading-dev`.`03_gold`.`causal_chains_vs`"

vs_status = "UNKNOWN"
vs_doc_count = 0
staging_count = 0
gold_accepted_count_vs = 0  # Rename to avoid collision with Section 12 variable
vs_last_sync = None
vs_index_accessible = False

# ── 1. Check VS Index via VectorSearchClient ──────────────────────────────
try:
    vsc = VectorSearchClient(disable_notice=True)
    idx = vsc.get_index(
        endpoint_name=VS_ENDPOINT_NAME,
        index_name=VS_INDEX_NAME,
    )
    idx_desc = idx.describe()

    vs_status = idx_desc.get("status", {}).get("ready", "UNKNOWN")
    # Handle different response shapes from describe()
    if isinstance(vs_status, bool):
        vs_status = "ONLINE" if vs_status else "NOT_READY"
    elif isinstance(vs_status, str):
        vs_status = vs_status.upper()

    vs_doc_count = idx_desc.get("status", {}).get("num_documents", 0) or 0
    # Alternative location for num_docs
    if vs_doc_count == 0:
        vs_doc_count = idx_desc.get("num_documents", 0) or 0
    # Try indexed_row_count
    if vs_doc_count == 0:
        vs_doc_count = idx_desc.get("status", {}).get("indexed_row_count", 0) or 0

    vs_last_sync = idx_desc.get("status", {}).get("last_update_time", None)
    vs_index_accessible = True

    print(f"✓ VS Index accessible: {VS_INDEX_NAME}")
    print(f"  Status:     {vs_status}")
    print(f"  Documents:  {vs_doc_count}")
    print(f"  Last sync:  {vs_last_sync}")

except Exception as e:
    print(f"✗ VS Index not accessible: {e}")
    print(f"  Endpoint: {VS_ENDPOINT_NAME}")
    print(f"  Index:    {VS_INDEX_NAME}")

# ── 2. Query staging table count ───────────────────────────────────────────
try:
    staging_count = spark.table(STAGING_TABLE).count()
    print(f"\n  Staging table ({STAGING_TABLE}): {staging_count} rows")
except Exception as e:
    print(f"\n  ⚠ Staging table not found: {e}")

# ── 3. Query gold table accepted count ─────────────────────────────────────
try:
    gold_accepted_count_vs = (
        spark.table(GOLD_TABLE)
        .filter(F.col("validation_status") == "accepted")
        .count()
    )
    print(f"  Gold table accepted count: {gold_accepted_count_vs}")
except Exception as e:
    print(f"  ⚠ Gold table query failed: {e}")
    # Fallback to the variable from Section 12 if available
    gold_accepted_count_vs = accepted_gold if 'accepted_gold' in dir() else 0
    print(f"  Using Section 12 fallback: {gold_accepted_count_vs}")

# ── 4. Consistency checks ───────────────────────────────────────────────────
print(f"\nVS Pipeline Consistency")
print("─" * 50)
print(f"  Gold accepted:    {gold_accepted_count_vs}")
print(f"  Staging rows:     {staging_count}")
print(f"  VS doc count:     {vs_doc_count}")

staging_gold_match = staging_count == gold_accepted_count_vs
vs_staging_match = vs_doc_count == staging_count

print(f"  Staging == Gold:  {'✓' if staging_gold_match else '✗'} ({staging_count} vs {gold_accepted_count_vs})")
print(f"  VS == Staging:    {'✓' if vs_staging_match else '✗'} ({vs_doc_count} vs {staging_count})")

# ── Log metrics ──────────────────────────────────────────────────────────────
mlflow.log_metric("vs.index_online", 1.0 if vs_status == "ONLINE" else 0.0)
mlflow.log_metric("vs.doc_count", vs_doc_count)
mlflow.log_metric("vs.staging_count", staging_count)
mlflow.log_metric("vs.gold_accepted_count", gold_accepted_count_vs)

# ── Assertions (soft — warn but continue) ─────────────────────────────────
if vs_index_accessible:
    assert vs_status == "ONLINE", f"VS index not ONLINE: {vs_status}"
if staging_count > 0 and gold_accepted_count_vs > 0:
    # Allow small discrepancy due to sync timing
    assert abs(staging_count - gold_accepted_count_vs) <= 5, (
        f"Staging ({staging_count}) != gold accepted ({gold_accepted_count_vs}) — sync may be in progress"
    )

In [0]:
# ── 8 representative headlines (one per L1 taxonomy category) ─────────────────
SEMANTIC_TEST_QUERIES = [
    {"headline": "SEC approves spot Bitcoin ETF applications", "expected_l1": "regulation"},
    {"headline": "Federal Reserve cuts interest rates by 50 basis points", "expected_l1": "macro"},
    {"headline": "MicroStrategy purchases additional 10,000 BTC", "expected_l1": "adoption"},
    {"headline": "Major exchange confirms $200M hack affecting user funds", "expected_l1": "security"},
    {"headline": "Bitcoin halving complete, block reward drops to 3.125 BTC", "expected_l1": "supply"},
    {"headline": "Dormant whale moves $500M BTC to Coinbase deposit address", "expected_l1": "sentiment"},
    {"headline": "Uniswap activates fee switch for token holders", "expected_l1": "defi"},
    {"headline": "Binance announces spot listing for new token pair", "expected_l1": "exchange"},
]

# VS index columns (from causal_chains_vs schema):
# story_id, treatment_class, mechanism, expected_direction, confidence, trigger_event, causal_chain_json, validation_status
VS_FETCH_COLUMNS = ["treatment_class", "mechanism", "trigger_event"]

semantic_results: list[dict] = []
semantic_precision = 0.0

if vs_index_accessible and vs_doc_count > 0:
    print(f"VS Semantic Quality Test ({len(SEMANTIC_TEST_QUERIES)} queries, top-5 each)")
    print("─" * 70)

    top1_matches = 0

    for q in SEMANTIC_TEST_QUERIES:
        try:
            search_results = idx.similarity_search(
                query_text=q["headline"],
                columns=VS_FETCH_COLUMNS,
                num_results=5,
            )

            # Parse results — handle different response formats
            result_data = search_results.get("result", {}).get("data_array", [])
            result_cols = [c["name"] for c in search_results.get("manifest", {}).get("columns", [])]

            top_results: list[dict] = []
            for row in result_data:
                row_dict = dict(zip(result_cols, row)) if result_cols else {}
                top_results.append(row_dict)

            # Check if top-1 result's treatment_class L1 matches expected
            top1_match = False
            top1_treatment = None
            if top_results:
                top1_treatment = top_results[0].get("treatment_class", "")
                if top1_treatment:
                    top1_l1 = top1_treatment.split(".")[0].lower()
                    top1_match = top1_l1 == q["expected_l1"].lower()
                    if top1_match:
                        top1_matches += 1

            semantic_results.append({
                "query_headline": q["headline"],
                "expected_l1": q["expected_l1"],
                "top1_treatment": top1_treatment,
                "top1_match": top1_match,
                "num_results": len(top_results),
            })

            match_symbol = "✓" if top1_match else "✗"
            print(f"  {match_symbol} [{q['expected_l1']:<10}] \"{q['headline'][:45]}...\"")
            print(f"      Top-1: {top1_treatment or '(empty)'} | Results: {len(top_results)}")

        except Exception as e:
            print(f"  ✗ [{q['expected_l1']:<10}] Query failed: {e}")
            semantic_results.append({
                "query_headline": q["headline"],
                "expected_l1": q["expected_l1"],
                "top1_treatment": None,
                "top1_match": False,
                "num_results": 0,
            })

    # Compute semantic precision
    semantic_precision = top1_matches / max(len(SEMANTIC_TEST_QUERIES), 1)

    print(f"\nSemantic Precision: {semantic_precision:.0%} ({top1_matches}/{len(SEMANTIC_TEST_QUERIES)})")

    # Build results DataFrame
    semantic_quality_df = pl.DataFrame(semantic_results)

    mlflow.log_metric("vs.semantic_precision", semantic_precision)
    mlflow.log_metric("vs.queries_with_results", sum(1 for r in semantic_results if r["num_results"] > 0))

else:
    if not vs_index_accessible:
        print("⚠ Skipping semantic quality test — VS index not accessible")
    else:
        print("⚠ Skipping semantic quality test — VS index is empty (0 documents)")
    semantic_quality_df = pl.DataFrame()

In [0]:
# ── Flywheel stage thresholds ────────────────────────────────────────────────
MIN_INDEX_SIZE = config.router.min_index_size  # 50

if gold_accepted_count_vs == 0:
    flywheel_stage = "COLD START"
elif gold_accepted_count_vs < MIN_INDEX_SIZE:
    flywheel_stage = "BOOTSTRAPPING"
else:
    flywheel_stage = "SELF-IMPROVING"

# ── Log metrics ──────────────────────────────────────────────────────────────
mlflow.log_param("vs.flywheel_stage", flywheel_stage)
mlflow.log_metric("vs.min_index_threshold", MIN_INDEX_SIZE)

# ── Summary output ───────────────────────────────────────────────────────────
_stage_symbol = "✓" if flywheel_stage == "SELF-IMPROVING" else ""
print("═" * 55)
print("  FLYWHEEL STATUS")
print("═" * 55)
print(f"  Accepted chains:    {gold_accepted_count_vs}")
print(f"  Threshold:          {MIN_INDEX_SIZE}")
print(f"  Current stage:      {flywheel_stage} {_stage_symbol}")
print(f"  VS index status:    {vs_status}")
print(f"  VS documents:       {vs_doc_count}")
print(f"  Staging table:      {staging_count} rows")
print(f"  Semantic precision: {semantic_precision:.0%}")
print("═" * 55)

# ── Stage progress visualization ───────────────────────────────────────────
fig_flywheel = go.Figure()

# Gauge chart for flywheel stage
max_range = max(gold_accepted_count_vs * 1.5, MIN_INDEX_SIZE * 2)
fig_flywheel.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=gold_accepted_count_vs,
    title={"text": "Accepted Causal Chains", "font": {"color": "#E6EDF3"}},
    delta={"reference": MIN_INDEX_SIZE, "increasing": {"color": "#52C41A"}},
    gauge={
        "axis": {"range": [0, max_range], "tickcolor": "#8B949E"},
        "bar": {"color": "#58A6FF"},
        "bgcolor": "#21262D",
        "borderwidth": 2,
        "bordercolor": "#30363D",
        "steps": [
            {"range": [0, 1], "color": "#FF4D4F"},            # Cold Start
            {"range": [1, MIN_INDEX_SIZE], "color": "#F0883E"},  # Bootstrapping
            {"range": [MIN_INDEX_SIZE, max_range], "color": "#52C41A"},  # Self-improving
        ],
        "threshold": {
            "line": {"color": "#E6EDF3", "width": 4},
            "thickness": 0.75,
            "value": MIN_INDEX_SIZE,
        },
    },
))

fig_flywheel.update_layout(
    title=f"VS Flywheel Stage: {flywheel_stage}",
    template="plotly_dark",
    plot_bgcolor="#0D1117",
    paper_bgcolor="#161B22",
    font_color="#E6EDF3",
    height=350,
)
mlflow.log_figure(fig_flywheel, "vs_flywheel_gauge.html")
fig_flywheel.show()

# ── Assertions ───────────────────────────────────────────────────────────────
assert gold_accepted_count_vs >= MIN_INDEX_SIZE, (
    f"Not in self-improving stage: {gold_accepted_count_vs} < {MIN_INDEX_SIZE}"
)

# Semantic precision gate: soft assertion (log + warn, don't block)
# Low precision is expected during early flywheel stages when the index
# is dominated by one category (Adoption.corporate_treasury = most articles).
semantic_gate_passed = semantic_precision >= 0.50
mlflow.log_metric("vs.semantic_gate_passed", 1.0 if semantic_gate_passed else 0.0)

if not semantic_gate_passed:
    print(f"\n⚠ Semantic precision {semantic_precision:.0%} < 50% threshold")
    print(f"  Root cause: Index category imbalance (most chains are Adoption.*)")
    print(f"  Recommendation: Seed more diverse articles across all 8 L1 categories")
else:
    print(f"\n✓ Semantic precision gate passed: {semantic_precision:.0%}")

print(f"\n✓ Section 14 hard assertions passed (flywheel stage: {flywheel_stage})")
_sep = "═" * 55
print(f"\n{_sep}")
print("SECTION 14 COMPLETE — VS Bootstrap Flywheel")
print(_sep)

---
### Re-log MAS Agent Model (v7)

**Fixes bundled in this version:**
1. `serve.py` — `expected_direction` now reads the correct key from LLM output (was reading stale `direction` key)
2. `serve.py` — `confidence` field now extracted from LLM output (was hardcoded to `None`)
3. `schemas.py` — `CausalChain` Pydantic model renamed `direction` → `expected_direction`, added `confidence` and `mechanism_candidate` fields

In [0]:
import importlib
import json
import tempfile
from pathlib import Path

import trading_databricks.agents.serve
importlib.reload(trading_databricks.agents.serve)

from trading_databricks.agents.serve import (
    MASPyfuncModel,
    MODEL_SIGNATURE,
    _extract_causal_features,
    _serialize_whitelist,
)
from mlflow.models.resources import (
    DatabricksServingEndpoint,
    DatabricksVectorSearchIndex,
)

# ── Quick sanity check: verify fix is loaded ──────────────────────────────
_test_state = {
    "extraction": {"expected_direction": "bullish", "confidence": 0.9},
    "validation_status": "accepted",
}
_test_out = _extract_causal_features(_test_state)
assert _test_out["expected_direction"] == "bullish", "Fix not loaded!"
assert _test_out["confidence"] == 0.9, "Confidence fix not loaded!"
print("✓ serve.py fixes verified in memory")

# ── Bundle taxonomy + whitelist as JSON artifacts ─────────────────────────
tmp = Path(tempfile.mkdtemp())
taxonomy_path = tmp / "taxonomy.json"
whitelist_path = tmp / "whitelist.json"
taxonomy_path.write_text(json.dumps(taxonomy, default=str))
whitelist_path.write_text(json.dumps(_serialize_whitelist(whitelist), default=str))
print(f"  Artifacts serialized: taxonomy ({len(taxonomy)} entries), whitelist ({len(whitelist)} entries)")

input_example = pd.DataFrame([{
    "headline": "Fed cuts interest rates by 50 basis points",
    "body": (
        "The Federal Reserve announced a 50 basis point rate cut, "
        "signaling a shift toward monetary easing."
    ),
    "published_at": "2026-04-14T00:00:00Z",
}])

# ── Log model in a dedicated MLflow run ──────────────────────────────────
print("\nLogging MAS agent model to Unity Catalog...")

with mlflow.start_run(run_name="relog_model_v7_direction_fix", nested=True) as relog_run:
    mlflow.log_param("fix_description", "expected_direction key + confidence extraction")
    mlflow.log_param("changes", "serve.py, schemas.py")

    model_info = mlflow.pyfunc.log_model(
        artifact_path="mas-news-causal",
        python_model=MASPyfuncModel(),
        registered_model_name="smart-trading-dev.03_gold.mas_news_causal",
        signature=MODEL_SIGNATURE,
        input_example=input_example,
        artifacts={
            "taxonomy": str(taxonomy_path),
            "whitelist": str(whitelist_path),
        },
        pip_requirements=[
            "langgraph>=1.1.6",
            "spacy==3.8.4",
            "pydantic>=2.0",
            "mlflow>=2.18",
        ],
        resources=[
            DatabricksServingEndpoint(endpoint_name="databricks-meta-llama-3-1-8b-instruct"),
            DatabricksServingEndpoint(endpoint_name="databricks-gte-large-en"),
            DatabricksVectorSearchIndex(index_name="smart-trading-dev.03_gold.causal_chains_idx"),
        ],
    )

    print(f"\n✓ Model logged successfully")
    print(f"  Run ID:     {relog_run.info.run_id}")
    print(f"  Model URI:  {model_info.model_uri}")
    print(f"  Registered: smart-trading-dev.03_gold.mas_news_causal")
    print(f"\n  To deploy: update endpoint 'mas-news-causal' to serve this new version")

---
### Section 14.5: Prompt Versioning & Comparison (P0 Enhancement)

**Objective 3.5:** *Create a prompt that adjusts an LLM's response from baseline to desired output*

**Approach:**
1. Load the production `hypothesis_generator` prompt (baseline)
2. Create an optimized variant with explicit chain-of-thought + structured output enforcement
3. Run both prompts on the same 5-article subset (from E2E results)
4. Compare: schema_valid rate, treatment taxonomy compliance, evidence verbatim rate
5. Log both prompt versions and comparison metrics to MLflow

**Why this matters:** Prompt engineering is invisible if we don't track versions. This demonstrates the baseline → optimized workflow required for systematic prompt development.

In [0]:
# ── Prompt Versioning: Baseline vs Optimized ───────────────────────────
import importlib
import yaml
from pathlib import Path

import trading_databricks.agents.prompts
importlib.reload(trading_databricks.agents.prompts)
from trading_databricks.agents.prompts import load_prompt, render_messages

PROMPTS_DIR = Path(
    "/Workspace/Users/aaronng.dev1@gmail.com/trading-databricks"
    "/src/trading_databricks/agents/prompts"
)

# ── 1. Load baseline prompt (production version from YAML) ──────────────
# The hypothesis_generator prompt is the core extraction prompt
try:
    baseline_messages = load_prompt("hypothesis_generator", version="production")
    baseline_content = baseline_messages[0]["content"] if baseline_messages else ""
    print(f"✓ Baseline prompt loaded: {len(baseline_content)} chars")
    print(f"  First 200 chars: {baseline_content[:200]}...")
except Exception as e:
    # Fall back to YAML
    yaml_path = PROMPTS_DIR / "hypothesis_generator.yaml"
    if yaml_path.exists():
        doc = yaml.safe_load(yaml_path.read_text())
        baseline_messages = [dict(m) for m in doc["messages"]]
        baseline_content = baseline_messages[0]["content"]
        print(f"✓ Baseline loaded from YAML: {len(baseline_content)} chars")
    else:
        baseline_content = ""
        baseline_messages = []
        print(f"⚠ Could not load baseline: {e}")

# ── 2. Load all production prompts for inventory ─────────────────────
prompt_files = sorted(PROMPTS_DIR.glob("*.yaml"))
print(f"\nPrompt Registry Inventory ({len(prompt_files)} prompts):")
print("-" * 70)
for pf in prompt_files:
    doc = yaml.safe_load(pf.read_text())
    messages = doc.get("messages", [])
    total_chars = sum(len(m.get("content", "")) for m in messages)
    n_vars = sum(m.get("content", "").count("{") for m in messages) // 2  # rough count
    print(f"  {pf.stem:<35} {len(messages)} msgs  {total_chars:>5} chars  ~{n_vars} vars")

# ── 3. Create optimized prompt variant ───────────────────────────────
# Add chain-of-thought reasoning and stricter output formatting
OPTIMIZED_SUFFIX = """

## Chain-of-Thought Reasoning (REQUIRED)
Before generating the JSON output, you MUST think step-by-step:
1. **Identify the trigger event**: What specific news event is described?
2. **Classify the treatment**: Which taxonomy category (L1.L2) best matches?
3. **Determine the mechanism**: Through what causal mechanism does this event affect crypto markets?
4. **Assess direction**: Based on the mechanism, is the expected impact bullish, bearish, or neutral?
5. **Ground in evidence**: Quote exact spans from the article body that support your reasoning.
6. **Calibrate confidence**: How certain are you? Consider: (a) how direct the causal link is, (b) whether the evidence is specific or vague, (c) historical precedent strength.

## Output Format Enforcement
- ALL fields in the JSON output are REQUIRED. Missing fields = invalid output.
- `evidence_spans` MUST be EXACT verbatim quotes from the article body. No paraphrasing.
- `confidence` MUST be a float between 0.0 and 1.0. Use 0.3-0.5 for weak evidence, 0.6-0.8 for moderate, 0.8+ for strong direct evidence.
- `expected_direction` MUST be one of: "bullish", "bearish", "neutral".
"""

if baseline_content:
    optimized_content = baseline_content + OPTIMIZED_SUFFIX
    optimized_messages = [{"role": baseline_messages[0]["role"], "content": optimized_content}]
    if len(baseline_messages) > 1:
        optimized_messages.extend(baseline_messages[1:])
    print(f"\n✓ Optimized prompt created: {len(optimized_content)} chars (+{len(OPTIMIZED_SUFFIX)} chars added)")
else:
    optimized_messages = []
    print("⚠ No baseline to optimize")

# ── 4. Log both versions to MLflow ───────────────────────────────────
import json

if baseline_content:
    mlflow.log_text(json.dumps(baseline_messages, indent=2), "prompts/hypothesis_generator_baseline.json")
    mlflow.log_text(json.dumps(optimized_messages, indent=2), "prompts/hypothesis_generator_optimized.json")
    mlflow.log_metric("prompt.baseline_chars", len(baseline_content))
    mlflow.log_metric("prompt.optimized_chars", len(optimized_content))
    mlflow.log_metric("prompt.cot_suffix_chars", len(OPTIMIZED_SUFFIX))
    print("✓ Both prompt versions logged to MLflow artifacts")

In [0]:
# ── Analyze prompt effectiveness using existing E2E results ─────────────
import plotly.graph_objects as go

# The current E2E results (Section 11) were generated with the production
# (baseline) prompt. We analyze the quality distribution to establish
# the baseline metrics that a future optimized prompt must beat.

if e2e_df.height > 0 and baseline_content:
    # ── Current prompt performance (baseline) — from e2e_metrics dict ──
    baseline_metrics = {
        "schema_valid": e2e_metrics.get("schema_valid", 0),
        "taxonomy_compliance": e2e_metrics.get("treatment_in_taxonomy", 0),
        "mechanism_compliance": e2e_metrics.get("mechanism_in_whitelist", 0),
        "evidence_verbatim": e2e_metrics.get("evidence_spans_verbatim", 0),
        "validator_accepted": e2e_metrics.get("validator_accepted", 0),
    }

    print("═" * 60)
    print("  PROMPT VERSION COMPARISON BASELINE")
    print("═" * 60)
    print(f"  Prompt version: production (hypothesis_generator)")
    print(f"  Prompt size: {len(baseline_content):,} chars")
    print(f"  Test set: {e2e_df.height} articles")
    print("─" * 60)
    print(f"  {'Metric':<30} {'Score':>8}")
    print("  " + "-" * 42)
    for metric_name, value in baseline_metrics.items():
        bar = "█" * int(value * 20) + "░" * (20 - int(value * 20))
        print(f"  {metric_name:<30} {value:>8.3f}  {bar}")
    print("─" * 60)

    # Log baseline metrics for future comparison
    for k, v in baseline_metrics.items():
        mlflow.log_metric(f"prompt.baseline.{k}", v)

    # ── Identify failure patterns from e2e_scores_df (Cell 11.2) ─────
    # e2e_scores_df has per-article scorer columns; e2e_df only has graph outputs
    failure_analysis = {}
    try:
        if e2e_scores_df.height > 0:
            for col in ["schema_valid", "treatment_in_taxonomy", "mechanism_in_whitelist", "evidence_spans_verbatim"]:
                if col in e2e_scores_df.columns:
                    n_fail = e2e_scores_df.filter(pl.col(col) < 1.0).height
                    failure_analysis[f"{col}_failures"] = n_fail
    except NameError:
        # e2e_scores_df may not exist if Section 11.2 wasn't run
        pass

    if failure_analysis:
        print(f"\nFailure Analysis (areas for prompt optimization):")
        for failure_type, count in failure_analysis.items():
            total = e2e_df.height
            pct = count / total * 100 if total > 0 else 0
            indicator = "🟢" if pct < 10 else ("🟡" if pct < 30 else "🔴")
            print(f"  {indicator} {failure_type:<35} {count:>3}/{total} ({pct:.1f}%)")

    # ── Visualization: Baseline quality distribution ─────────────────
    fig = go.Figure()

    metric_names = list(baseline_metrics.keys())
    metric_values = list(baseline_metrics.values())
    colors = ["#58A6FF", "#52C41A", "#F0883E", "#A371F7", "#FF4D4F"]

    fig.add_trace(go.Bar(
        x=metric_names,
        y=metric_values,
        marker_color=colors[:len(metric_names)],
        text=[f"{v:.3f}" for v in metric_values],
        textposition="outside",
        textfont_color="#E6EDF3",
    ))

    # Add target line at 0.9 (our quality gate)
    fig.add_hline(
        y=0.9, line_dash="dash", line_color="#52C41A",
        annotation_text="Quality Gate (0.9)",
        annotation_font_color="#52C41A",
    )
    fig.add_hline(
        y=0.5, line_dash="dot", line_color="#F0883E",
        annotation_text="Minimum (0.5)",
        annotation_font_color="#F0883E",
    )

    fig.update_layout(
        title="Baseline Prompt Quality Metrics (hypothesis_generator)",
        xaxis_title="Metric",
        yaxis_title="Score",
        yaxis=dict(range=[0, 1.15], gridcolor="#21262D"),
        plot_bgcolor="#0D1117",
        paper_bgcolor="#161B22",
        font_color="#E6EDF3",
    )
    mlflow.log_figure(fig, "prompt_baseline_quality.html")
    fig.show()

    print("\n✓ Baseline established. Run with optimized prompt to compare:")
    print("  1. Update prompts/hypothesis_generator.yaml with CoT suffix")
    print("  2. Re-run Section 11 (Full Graph E2E)")
    print("  3. Compare prompt.baseline.* vs prompt.optimized.* in MLflow")
else:
    print("⚠ No E2E results or baseline prompt available for comparison")

---
## Section 15: Summary Dashboard (2 cells)

**Purpose:** Executive summary of all component evaluations. Single source of truth for MAS health.

### Cell 15.1 `[python]` — Summary Metrics Table + MLflow End Run

**Purpose:** Compile all metrics into a single summary table and end the MLflow run.

**Output: Master summary table**

| Section | Component | Key Metric | Value | Threshold | Status |
| --- | --- | --- | --- | --- | --- |
| 1 | Gatekeeper | spaCy/RLIKE agreement | X.X% | ≥ 90% | PASS/FAIL |
| 2 | Context Retriever | Stub validation | N/N pass | All None | PASS/FAIL |
| 3 | Few-Shot Retriever | Mean top-1 similarity | X.XX | > 0.50 | PASS/FAIL |
| 4 | Uncertainty Router | All scenarios pass | N/N | 8/8 | PASS/FAIL |
| 5 | Evidence Extractor | Parse success rate | X.X% | ≥ 80% | PASS/FAIL |
| 6 | RAG Retriever | Graceful degradation | N/N | All pass | PASS/FAIL |
| 7 | Hypothesis Generator | Parse success rate | X.X% | ≥ 70% | PASS/FAIL |
| 7 | Hypothesis Generator | Taxonomy compliance | X.X% | (no gate) | INFO |
| 7 | Hypothesis Generator | Whitelist compliance | X.X% | (no gate) | INFO |
| 8 | Judge Panel | Panel pass rate | X.X% | 20-95% | PASS/FAIL |
| 9 | Reflector | All scenarios pass | N/N | 7/7 | PASS/FAIL |
| 10 | Mechanism Validator | Acceptance rate | X.X% | ≥ 40% | PASS/FAIL |
| 11 | Full Graph E2E | schema_valid | X.XXX | > 0.9 | PASS/FAIL |
| 11 | Full Graph E2E | validator_accepted | X.XXX | > 0.5 | PASS/FAIL |
| 12 | Gold Table | Row count | N | > 0 | PASS/FAIL |
| 12 | Gold Table | Freshness | Xh | < 168h | PASS/FAIL |
| 13 | Serving Endpoint | Responsive | Y/N | READY | PASS/FAIL |
| 14 | VS Flywheel | Stage | self-improving | ≥ 50 chains | PASS/FAIL |
| 14 | VS Flywheel | Semantic precision | X.X% | ≥ 50% | PASS/FAIL |

**Logic:**
1. Build the table from all logged metrics
2. Compute overall: `all_passed = all(status == 'PASS' for required_gates)`
3. `mlflow.end_run()` with status = 'FINISHED' or 'FAILED'

**Print:** Full formatted table + overall verdict.

In [0]:
import plotly.graph_objects as go
from collections import OrderedDict

# ── Retrieve all logged metrics from active MLflow run ────────────────────────
run_data = mlflow.get_run(run.info.run_id).data
all_metrics = run_data.metrics
print(f"Total metrics logged: {len(all_metrics)}")

# ── Define the summary table structure ────────────────────────────────────────
# Each row: (section, component, metric_display_name, mlflow_key_or_variable, threshold, comparison)
# comparison: ">=" means value >= threshold → PASS

SUMMARY_SPEC: list[dict] = [
    {"section": 1, "component": "Gatekeeper", "metric": "spaCy/RLIKE agreement",
     "key": "gatekeeper.spacy_sql_agreement", "threshold": 0.90, "cmp": ">=", "fmt": ".1%"},
    {"section": 2, "component": "Context Retriever", "metric": "Stub validation",
     "key": "context_retriever.all_none", "threshold": 1.0, "cmp": ">=", "fmt": ".0%"},
    {"section": 3, "component": "Few-Shot Retriever", "metric": "Mean top-1 similarity",
     "key": "few_shot.mean_top1_similarity", "threshold": 0.50, "cmp": ">", "fmt": ".4f"},
    {"section": 4, "component": "Uncertainty Router", "metric": "All scenarios pass",
     "key": "router.all_scenarios_pass", "threshold": 1.0, "cmp": ">=", "fmt": ".0f"},
    {"section": 5, "component": "Evidence Extractor", "metric": "Parse success rate",
     "key": "evidence.parse_success_rate", "threshold": 0.80, "cmp": ">=", "fmt": ".1%"},
    {"section": 6, "component": "RAG Retriever", "metric": "Graceful degradation",
     "key": "rag.graceful_degradation", "threshold": 1.0, "cmp": ">=", "fmt": ".0f"},
    {"section": 7, "component": "Hypothesis Generator", "metric": "Parse success rate",
     "key": "hypothesis.parse_success_rate", "threshold": 0.70, "cmp": ">=", "fmt": ".1%"},
    {"section": 7, "component": "Hypothesis Generator", "metric": "Taxonomy compliance",
     "key": "hypothesis.taxonomy_compliance", "threshold": None, "cmp": "info", "fmt": ".1%"},
    {"section": 7, "component": "Hypothesis Generator", "metric": "Whitelist compliance",
     "key": "hypothesis.whitelist_compliance", "threshold": None, "cmp": "info", "fmt": ".1%"},
    {"section": 8, "component": "Judge Panel", "metric": "Panel pass rate",
     "key": "judge.panel_pass_rate", "threshold": 0.20, "cmp": ">=", "fmt": ".1%"},
    {"section": 9, "component": "Reflector", "metric": "All scenarios pass",
     "key": "reflector.all_scenarios_pass", "threshold": 1.0, "cmp": ">=", "fmt": ".0f"},
    {"section": 10, "component": "Mechanism Validator", "metric": "Acceptance rate",
     "key": "validator.acceptance_rate", "threshold": 0.40, "cmp": ">=", "fmt": ".1%"},
    {"section": 11, "component": "Full Graph E2E", "metric": "schema_valid",
     "key": "e2e.schema_valid", "threshold": 0.90, "cmp": ">", "fmt": ".3f"},
    {"section": 11, "component": "Full Graph E2E", "metric": "validator_accepted",
     "key": "e2e.validator_accepted", "threshold": 0.50, "cmp": ">", "fmt": ".3f"},
    {"section": 12, "component": "Gold Table", "metric": "Row count",
     "key": "gold.row_count", "threshold": 0, "cmp": ">", "fmt": ".0f"},
    {"section": 12, "component": "Gold Table", "metric": "Freshness (hours)",
     "key": "gold.hours_since_last_process", "threshold": 168, "cmp": "<", "fmt": ".1f"},
    {"section": 13, "component": "Serving Endpoint", "metric": "Responsive",
     "key": "endpoint.responsive", "threshold": 1.0, "cmp": ">=", "fmt": ".0f"},
    {"section": 14, "component": "VS Flywheel", "metric": "Accepted chains",
     "key": "vs.accepted_chain_count", "threshold": 50, "cmp": ">=", "fmt": ".0f"},
    {"section": 14, "component": "VS Flywheel", "metric": "Semantic precision",
     "key": "vs.semantic_precision", "threshold": 0.50, "cmp": ">=", "fmt": ".1%"},
]

# ── Build summary rows ────────────────────────────────────────────────────────
def _evaluate_gate(value: float | None, threshold: float | None, cmp: str) -> str:
    if value is None:
        return "N/A"
    if cmp == "info":
        return "INFO"
    if threshold is None:
        return "INFO"
    checks = {
        ">=": value >= threshold,
        ">": value > threshold,
        "<": value < threshold,
        "<=": value <= threshold,
    }
    return "PASS" if checks.get(cmp, False) else "FAIL"

summary_rows: list[dict] = []
for spec in SUMMARY_SPEC:
    value = all_metrics.get(spec["key"])
    status = _evaluate_gate(value, spec["threshold"], spec["cmp"])

    # Format value
    if value is not None:
        fmt = spec["fmt"]
        if "%" in fmt:
            display_value = f"{value:{fmt}}"
        else:
            display_value = f"{value:{fmt}}"
    else:
        display_value = "—"

    # Format threshold
    if spec["threshold"] is not None and spec["cmp"] != "info":
        cmp_symbol = {">=": "\u2265", ">": ">", "<": "<", "<=": "\u2264"}.get(spec["cmp"], "")
        if "%" in spec["fmt"]:
            thr_display = f"{cmp_symbol} {spec['threshold']:{spec['fmt']}}"
        else:
            thr_display = f"{cmp_symbol} {spec['threshold']:{spec['fmt']}}"
    else:
        thr_display = "(no gate)"

    summary_rows.append({
        "Section": spec["section"],
        "Component": spec["component"],
        "Key Metric": spec["metric"],
        "Value": display_value,
        "Threshold": thr_display,
        "Status": status,
    })

summary_df = pl.DataFrame(summary_rows)

# ── Overall verdict ───────────────────────────────────────────────────────────
required_gates = [r for r in summary_rows if r["Status"] not in ("INFO", "N/A")]
passed_gates = [r for r in required_gates if r["Status"] == "PASS"]
failed_gates = [r for r in required_gates if r["Status"] == "FAIL"]
na_gates = [r for r in required_gates if r["Status"] == "N/A"]

all_gates_passed = len(failed_gates) == 0 and len(na_gates) == 0
overall_status = "\u2713 ALL GATES PASSED" if all_gates_passed else f"\u2717 {len(failed_gates)} GATE(S) FAILED"

# ── Print formatted table ─────────────────────────────────────────────────────
print("\u2550" * 90)
print("  MAS COMPONENT EVALUATION \u2014 SUMMARY")
print("\u2550" * 90)
print(f"{'Sec':<4} {'Component':<22} {'Key Metric':<25} {'Value':<10} {'Threshold':<12} {'Status':<6}")
print("\u2500" * 90)
for row in summary_rows:
    status_icon = {"PASS": "\u2713", "FAIL": "\u2717", "INFO": "\u2022", "N/A": "?"}.get(row["Status"], "")
    print(f"{row['Section']:<4} {row['Component']:<22} {row['Key Metric']:<25} {row['Value']:<10} {row['Threshold']:<12} {status_icon} {row['Status']}")
print("\u2500" * 90)
print(f"\n  {overall_status}")
print(f"  Gates: {len(passed_gates)} passed, {len(failed_gates)} failed, {len(na_gates)} N/A")
print("\u2550" * 90)

# ── Plotly summary table ──────────────────────────────────────────────────────
status_colors = []
for row in summary_rows:
    if row["Status"] == "PASS":
        status_colors.append("#52C41A")
    elif row["Status"] == "FAIL":
        status_colors.append("#FF4D4F")
    elif row["Status"] == "INFO":
        status_colors.append("#58A6FF")
    else:
        status_colors.append("#8B949E")

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Sec</b>", "<b>Component</b>", "<b>Key Metric</b>",
                "<b>Value</b>", "<b>Threshold</b>", "<b>Status</b>"],
        fill_color="#161B22",
        font=dict(color="#E6EDF3", size=12),
        align="left",
        line_color="#30363D",
    ),
    cells=dict(
        values=[
            [r["Section"] for r in summary_rows],
            [r["Component"] for r in summary_rows],
            [r["Key Metric"] for r in summary_rows],
            [r["Value"] for r in summary_rows],
            [r["Threshold"] for r in summary_rows],
            [r["Status"] for r in summary_rows],
        ],
        fill_color=[
            ["#0D1117"] * len(summary_rows),
            ["#0D1117"] * len(summary_rows),
            ["#0D1117"] * len(summary_rows),
            ["#0D1117"] * len(summary_rows),
            ["#0D1117"] * len(summary_rows),
            status_colors,
        ],
        font=dict(
            color=[
                ["#8B949E"] * len(summary_rows),
                ["#E6EDF3"] * len(summary_rows),
                ["#E6EDF3"] * len(summary_rows),
                ["#E6EDF3"] * len(summary_rows),
                ["#8B949E"] * len(summary_rows),
                ["#0D1117"] * len(summary_rows),
            ],
            size=11,
        ),
        align="left",
        line_color="#21262D",
        height=28,
    ),
)])

fig.update_layout(
    title=dict(
        text=f"MAS Component Evaluation Summary — {overall_status}",
        font=dict(color="#E6EDF3", size=16),
    ),
    template="plotly_dark",
    paper_bgcolor="#161B22",
    plot_bgcolor="#0D1117",
    margin=dict(l=20, r=20, t=50, b=20),
    height=max(400, 28 * len(summary_rows) + 100),
)

mlflow.log_figure(fig, "summary_table.html")
fig.show()

# ── End MLflow run ────────────────────────────────────────────────────────────
run_status = "FINISHED" if all_gates_passed else "FAILED"
mlflow.log_metric("overall.gates_passed", len(passed_gates))
mlflow.log_metric("overall.gates_failed", len(failed_gates))
mlflow.log_metric("overall.all_passed", 1.0 if all_gates_passed else 0.0)
mlflow.end_run(status=run_status)

print(f"\n\u2713 MLflow run ended with status: {run_status}")
print(f"  Run ID: {run.info.run_id}")

### Cell 15.2 `[markdown]` — Recommendations

**Purpose:** Auto-generated markdown cell with actionable recommendations based on results.

**Template (filled by Cell 15.1):**

```
## Recommendations

### Critical (must fix before next deployment)
- [If validator_accepted < 0.5]: Whitelist compliance is the bottleneck. Top invalid mechanisms: {list}. Consider expanding whitelist or tuning hypothesis_generator prompt.
- [If schema_valid < 0.9]: CausalChain parse failures. Check hypothesis_generator prompt for JSON formatting issues.

### Improvement Opportunities
- [If evidence_verbatim < 0.7]: LLM is paraphrasing evidence spans. Add stronger verbatim instruction in prompt.
- [If grounding scores bimodal]: Consider adjusting grounding_threshold from 0.5 to {suggested_value}.
- [If reflection_rate > 0.5]: Judges are too strict or hypothesis_generator needs better few-shot examples.

### Infrastructure
- [If hours_since_last > 48]: Pipeline has not run recently. Check agents-refresh-dev job.
- [If VS semantic_precision < 0.5]: Index quality is degrading. Consider reindexing with better embeddings.
- [If cold_start_latency > 180s]: Consider setting min_instances=1 for production.
```

**Notes:** This cell is dynamically generated Python code that writes markdown. The recommendations are conditional on the actual metric values collected throughout the notebook.

In [0]:
# ── Generate conditional recommendations based on evaluation results ───────

critical: list[str] = []
improvements: list[str] = []
infrastructure: list[str] = []

# ── Critical (deployment blockers) ───────────────────────────────────────────
_schema_valid = all_metrics.get("e2e.schema_valid")
_validator_accepted = all_metrics.get("e2e.validator_accepted")
_evidence_parse = all_metrics.get("evidence.parse_success_rate")
_hypothesis_parse = all_metrics.get("hypothesis.parse_success_rate")

if _schema_valid is not None and _schema_valid < 0.9:
    critical.append(
        f"CausalChain parse failures (schema_valid={_schema_valid:.3f} < 0.9). "
        "Check hypothesis_generator prompt for JSON formatting issues. "
        "Review evidence_extractor output for malformed extraction dicts."
    )

if _validator_accepted is not None and _validator_accepted < 0.5:
    critical.append(
        f"Validator acceptance too low (validator_accepted={_validator_accepted:.3f} < 0.5). "
        "Top causes: (1) Mechanism not in whitelist — expand whitelist or tune hypothesis_generator prompt. "
        "(2) Evidence spans not verbatim — strengthen extraction instruction."
    )

if _evidence_parse is not None and _evidence_parse < 0.8:
    critical.append(
        f"Evidence extractor parse success rate low ({_evidence_parse:.1%} < 80%). "
        "LLM may be returning malformed JSON. Check prompt template and temperature settings."
    )

if _hypothesis_parse is not None and _hypothesis_parse < 0.7:
    critical.append(
        f"Hypothesis generator parse success rate low ({_hypothesis_parse:.1%} < 70%). "
        "Reduce few-shot example complexity or add explicit JSON schema in prompt."
    )

# ── Improvement opportunities ─────────────────────────────────────────────
_evidence_verbatim = all_metrics.get("e2e.evidence_spans_verbatim")
_taxonomy_compliance = all_metrics.get("hypothesis.taxonomy_compliance")
_whitelist_compliance = all_metrics.get("hypothesis.whitelist_compliance")
_judge_pass_rate = all_metrics.get("judge.panel_pass_rate")
_agreement = all_metrics.get("gatekeeper.spacy_sql_agreement")

if _evidence_verbatim is not None and _evidence_verbatim < 0.7:
    improvements.append(
        f"Evidence verbatim rate is {_evidence_verbatim:.1%}. "
        "LLM is paraphrasing evidence spans. Add stronger verbatim instruction "
        "in evidence_extractor prompt (e.g., 'Copy exact text from the article')."
    )

if _taxonomy_compliance is not None and _taxonomy_compliance < 0.8:
    improvements.append(
        f"Taxonomy compliance at {_taxonomy_compliance:.1%}. "
        "Hypothesis generator is selecting treatment classes outside the taxonomy. "
        "Include full L1.L2 list in prompt or add constrained decoding."
    )

if _whitelist_compliance is not None and _whitelist_compliance < 0.6:
    improvements.append(
        f"Whitelist compliance at {_whitelist_compliance:.1%}. "
        "Generator is proposing mechanisms not in the (L1, mechanism) whitelist. "
        "Consider expanding the whitelist or adding mechanism examples to few-shot."
    )

if _judge_pass_rate is not None and _judge_pass_rate > 0.95:
    improvements.append(
        f"Judge panel pass rate is very high ({_judge_pass_rate:.1%}). "
        "Judges may be too lenient. Consider raising grounding_threshold or "
        "adding stricter evaluation criteria."
    )
elif _judge_pass_rate is not None and _judge_pass_rate < 0.20:
    improvements.append(
        f"Judge panel pass rate is very low ({_judge_pass_rate:.1%}). "
        "Judges may be too strict or hypothesis_generator needs better few-shot examples. "
        "Consider lowering grounding_threshold or improving extraction quality."
    )

if _agreement is not None and _agreement < 0.95:
    improvements.append(
        f"Gatekeeper spaCy/RLIKE agreement at {_agreement:.1%}. "
        "Tokenization edge cases between spaCy patterns and SQL RLIKE. "
        "Review disagreements for potential false negatives in production (SQL mode)."
    )

# ── Infrastructure ────────────────────────────────────────────────────────
_hours_since = all_metrics.get("gold.hours_since_last_process")
_vs_semantic = all_metrics.get("vs.semantic_precision")
_cold_start = all_metrics.get("endpoint.cold_start_latency_s")
_endpoint_responsive = all_metrics.get("endpoint.responsive")
_vs_chain_count = all_metrics.get("vs.accepted_chain_count")

if _hours_since is not None and _hours_since > 48:
    infrastructure.append(
        f"Pipeline has not run in {_hours_since:.0f} hours (> 48h). "
        "Check agents-refresh-dev job schedule. May need manual trigger: "
        "`databricks bundle run agents_pipeline --target dev`"
    )

if _vs_semantic is not None and _vs_semantic < 0.5:
    infrastructure.append(
        f"VS semantic precision is {_vs_semantic:.0%} (< 50%). "
        "Index quality is degrading due to category imbalance. "
        "Seed more diverse articles across all 8 L1 categories "
        "or consider reindexing with improved embeddings."
    )

if _cold_start is not None and _cold_start > 180:
    infrastructure.append(
        f"Cold start latency is {_cold_start:.0f}s (> 180s). "
        "Consider setting min_instances=1 for production workloads "
        "to avoid cold-start penalty on first request."
    )

if _endpoint_responsive is not None and _endpoint_responsive < 1.0:
    infrastructure.append(
        "Serving endpoint is NOT responsive. "
        "Check endpoint state in Model Serving UI. "
        "May need redeployment with latest model version."
    )

if _vs_chain_count is not None and _vs_chain_count < 50:
    infrastructure.append(
        f"Only {int(_vs_chain_count)} accepted chains in VS index (< 50). "
        "Flywheel is not yet self-improving. Run more articles through the pipeline "
        "to bootstrap the few-shot retriever."
    )

# ── Render recommendations ─────────────────────────────────────────────────
_double_line = "\u2550" * 90
_single_line = "\u2500" * 70

print(_double_line)
print("  RECOMMENDATIONS")
print(_double_line)

if critical:
    print("\n\u250c\u2500 CRITICAL (must fix before next deployment)")
    for i, rec in enumerate(critical, 1):
        print(f"\u2502  {i}. {rec}")
    print("\u2514" + _single_line)
else:
    print("\n\u2713 No critical issues \u2014 safe for deployment")

if improvements:
    print("\n\u250c\u2500 IMPROVEMENT OPPORTUNITIES")
    for i, rec in enumerate(improvements, 1):
        print(f"\u2502  {i}. {rec}")
    print("\u2514" + _single_line)
else:
    print("\n\u2713 No immediate improvement opportunities identified")

if infrastructure:
    print("\n\u250c\u2500 INFRASTRUCTURE")
    for i, rec in enumerate(infrastructure, 1):
        print(f"\u2502  {i}. {rec}")
    print("\u2514" + _single_line)
else:
    print("\n\u2713 Infrastructure is healthy")

# ── Final summary line ─────────────────────────────────────────────────────
total_recs = len(critical) + len(improvements) + len(infrastructure)
print(f"\n{_double_line}")
print(f"  Total: {total_recs} recommendation(s) ")
print(f"    Critical: {len(critical)} | Improvements: {len(improvements)} | Infrastructure: {len(infrastructure)}")
print(_double_line)
print("\n\u2713 SECTION 15 COMPLETE \u2014 MAS Component Evaluation finished")